# Prepare

Local run notes:
- Run this notebook from the project root so `profile/` can be found.
- Install core dependencies from `requirements.txt` first.
- `gurobipy` is optional and only needed for the Gurobi-based sections later in the notebook.


In [1]:
# Recommended once per environment:
# python3 -m pip install -r requirements.txt

# Dependencies are managed outside the notebook.

In [2]:
import os
import math
import time
import random
from collections import defaultdict, Counter
from itertools import product

import numpy as np
import pandas as pd

from tabulate import tabulate

In [3]:
# Local mode: use the notebook/project directory directly.
PROJECT_DIR = os.getcwd()
PROFILE_DIR = os.path.join(PROJECT_DIR, "profile")

print("PROJECT_DIR =", PROJECT_DIR)
print("PROFILE_DIR =", PROFILE_DIR)

PROJECT_DIR = /Users/weiyiqin/Downloads/OR_sim_planner
PROFILE_DIR = /Users/weiyiqin/Downloads/OR_sim_planner/profile


In [4]:
BASE_DIR = PROFILE_DIR

CNN_CSV   = os.path.join(BASE_DIR, "cnn_bench.csv")
VIT_CSV   = os.path.join(BASE_DIR, "vit_base_bench.csv")
BERT_CSV  = os.path.join(BASE_DIR, "bert_base_bench.csv")
GPT2_CSV  = os.path.join(BASE_DIR, "gpt2m_streaming_bench.csv")
LLAMA_CSV = os.path.join(BASE_DIR, "llama32_3b_streaming_bench.csv")

for p in [CNN_CSV, VIT_CSV, BERT_CSV, GPT2_CSV, LLAMA_CSV]:
    print(p, "->", os.path.exists(p))

/Users/weiyiqin/Downloads/OR_sim_planner/profile/cnn_bench.csv -> True
/Users/weiyiqin/Downloads/OR_sim_planner/profile/vit_base_bench.csv -> True
/Users/weiyiqin/Downloads/OR_sim_planner/profile/bert_base_bench.csv -> True
/Users/weiyiqin/Downloads/OR_sim_planner/profile/gpt2m_streaming_bench.csv -> True
/Users/weiyiqin/Downloads/OR_sim_planner/profile/llama32_3b_streaming_bench.csv -> True


In [5]:
PROFILE_ORDER = ["7g", "4g", "3g", "2g", "1g"]

PROFILE_MEM_MB = {
    "1g": 5 * 1024,
    "2g": 10 * 1024,
    "3g": 20 * 1024,
    "4g": 20 * 1024,
    "7g": 40 * 1024,
}

In [6]:
TEMPLATES = [
    ("7",               (1, 0, 0, 0, 0)),
    ("4+3",             (0, 1, 1, 0, 0)),
    ("4+2+1",           (0, 1, 0, 1, 1)),
    ("4+1+1+1",         (0, 1, 0, 0, 3)),
    ("3+3",             (0, 0, 2, 0, 0)),
    ("3+2+1",           (0, 0, 1, 1, 1)),
    ("3+1+1+1",         (0, 0, 1, 0, 3)),
    ("2+2+3",           (0, 0, 1, 2, 0)), # Config 8
    ("3+2+1+1",         (0, 0, 1, 1, 2)), # Config 9, 10
    ("3+1+1+1+1",       (0, 0, 1, 0, 4)), # Config 11
    ("2+2+2+1",         (0, 0, 0, 3, 1)), # Config 12
    ("2+2+1+1+1",       (0, 0, 0, 2, 3)), # Config 13, 14
    ("2+1+1+1+1+1",     (0, 0, 0, 1, 5)), # Config 15, 16, 17, 18
    ("1+1+1+1+1+1+1",   (0, 0, 0, 0, 7)), # Config 19
]

TEMPLATE_K = []
for name, vec in TEMPLATES:
    TEMPLATE_K.append({
        "7g": vec[0],
        "4g": vec[1],
        "3g": vec[2],
        "2g": vec[3],
        "1g": vec[4],
    })

print("Number of templates =", len(TEMPLATES))
for i, (name, _) in enumerate(TEMPLATES):
    print(f"{i:2d}: {name:18s} -> {TEMPLATE_K[i]}")

Number of templates = 14
 0: 7                  -> {'7g': 1, '4g': 0, '3g': 0, '2g': 0, '1g': 0}
 1: 4+3                -> {'7g': 0, '4g': 1, '3g': 1, '2g': 0, '1g': 0}
 2: 4+2+1              -> {'7g': 0, '4g': 1, '3g': 0, '2g': 1, '1g': 1}
 3: 4+1+1+1            -> {'7g': 0, '4g': 1, '3g': 0, '2g': 0, '1g': 3}
 4: 3+3                -> {'7g': 0, '4g': 0, '3g': 2, '2g': 0, '1g': 0}
 5: 3+2+1              -> {'7g': 0, '4g': 0, '3g': 1, '2g': 1, '1g': 1}
 6: 3+1+1+1            -> {'7g': 0, '4g': 0, '3g': 1, '2g': 0, '1g': 3}
 7: 2+2+3              -> {'7g': 0, '4g': 0, '3g': 1, '2g': 2, '1g': 0}
 8: 3+2+1+1            -> {'7g': 0, '4g': 0, '3g': 1, '2g': 1, '1g': 2}
 9: 3+1+1+1+1          -> {'7g': 0, '4g': 0, '3g': 1, '2g': 0, '1g': 4}
10: 2+2+2+1            -> {'7g': 0, '4g': 0, '3g': 0, '2g': 3, '1g': 1}
11: 2+2+1+1+1          -> {'7g': 0, '4g': 0, '3g': 0, '2g': 2, '1g': 3}
12: 2+1+1+1+1+1        -> {'7g': 0, '4g': 0, '3g': 0, '2g': 1, '1g': 5}
13: 1+1+1+1+1+1+1      -> {'7g': 0, '4g

In [7]:
WORKLOAD_SPECS = [
    {
        "name": "llama",
        "family": "llm",
        "csv": LLAMA_CSV,
        "model_match": "Llama-3.2-3B-Instruct",
        "batch_candidates": None,   # auto-discover from CSV
        "prompt_len": 1024,
        "output_tokens": 64,
        "arrival_rate": 3,       # req/s
        "slo_type": "TTFT/TPOT",
        "ttft_slo_ms": 100.0,
        "tpot_slo_ms": 25.0,
    },
    {
        "name": "gpt2",
        "family": "llm",
        "csv": GPT2_CSV,
        "model_match": "gpt2-medium",
        "batch_candidates": None,
        "prompt_len": 64,
        "output_tokens": 64,
        "arrival_rate": 20, #20
        "slo_type": "TTFT/TPOT",
        "ttft_slo_ms": 20.0,
        "tpot_slo_ms": 15.0,
    },
    {
        "name": "vgg16",
        "family": "cv",
        "csv": CNN_CSV,
        "model_match": "vgg16",
        "batch_candidates": None,
        "arrival_rate": 300, #300
        "slo_type": "E2E",
        "e2e_slo_ms": 50.0,
    },
    {
        "name": "resnet50",
        "family": "cv",
        "csv": CNN_CSV,
        "model_match": "resnet50",
        "batch_candidates": None,
        "arrival_rate": 300, #200
        "slo_type": "E2E",
        "e2e_slo_ms": 50.0,
    },
    {
        "name": "vit_base",
        "family": "cv",
        "csv": VIT_CSV,
        "model_match": "vit-base-patch16-224",
        "batch_candidates": None,
        "arrival_rate": 3000.0, #350
        "slo_type": "E2E",
        "e2e_slo_ms": 50.0,
    },
]

WORKLOAD_NAMES = [w["name"] for w in WORKLOAD_SPECS]
N_WORKLOADS = len(WORKLOAD_SPECS)

In [8]:
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def normalize_mig_name(s: str):
    """
    Map strings like:
      'NVIDIA A100-PCIE-40GB MIG 1g.5gb' -> '1g'
      'MIG 3g.20gb' -> '3g'
    """
    if not isinstance(s, str):
        return None
    s = s.lower().replace(" ", "")
    if "1g." in s or "1g" in s:
        return "1g"
    if "2g." in s or "2g" in s:
        return "2g"
    if "3g." in s or "3g" in s:
        return "3g"
    if "4g." in s or "4g" in s:
        return "4g"
    if "7g." in s or "7g" in s:
        return "7g"
    return None


def decode_tps_to_tpot_ms(decode_tps: float):
    """
    decode_tps = tokens / second
    TPOT(ms/token) = 1000 / decode_tps
    """
    if decode_tps is None or np.isnan(decode_tps) or decode_tps <= 0:
        return np.nan
    return 1000.0 / decode_tps


def safe_req_rate_from_latency_ms(lat_ms: float, batch: int):
    """
    service rate (req/s) = 1000 / latency_ms
    """
    if lat_ms is None or np.isnan(lat_ms) or lat_ms <= 0:
        return np.nan
    return batch * 1000.0 / lat_ms

In [9]:
def discover_batch_candidates_for_spec(spec, profile_order):
    df = pd.read_csv(spec["csv"]).copy()

    mig_col = pick_col(df, ["mig_name"])
    model_col = pick_col(df, ["model"])
    batch_col = pick_col(df, ["batch"])

    if mig_col is None or model_col is None or batch_col is None:
        raise ValueError(f"Missing essential columns in {spec['csv']}")

    df["profile"] = df[mig_col].apply(normalize_mig_name)
    df = df[df["profile"].isin(profile_order)].copy()
    df = df[df[model_col] == spec["model_match"]].copy()

    if spec["family"] == "llm":
        prompt_col = pick_col(df, ["prompt_len"])
        output_col = pick_col(df, ["output_tokens"])
        if prompt_col is None or output_col is None:
            raise ValueError(f"Missing LLM columns in {spec['csv']}")
        df = df[df[prompt_col] == spec["prompt_len"]].copy()
        df = df[df[output_col] == spec["output_tokens"]].copy()

    batches = sorted([int(b) for b in df[batch_col].dropna().unique().tolist() if float(b) > 0])

    if spec.get("batch_candidates") is not None:
        allow = set(int(x) for x in spec["batch_candidates"])
        batches = [b for b in batches if b in allow]

    return batches

In [10]:
BATCHES_PER_WORKLOAD = {}
for spec in WORKLOAD_SPECS:
    BATCHES_PER_WORKLOAD[spec["name"]] = discover_batch_candidates_for_spec(spec, PROFILE_ORDER)

print("=== Batch candidates per workload ===")
for k, v in BATCHES_PER_WORKLOAD.items():
    print(f"{k:10s} -> {v}")

BATCH_LEVELS = sorted(set(b for lst in BATCHES_PER_WORKLOAD.values() for b in lst))
BATCH_TO_IDX = {b: idx for idx, b in enumerate(BATCH_LEVELS)}
IDX_TO_BATCH = {idx: b for b, idx in BATCH_TO_IDX.items()}

print("\nGlobal batch levels =", BATCH_LEVELS)
print("N_BATCH_LEVELS =", len(BATCH_LEVELS))

=== Batch candidates per workload ===
llama      -> [1]
gpt2       -> [1]
vgg16      -> [4, 8, 16, 32, 64, 128]
resnet50   -> [4, 8, 16, 32, 64, 128]
vit_base   -> [1, 2, 4, 8, 16, 32, 64]

Global batch levels = [1, 2, 4, 8, 16, 32, 64, 128]
N_BATCH_LEVELS = 8


In [11]:
def extract_rows_for_workload_batch(spec, batch_value, profile_order):
    """
    Return: dict profile -> stats for this (workload, batch)
    """
    df = pd.read_csv(spec["csv"]).copy()

    mig_col = pick_col(df, ["mig_name"])
    model_col = pick_col(df, ["model"])
    batch_col = pick_col(df, ["batch"])
    peak_mem_col = pick_col(df, ["peak_reserved_mb", "peak_alloc_mb"])
    time_mean_col = pick_col(df, ["time_ms_mean"])

    if mig_col is None or model_col is None or batch_col is None:
        raise ValueError(f"Missing essential columns in {spec['csv']}")

    df["profile"] = df[mig_col].apply(normalize_mig_name)
    df = df[df["profile"].isin(profile_order)].copy()
    df = df[df[model_col] == spec["model_match"]].copy()
    df = df[df[batch_col] == batch_value].copy()

    out = {}

    if spec["family"] == "cv":
        if len(df) == 0:
            return out

        g = df.groupby("profile", as_index=False)[time_mean_col].idxmin()
        chosen = df.loc[g[time_mean_col].values].copy()

        for _, row in chosen.iterrows():
            p = row["profile"]
            latency_ms = float(row[time_mean_col])
            peak_mb = float(row[peak_mem_col]) if peak_mem_col is not None else np.nan

            mem_ok = (peak_mb <= PROFILE_MEM_MB[p]) if not np.isnan(peak_mb) else False
            slo_ok = (latency_ms <= spec["e2e_slo_ms"])
            fit = bool(mem_ok and slo_ok)

            mu = safe_req_rate_from_latency_ms(latency_ms, batch_value) if fit else np.nan

            out[p] = {
                "batch": int(batch_value),
                "latency_ms": latency_ms,
                "peak_mem_mb": peak_mb,
                "ttft_ms": np.nan,
                "tpot_ms": np.nan,
                "service_time_ms": latency_ms,
                "mu_req_per_s": mu,
                "fit_mem": bool(mem_ok),
                "fit_slo": bool(slo_ok),
                "fit": fit,
            }

    elif spec["family"] == "llm":
        prompt_col = pick_col(df, ["prompt_len"])
        output_col = pick_col(df, ["output_tokens"])
        ttft_col = pick_col(df, ["ttft_ms"])
        decode_tps_col = pick_col(df, ["decode_tps"])

        if prompt_col is None or output_col is None or ttft_col is None or decode_tps_col is None:
            raise ValueError(f"Missing LLM columns in {spec['csv']}")

        df = df[df[prompt_col] == spec["prompt_len"]].copy()
        df = df[df[output_col] == spec["output_tokens"]].copy()

        if len(df) == 0:
            return out

        chosen = df.groupby("profile", as_index=False).first()

        for _, row in chosen.iterrows():
            p = row["profile"]
            ttft_ms = float(row[ttft_col])
            decode_tps = float(row[decode_tps_col])
            tpot_ms = decode_tps_to_tpot_ms(decode_tps)
            peak_mb = float(row[peak_mem_col]) if peak_mem_col is not None else np.nan

            mem_ok = (peak_mb <= PROFILE_MEM_MB[p]) if not np.isnan(peak_mb) else False
            slo_ok = (ttft_ms <= spec["ttft_slo_ms"]) and (tpot_ms <= spec["tpot_slo_ms"])
            fit = bool(mem_ok and slo_ok)

            # 这里沿用你之前的近似：service_time = TTFT + output_tokens * TPOT
            service_time_ms = ttft_ms + spec["output_tokens"] * tpot_ms if np.isfinite(tpot_ms) else np.nan
            mu = safe_req_rate_from_latency_ms(service_time_ms, batch_value) if fit else np.nan

            out[p] = {
                "batch": int(batch_value),
                "latency_ms": np.nan,
                "peak_mem_mb": peak_mb,
                "ttft_ms": ttft_ms,
                "tpot_ms": tpot_ms,
                "service_time_ms": service_time_ms,
                "mu_req_per_s": mu,
                "fit_mem": bool(mem_ok),
                "fit_slo": bool(slo_ok),
                "fit": fit,
            }

    else:
        raise ValueError(f"Unknown family: {spec['family']}")

    return out

In [12]:
def build_workload_batch_profile_tensors(workload_specs, profile_order, batch_levels, batches_per_workload):
    W = len(workload_specs)
    B = len(batch_levels)
    P = len(profile_order)

    mu = np.full((W, B, P), np.nan, dtype=float)
    service_time_ms = np.full((W, B, P), np.nan, dtype=float)
    latency_ms = np.full((W, B, P), np.nan, dtype=float)
    ttft_ms = np.full((W, B, P), np.nan, dtype=float)
    tpot_ms = np.full((W, B, P), np.nan, dtype=float)
    peak_mem_mb = np.full((W, B, P), np.nan, dtype=float)

    fit_mem = np.zeros((W, B, P), dtype=bool)
    fit_slo = np.zeros((W, B, P), dtype=bool)
    fit = np.zeros((W, B, P), dtype=bool)
    batch_allowed = np.zeros((W, B), dtype=bool)

    raw = {}

    for i, spec in enumerate(workload_specs):
        wname = spec["name"]
        raw[wname] = {}

        for batch_value in batches_per_workload[wname]:
            bidx = BATCH_TO_IDX[batch_value]
            batch_allowed[i, bidx] = True

            row_dict = extract_rows_for_workload_batch(spec, batch_value, profile_order)
            raw[wname][batch_value] = row_dict

            for j, p in enumerate(profile_order):
                if p not in row_dict:
                    continue

                d = row_dict[p]
                mu[i, bidx, j] = d["mu_req_per_s"]
                service_time_ms[i, bidx, j] = d["service_time_ms"]
                latency_ms[i, bidx, j] = d["latency_ms"]
                ttft_ms[i, bidx, j] = d["ttft_ms"]
                tpot_ms[i, bidx, j] = d["tpot_ms"]
                peak_mem_mb[i, bidx, j] = d["peak_mem_mb"]
                fit_mem[i, bidx, j] = d["fit_mem"]
                fit_slo[i, bidx, j] = d["fit_slo"]
                fit[i, bidx, j] = d["fit"]

    return {
        "mu": mu,
        "service_time_ms": service_time_ms,
        "latency_ms": latency_ms,
        "ttft_ms": ttft_ms,
        "tpot_ms": tpot_ms,
        "peak_mem_mb": peak_mem_mb,
        "fit_mem": fit_mem,
        "fit_slo": fit_slo,
        "fit": fit,
        "batch_allowed": batch_allowed,
        "raw": raw,
    }

In [13]:
mat3 = build_workload_batch_profile_tensors(
    workload_specs=WORKLOAD_SPECS,
    profile_order=PROFILE_ORDER,
    batch_levels=BATCH_LEVELS,
    batches_per_workload=BATCHES_PER_WORKLOAD,
)

mu_3d = mat3["mu"]
service_time_ms_3d = mat3["service_time_ms"]
latency_ms_3d = mat3["latency_ms"]
ttft_ms_3d = mat3["ttft_ms"]
tpot_ms_3d = mat3["tpot_ms"]
peak_mem_mb_3d = mat3["peak_mem_mb"]
fit_mem_3d = mat3["fit_mem"]
fit_slo_3d = mat3["fit_slo"]
fit_3d = mat3["fit"]
batch_allowed = mat3["batch_allowed"]

arrival_rate = np.array([w["arrival_rate"] for w in WORKLOAD_SPECS], dtype=float)

print("mu_3d shape =", mu_3d.shape)              # (W, B, P)
print("fit_3d shape =", fit_3d.shape)            # (W, B, P)
print("batch_allowed shape =", batch_allowed.shape)

mu_3d shape = (5, 8, 5)
fit_3d shape = (5, 8, 5)
batch_allowed shape = (5, 8)


In [14]:
def build_option_table(workload_specs, profile_order, batch_levels, mu_3d, fit_3d,
                       fit_mem_3d, fit_slo_3d, peak_mem_mb_3d, service_time_ms_3d,
                       latency_ms_3d, ttft_ms_3d, tpot_ms_3d):
    rows = []

    for i, spec in enumerate(workload_specs):
        for bidx, b in enumerate(batch_levels):
            for pidx, p in enumerate(profile_order):
                rows.append({
                    "w_idx": i,
                    "workload": spec["name"],
                    "family": spec["family"],
                    "batch_idx": bidx,
                    "batch": int(b),
                    "p_idx": pidx,
                    "profile": p,
                    "mu": mu_3d[i, bidx, pidx],
                    "service_time_ms": service_time_ms_3d[i, bidx, pidx],
                    "latency_ms": latency_ms_3d[i, bidx, pidx],
                    "ttft_ms": ttft_ms_3d[i, bidx, pidx],
                    "tpot_ms": tpot_ms_3d[i, bidx, pidx],
                    "peak_mem_mb": peak_mem_mb_3d[i, bidx, pidx],
                    "fit_mem": bool(fit_mem_3d[i, bidx, pidx]),
                    "fit_slo": bool(fit_slo_3d[i, bidx, pidx]),
                    "fit": bool(fit_3d[i, bidx, pidx]),
                })

    option_df = pd.DataFrame(rows)

    feasible_option_df = option_df[option_df["fit"]].copy().reset_index(drop=True)
    feasible_option_df["opt_idx"] = np.arange(len(feasible_option_df))

    return option_df, feasible_option_df

In [15]:
option_df, feasible_option_df = build_option_table(
    workload_specs=WORKLOAD_SPECS,
    profile_order=PROFILE_ORDER,
    batch_levels=BATCH_LEVELS,
    mu_3d=mu_3d,
    fit_3d=fit_3d,
    fit_mem_3d=fit_mem_3d,
    fit_slo_3d=fit_slo_3d,
    peak_mem_mb_3d=peak_mem_mb_3d,
    service_time_ms_3d=service_time_ms_3d,
    latency_ms_3d=latency_ms_3d,
    ttft_ms_3d=ttft_ms_3d,
    tpot_ms_3d=tpot_ms_3d,
)

print("All options =", len(option_df))
print("Feasible options =", len(feasible_option_df))

display(feasible_option_df)

All options = 200
Feasible options = 81


,w_idx,workload,family,batch_idx,batch,p_idx,profile,mu,service_time_ms,latency_ms,ttft_ms,tpot_ms,peak_mem_mb,fit_mem,fit_slo,fit,opt_idx
0,0,llama,llm,0,1,0,7g,0.776891,1287.181268,NaN,46.4961,19.385706,6700.0,True,True,True,0
1,0,llama,llm,0,1,1,4g,0.765989,1305.502207,NaN,71.6864,19.278372,6700.0,True,True,True,1
2,0,llama,llm,0,1,2,3g,0.740961,1349.599316,NaN,92.8640,19.636489,6700.0,True,True,True,2
3,1,gpt2,llm,0,1,0,7g,1.156114,864.966308,NaN,15.1986,13.277620,782.0,True,True,True,3
4,1,gpt2,llm,0,1,1,4g,1.122924,890.532504,NaN,15.5701,13.671288,782.0,True,True,True,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,4,vit_base,cv,5,32,2,3g,1356.794573,23.585000,23.5850,NaN,NaN,626.0,True,True,True,76
77,4,vit_base,cv,5,32,3,2g,857.524915,37.316700,37.3167,NaN,NaN,626.0,True,True,True,77
78,4,vit_base,cv,6,64,0,7g,2676.491621,23.911900,23.9119,NaN,NaN,1060.0,True,True,True,78
79,4,vit_base,cv,6,64,1,4g,1718.983758,37.231300,37.2313,NaN,NaN,1060.0,True,True,True,79


In [16]:
FEASIBLE_OPTIONS_BY_WORKLOAD = {
    i: feasible_option_df[feasible_option_df["w_idx"] == i]["opt_idx"].tolist()
    for i in range(N_WORKLOADS)
}

FEASIBLE_OPTIONS_BY_PROFILE = {
    j: feasible_option_df[feasible_option_df["p_idx"] == j]["opt_idx"].tolist()
    for j in range(len(PROFILE_ORDER))
}

print("=== #feasible options by workload ===")
for i, w in enumerate(WORKLOAD_NAMES):
    print(f"{w:10s} -> {len(FEASIBLE_OPTIONS_BY_WORKLOAD[i])}")

=== #feasible options by workload ===
llama      -> 3
gpt2       -> 5
vgg16      -> 18
resnet50   -> 23
vit_base   -> 32


In [17]:
def show_feasible_options_per_workload(feasible_option_df):
    rows = []
    for w, g in feasible_option_df.groupby("workload"):
        items = []
        for _, r in g.sort_values(["batch", "profile"]).iterrows():
            items.append(f"(b={int(r['batch'])}, p={r['profile']}, mu={r['mu']:.4f})")
        rows.append([w, len(g), "; ".join(items[:20]) + (" ..." if len(items) > 20 else "")])

    print(tabulate(
        rows,
        headers=["Workload", "#Feasible (batch,profile) options", "Options"],
        tablefmt="github"
    ))

show_feasible_options_per_workload(feasible_option_df)

| Workload   |   #Feasible (batch,profile) options | Options                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
|------------|-------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [18]:
def check_global_feasibility_batch(workload_specs, feasible_option_df):
    ok = True
    rows = []

    for i, spec in enumerate(workload_specs):
        g = feasible_option_df[feasible_option_df["w_idx"] == i].copy()
        pairs = list(zip(g["batch"].tolist(), g["profile"].tolist()))
        rows.append([
            spec["name"],
            len(pairs),
            pairs[:15] if len(pairs) <= 15 else pairs[:15] + ["..."]
        ])
        if len(pairs) == 0:
            ok = False

    print(tabulate(
        rows,
        headers=["Workload", "#Feasible (batch,profile)", "Feasible pairs"],
        tablefmt="github"
    ))

    if ok:
        print("\n[OK] Every workload has at least one feasible (batch, profile) option.")
    else:
        print("\n[WARN] Some workload has no feasible (batch, profile) option. MILP will be infeasible.")

check_global_feasibility_batch(WORKLOAD_SPECS, feasible_option_df)

| Workload   |   #Feasible (batch,profile) | Feasible pairs                                                                                                                                                                    |
|------------|-----------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| llama      |                           3 | [(1, '7g'), (1, '4g'), (1, '3g')]                                                                                                                                                 |
| gpt2       |                           5 | [(1, '7g'), (1, '4g'), (1, '3g'), (1, '2g'), (1, '1g')]                                                                                                                           |
| vgg16      |                          18 | [(4, '7g'), (4, '4g'), (4, '3g'), (4, '2g'), (4, '1g'),

In [19]:
def print_experiment_header_batch():
    print("=" * 90)
    print("Simulation: Min #GPU under arrival-rate / SLO / MIG constraints (batch-selectable)")
    print("=" * 90)
    print(f"Workloads      : {WORKLOAD_NAMES}")
    print(f"Profiles       : {PROFILE_ORDER}")
    print(f"Batch levels   : {BATCH_LEVELS}")
    print(f"#Templates     : {len(TEMPLATES)}")
    print()

    rows = []
    for w in WORKLOAD_SPECS:
        if w["family"] == "llm":
            slo_str = f"TTFT<={w['ttft_slo_ms']}ms, TPOT<={w['tpot_slo_ms']}ms"
            extra = f"prompt={w['prompt_len']}, out={w['output_tokens']}"
        else:
            slo_str = f"E2E<={w['e2e_slo_ms']}ms"
            extra = "-"

        rows.append([
            w["name"],
            w["family"],
            BATCHES_PER_WORKLOAD[w["name"]],
            w["arrival_rate"],
            slo_str,
            extra
        ])

    print(tabulate(
        rows,
        headers=["Workload", "Family", "Batch candidates", "Arrival(req/s)", "SLO", "Extra"],
        tablefmt="github",
        floatfmt=".4f"
    ))

print_experiment_header_batch()

Simulation: Min #GPU under arrival-rate / SLO / MIG constraints (batch-selectable)
Workloads      : ['llama', 'gpt2', 'vgg16', 'resnet50', 'vit_base']
Profiles       : ['7g', '4g', '3g', '2g', '1g']
Batch levels   : [1, 2, 4, 8, 16, 32, 64, 128]
#Templates     : 14

| Workload   | Family   | Batch candidates         |   Arrival(req/s) | SLO                         | Extra               |
|------------|----------|--------------------------|------------------|-----------------------------|---------------------|
| llama      | llm      | [1]                      |           3.0000 | TTFT<=100.0ms, TPOT<=25.0ms | prompt=1024, out=64 |
| gpt2       | llm      | [1]                      |          20.0000 | TTFT<=20.0ms, TPOT<=15.0ms  | prompt=64, out=64   |
| vgg16      | cv       | [4, 8, 16, 32, 64, 128]  |         300.0000 | E2E<=50.0ms                 | -                   |
| resnet50   | cv       | [4, 8, 16, 32, 64, 128]  |         300.0000 | E2E<=50.0ms                 | -          

In [20]:
def format_instance_list(instances):
    items = []
    for inst in instances:
        items.append(f"({inst['batch']}, {inst['profile']}, {inst['count']})")
    return "; ".join(items)

def print_solution_summary(
    method_name,
    elapsed,
    feasible,
    objective,
    K_total,
    templates_used,
    alloc
):
    print("=" * 90)
    print(f"Method        : {method_name}")
    print(f"Elapsed (s)   : {elapsed:.4f}")
    print(f"Feasible      : {feasible}")
    #print(f"GPU count     : {sum(K_total.values())}")
    print(f"GPU count     : {len(templates_used)}")
    print(f"Objective     : {objective:.4f}" if objective is not None else "Objective     : -")
    print(f"Templates     : {templates_used}")
    print(f"K_total       : {K_total}")
    print("=" * 90)


    rows = []
    for w in alloc:
        ratio = w["provided"] / w["arrival"] if w["arrival"] > 0 else 0.0

        rows.append([
            w["workload"],
            f"{w['arrival']:.4f}",
            f"{w['provided']:.4f}",
            f"{ratio:.3f}",
            f"{w['slack']:.4f}",
            format_instance_list(w["instances"])
        ])

    print("\nPer-workload allocation / rate summary:")
    print(tabulate(
        rows,
        headers=[
            "Workload",
            "Arrival",
            "Provided",
            "Provided/Arrival",
            "Slack",
            "Placement (batch,profile,count)"
        ],
        tablefmt="github"
    ))

In [21]:
def build_allocation_from_x(
    feasible_option_df,
    x_sol,
    arrival_rate
):
    """
    x_sol: dict {opt_idx: value}
    """

    alloc = []

    for i in range(len(arrival_rate)):
        g = feasible_option_df[feasible_option_df["w_idx"] == i]

        instances = []
        provided = 0.0

        for _, row in g.iterrows():
            o = row["opt_idx"]
            x_val = x_sol.get(o, 0)

            if x_val <= 0:
                continue

            mu = row["mu"]
            provided += x_val * mu

            instances.append({
                "batch": int(row["batch"]),
                "profile": row["profile"],
                "count": int(x_val),
                "mu": mu
            })

        arrival = arrival_rate[i]
        slack = provided - arrival

        alloc.append({
            "workload": g["workload"].iloc[0],
            "arrival": arrival,
            "provided": provided,
            "slack": slack,
            "instances": instances
        })

    return alloc

In [22]:
def compute_slo_violation_rate(alloc):
    rows = []

    for w in alloc:
        if w["arrival"] <= 0:
            viol = 0.0
        else:
            viol = max(0.0, 1.0 - w["provided"] / w["arrival"])

        rows.append([
            w["workload"],
            f"{viol:.4f}"
        ])

    print("\nSLO Violation Rate per workload:")
    print(tabulate(
        rows,
        headers=["Workload", "Violation Rate"],
        tablefmt="github"
    ))

# MILP

In [23]:
# Optional: only run this if you want the Gurobi-based cells below.
# It also requires a valid Gurobi installation/license.
# `gurobipy` is included in requirements.txt and should be installed in the local environment.

In [24]:

import time
import math
import numpy as np
from tabulate import tabulate
import gurobipy as gp
from gurobipy import GRB


def milp_expand_template_list(y_sol, templates):
    out = []
    for t_idx, cnt in sorted(y_sol.items()):
        out.extend([templates[t_idx][0]] * int(cnt))
    return out


def milp_build_K_total(y_sol, template_K, profile_order):
    out = {p: 0 for p in profile_order}
    for t_idx, cnt in y_sol.items():
        for p in profile_order:
            out[p] += int(cnt) * int(template_K[t_idx][p])
    return out


def print_batch_instance_details(feasible_option_df, x_sol):
    rows = []
    total_mu = 0.0
    chosen = feasible_option_df[feasible_option_df["opt_idx"].isin(x_sol.keys())].copy()
    chosen = chosen.sort_values(["w_idx", "batch", "p_idx"]).reset_index(drop=True)

    for _, row in chosen.iterrows():
        o = int(row["opt_idx"])
        xv = int(x_sol[o])
        mu_o = float(row["mu"])
        subtotal = xv * mu_o
        total_mu += subtotal
        rows.append([o, row["workload"], int(row["batch"]), row["profile"], xv, f"{mu_o:.6f}", f"{subtotal:.6f}"])

    print("\nChosen instances:")
    print(tabulate(rows, headers=["opt_idx", "workload", "batch", "profile", "count", "mu", "count*mu"], tablefmt="github"))
    print(f"\nTotal provided throughput = {total_mu:.6f}")


def prune_dominated_options(feasible_option_df):
    """
    Workload-wise Pareto pruning.
    Drop option B if there exists option A for the same workload with:
      mu_A >= mu_B and profile_size(A) <= profile_size(B),
    with at least one strict improvement.
    Keep opt_idx unchanged for compatibility with downstream extraction/warm-start.
    """
    def _profile_size_from_name(p):
        s = str(p)
        if s == "void":
            return 0
        return int(s.replace("g", ""))

    df = feasible_option_df.copy()
    keep_mask = np.ones(len(df), dtype=bool)

    for w_idx, grp in df.groupby("w_idx", sort=False):
        rows = grp[["opt_idx", "mu", "profile"]].copy()
        rows["size"] = rows["profile"].map(_profile_size_from_name).astype(int)

        idxs = rows.index.tolist()
        for i in idxs:
            if not keep_mask[i]:
                continue
            mu_i = float(rows.loc[i, "mu"])
            sz_i = int(rows.loc[i, "size"])

            dominated = False
            for j in idxs:
                if i == j:
                    continue
                mu_j = float(rows.loc[j, "mu"])
                sz_j = int(rows.loc[j, "size"])

                if (mu_j >= mu_i and sz_j <= sz_i) and (mu_j > mu_i or sz_j < sz_i):
                    dominated = True
                    break

            if dominated:
                keep_mask[i] = False

    pruned = df.loc[keep_mask].copy().reset_index(drop=True)
    return pruned


def compute_elastic_up_by_opt(base_option_df):
    """
    For each option r=(workload, profile, batch, mu), compute how much throughput
    can still be gained by increasing batch size while keeping the SAME profile.

    delta_mu_up[r] = max_{b' > b, same workload, same profile}(mu_{b'} - mu_b, 0)
    """
    base_df = base_option_df.copy()
    delta_map = {}

    # group by workload/profile on the FULL option table (before pruning)
    for (w_idx, profile), grp in base_df.groupby(["w_idx", "profile"], sort=False):
        grp2 = grp[["opt_idx", "batch", "mu"]].copy()
        grp2["batch"] = grp2["batch"].astype(int)
        grp2["mu"] = grp2["mu"].astype(float)
        grp2 = grp2.sort_values(["batch", "mu"]).reset_index(drop=True)

        batches = grp2["batch"].tolist()
        mus = grp2["mu"].tolist()
        opt_idxs = grp2["opt_idx"].tolist()

        # suffix max of future mu for strictly larger batch
        suffix_best = [0.0] * len(grp2)
        best = -float("inf")
        for i in range(len(grp2) - 1, -1, -1):
            suffix_best[i] = best
            best = max(best, mus[i])

        for i, opt_idx in enumerate(opt_idxs):
            cur_mu = mus[i]
            future_best_mu = suffix_best[i]
            if future_best_mu == -float("inf"):
                delta = 0.0
            else:
                delta = max(0.0, future_best_mu - cur_mu)
            delta_map[int(opt_idx)] = float(delta)

    # any missing key gets 0
    for opt_idx in base_df["opt_idx"].tolist():
        delta_map.setdefault(int(opt_idx), 0.0)

    return delta_map


def solve_milp_gurobi_batch_unified(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    time_limit_s=None,
    mip_gap=None,
    threads=None,
    verbose=False,
    apply_option_pruning=True,
    warm_start_res=None,
):
    start = time.time()

    base_df = feasible_option_df.copy()
    elastic_up_map = compute_elastic_up_by_opt(base_df)

    if apply_option_pruning:
        df = prune_dominated_options(base_df).reset_index(drop=True)
    else:
        df = base_df.reset_index(drop=True)

    # keep full-table opt_idx and attach elastic-up metadata
    df["elastic_up"] = df["opt_idx"].map(lambda o: float(elastic_up_map.get(int(o), 0.0)))

    n_opts = len(df)
    nT = len(templates)

    opt_rows = list(range(n_opts))
    t_ids = list(range(nT))

    options_by_workload = {i: [] for i in range(n_workloads)}
    options_by_profile = {p: [] for p in profile_order}

    for r in opt_rows:
        w_idx = int(df.loc[r, "w_idx"])
        p = df.loc[r, "profile"]
        options_by_workload[w_idx].append(r)
        options_by_profile[p].append(r)

    for i in range(n_workloads):
        if len(options_by_workload[i]) == 0:
            return {
                "method": "MILP-Gurobi(batch)",
                "feasible": False,
                "status": f"NO_OPTION_FOR_WORKLOAD_{i}",
                "elapsed": time.time() - start,
                "effective_option_df": df,
            }

    model = gp.Model("milp_batch_elastic")

    if not verbose:
        model.Params.OutputFlag = 0
    if time_limit_s is not None:
        model.Params.TimeLimit = float(time_limit_s)
    if mip_gap is not None:
        model.Params.MIPGap = float(mip_gap)
    if threads is not None:
        model.Params.Threads = int(threads)

    y = model.addVars(t_ids, vtype=GRB.INTEGER, lb=0, name="y")

    total_gpu = model.addVar(vtype=GRB.INTEGER, lb=0, name="total_gpu")
    model.addConstr(total_gpu == gp.quicksum(y[t] for t in t_ids), name="def_total_gpu")

    cap = {}
    for p in profile_order:
        cap[p] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"cap_{p}")
        model.addConstr(
            cap[p] == gp.quicksum(int(template_K[t][p]) * y[t] for t in t_ids),
            name=f"def_cap_{p}"
        )

    ub_gpu_loose = 0
    for i in range(n_workloads):
        g = df[df["w_idx"] == i]
        best_mu = float(g["mu"].max())
        ub_gpu_loose += int(math.ceil(arrival_rate[i] / best_mu)) if best_mu > 0 else 0
    ub_gpu_loose = max(1, ub_gpu_loose)

    x = {}
    for r in opt_rows:
        w_idx = int(df.loc[r, "w_idx"])
        p = df.loc[r, "profile"]
        mu_o = float(df.loc[r, "mu"])

        ub_by_demand = int(math.ceil(arrival_rate[w_idx] / mu_o)) + 2 if mu_o > 0 else 0
        max_profile_per_gpu = max(int(template_K[t][p]) for t in t_ids)
        ub_by_gpu = ub_gpu_loose * max_profile_per_gpu
        ub = max(1, min(ub_by_demand, ub_by_gpu))

        x[r] = model.addVar(vtype=GRB.INTEGER, lb=0, ub=ub, name=f"x_{r}")

    provided = {}
    slack = {}
    for i in range(n_workloads):
        provided[i] = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name=f"provided_{i}")
        slack[i] = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name=f"slack_{i}")

        expr = gp.quicksum(float(df.loc[r, "mu"]) * x[r] for r in options_by_workload[i])
        model.addConstr(provided[i] == expr, name=f"def_provided_{i}")
        model.addConstr(provided[i] >= float(arrival_rate[i]), name=f"demand_{i}")
        model.addConstr(slack[i] == provided[i] - float(arrival_rate[i]), name=f"def_slack_{i}")

    for p in profile_order:
        model.addConstr(
            gp.quicksum(x[r] for r in options_by_profile[p]) <= cap[p],
            name=f"profile_cap_{p}"
        )

    # diagnostic-only static slack (not the 2nd objective anymore)
    total_slack = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="total_slack")
    model.addConstr(total_slack == gp.quicksum(slack[i] for i in range(n_workloads)), name="def_total_slack")

    # NEW: elastic slack = future throughput gain reachable by increasing batch
    elastic_up = {}
    for r in opt_rows:
        elastic_up[r] = float(df.loc[r, "elastic_up"])

    total_elastic_slack = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="total_elastic_slack")
    model.addConstr(
        total_elastic_slack == gp.quicksum(elastic_up[r] * x[r] for r in opt_rows),
        name="def_total_elastic_slack",
    )

    total_instances = model.addVar(vtype=GRB.INTEGER, lb=0, name="total_instances")
    model.addConstr(total_instances == gp.quicksum(x[r] for r in opt_rows), name="def_total_instances")

    remaining = {}
    for p in profile_order:
        remaining[p] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"remaining_{p}")
        model.addConstr(remaining[p] == cap[p] - gp.quicksum(x[r] for r in options_by_profile[p]), name=f"def_remaining_{p}")

    total_remaining_slots = model.addVar(vtype=GRB.INTEGER, lb=0, name="total_remaining_slots")
    model.addConstr(
        total_remaining_slots == gp.quicksum(int(str(p).replace("g","")) * remaining[p] for p in profile_order),
        name="def_total_remaining_slots",
    )

    model.setObjectiveN(total_gpu, index=0, priority=3, weight=1.0, name="obj_gpu")
    model.setObjectiveN(total_elastic_slack, index=1, priority=2, weight=-1.0, name="obj_elastic_slack")
    model.setObjectiveN(total_remaining_slots, index=2, priority=1, weight=-1.0, name="obj_remaining")

    # ---- warm start from previous MILP result ----
    if warm_start_res is not None:
        prev_x = dict(warm_start_res.get("x_sol", {}) or {})
        prev_y = dict(warm_start_res.get("y_sol", {}) or {})
        global_to_local = {int(df.loc[r, "opt_idx"]): r for r in opt_rows}

        for t in t_ids:
            if t in prev_y:
                y[t].Start = float(prev_y[t])

        for global_opt_idx, val in prev_x.items():
            if global_opt_idx in global_to_local:
                r = global_to_local[global_opt_idx]
                x[r].Start = float(val)

    model.optimize()

    elapsed = time.time() - start
    status_str = {
        GRB.OPTIMAL: "OPTIMAL",
        GRB.TIME_LIMIT: "TIME_LIMIT",
        GRB.SUBOPTIMAL: "SUBOPTIMAL",
        GRB.INFEASIBLE: "INFEASIBLE",
    }.get(model.Status, str(model.Status))

    if model.SolCount == 0:
        return {
            "method": "MILP-Gurobi(batch)",
            "feasible": False,
            "status": status_str,
            "elapsed": elapsed,
            "effective_option_df": df,
        }

    y_sol = {}
    for t in t_ids:
        v = int(round(y[t].X))
        if v > 0:
            y_sol[t] = v

    x_sol = {}
    for r in opt_rows:
        v = int(round(x[r].X))
        if v > 0:
            global_opt_idx = int(df.loc[r, "opt_idx"])
            x_sol[global_opt_idx] = v

    K_total = milp_build_K_total(y_sol, template_K, profile_order)
    chosen_templates = milp_expand_template_list(y_sol, templates)

    alloc = build_allocation_from_x(
        feasible_option_df=base_df,
        x_sol=x_sol,
        arrival_rate=arrival_rate
    )
    provided_rate = np.array([w["provided"] for w in alloc], dtype=float)
    used_profile_types = sum(1 for p in profile_order if K_total[p] > 0)

    return {
        "method": "MILP-Gurobi(batch)",
        "feasible": True,
        "status": status_str,
        "elapsed": elapsed,
        "gpu_count": int(round(total_gpu.X)),
        "objective": int(round(total_gpu.X)),
        "chosen_templates": chosen_templates,
        "K_total": K_total,
        "x_sol": x_sol,
        "y_sol": y_sol,
        "alloc": alloc,
        "provided_rate": provided_rate,
        "total_slack": float(total_slack.X),                       # diagnostic
        "total_elastic_slack": float(total_elastic_slack.X),     # objective-2 value
        "total_remaining_slots": int(round(total_remaining_slots.X)),
        "total_instances": int(round(total_instances.X)),
        "used_profile_types": used_profile_types,
        "effective_option_df": df,
    }


def print_milp_gurobi_batch_result(res):
    if not res["feasible"]:
        print("=" * 90)
        print(f"Method         : {res['method']}")
        print(f"Elapsed (s)    : {res['elapsed']:.4f}")
        print("Feasible       : False")
        print(f"Status         : {res.get('status', '-')}")
        if "effective_option_df" in res and res["effective_option_df"] is not None:
            print(f"Effective opts : {len(res['effective_option_df'])}")
        print("=" * 90)
        return

    print_solution_summary(
        method_name=res["method"],
        elapsed=res["elapsed"],
        feasible=res["feasible"],
        objective=res["objective"],
        K_total=res["K_total"],
        templates_used=res["chosen_templates"],
        alloc=res["alloc"],
    )

    print(f"\nSolver status        : {res['status']}")
    print(f"Total static slack   : {res['total_slack']:.6f}")
    print(f"Total elastic slack  : {res.get('total_elastic_slack', 0.0):.6f}")
    print(f"Total instances      : {res['total_instances']}")
    print(f"Total remaining      : {res['total_remaining_slots']}")
    print(f"Used profile types   : {res['used_profile_types']}")
    if "effective_option_df" in res and res["effective_option_df"] is not None:
        print(f"Effective options    : {len(res['effective_option_df'])}")

    compute_slo_violation_rate(res["alloc"])


def print_milp_template_counts_unified(y_sol, templates):
    rows = []
    for t_idx, cnt in sorted(y_sol.items()):
        rows.append([t_idx, templates[t_idx][0], cnt])

    print("\nSelected template counts:")
    print(tabulate(rows, headers=["t_idx", "template", "count"], tablefmt="github"))


### MILP result

In [25]:
milp_res = solve_milp_gurobi_batch_unified(
    feasible_option_df=feasible_option_df,
    arrival_rate=arrival_rate,
    templates=TEMPLATES,
    template_K=TEMPLATE_K,
    profile_order=PROFILE_ORDER,
    n_workloads=N_WORKLOADS,
    time_limit_s=None,
    mip_gap=None,
    threads=None,
    verbose=False,
)

print_milp_gurobi_batch_result(milp_res)

if milp_res["feasible"]:
    print_milp_template_counts_unified(milp_res["y_sol"], TEMPLATES)
    print_batch_instance_details(feasible_option_df, milp_res["x_sol"])



Restricted license - for non-production use only - expires 2027-11-29
Method        : MILP-Gurobi(batch)
Elapsed (s)   : 0.0839
Feasible      : True
GPU count     : 6
Objective     : 6.0000
Templates     : ['4+3', '4+1+1+1', '4+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '2+1+1+1+1+1']
K_total       : {'7g': 0, '4g': 3, '3g': 3, '2g': 1, '1g': 19}

Per-workload allocation / rate summary:
| Workload   |   Arrival |   Provided |   Provided/Arrival |    Slack | Placement (batch,profile,count)   |
|------------|-----------|------------|--------------------|----------|-----------------------------------|
| llama      |         3 |     3.0139 |              1.005 |   0.0139 | (1, 4g, 2); (1, 3g, 2)            |
| gpt2       |        20 |    20.3592 |              1.018 |   0.3592 | (1, 1g, 18)                       |
| vgg16      |       300 |   483.876  |              1.613 | 183.876  | (16, 2g, 1)                       |
| resnet50   |       300 |   351.034  |              1.17  |  51.0342 | (16, 1g

# Transition

### Code Block 1: Imports / notebook-aligned constants / state classes


In [182]:
# ============================================================
# [Transition Cell 1] Imports / notebook-aligned constants / state classes
# Role:
#   - import dependencies
#   - build template/profile lookup tables from prepare-stage variables
#   - define state classes used only after #transition
# ============================================================

from dataclasses import dataclass, field
from collections import defaultdict, Counter
from typing import Dict, Any, List, Optional, Tuple
import time
import math
import copy

# ---------- notebook-aligned constants ----------
if "PROFILE_ORDER" not in globals():
    PROFILE_ORDER = ["7g", "4g", "3g", "2g", "1g"]

PROFILE_SIZE = {
    "7g": 7,
    "4g": 4,
    "3g": 3,
    "2g": 2,
    "1g": 1,
    "void": 0,
}
SIZE_TO_PROFILE = {7: "7g", 4: "4g", 3: "3g", 2: "2g", 1: "1g"}

def _template_name_list_from_globals():
    return [name for name, _ in TEMPLATES]

def _template_capacity_dict(template_name: str) -> Dict[str, int]:
    for name, vec in TEMPLATES:
        if name == template_name:
            return {"7g": int(vec[0]), "4g": int(vec[1]), "3g": int(vec[2]), "2g": int(vec[3]), "1g": int(vec[4])}
    raise KeyError(f"Unknown template: {template_name}")

TEMPLATE_NAME_TO_K = {name: _template_capacity_dict(name) for name, _ in TEMPLATES}

def _template_to_parts(template_name: str) -> Tuple[int, ...]:
    return tuple(int(x) for x in template_name.split("+"))

def _parts_to_profiles(parts: Tuple[int, ...]) -> Tuple[str, ...]:
    return tuple(SIZE_TO_PROFILE[int(x)] for x in parts)

def _physical_profiles_to_string(profiles: Tuple[str, ...]) -> str:
    return "+".join(str(PROFILE_SIZE[p]) for p in profiles)

def _physical_profiles_to_intervals(profiles: Tuple[str, ...]) -> List[Tuple[int, int, str]]:
    out = []
    cur = 0
    for p in profiles:
        sz = PROFILE_SIZE[p]
        out.append((cur, cur + sz, p))
        cur += sz
    if cur < 7:
        out.append((cur, 7, "void"))
    return out

# 14 abstract templates -> 19 legal physical configurations
ABSTRACT_TO_PHYSICAL = {
    "7": [("7g",)],
    "4+3": [("4g", "3g")],
    "4+2+1": [("4g", "2g", "1g")],
    "4+1+1+1": [("4g", "1g", "1g", "1g")],
    "3+3": [("3g", "3g")],
    "3+2+1": [("3g", "2g", "1g")],
    "3+1+1+1": [("3g", "1g", "1g", "1g")],
    "2+2+3": [("2g", "2g", "3g")],
    "3+2+1+1": [
        ("2g", "1g", "1g", "3g"),
        ("1g", "1g", "2g", "3g"),
    ],
    "3+1+1+1+1": [("1g", "1g", "1g", "1g", "3g")],
    "2+2+2+1": [("2g", "2g", "2g", "1g")],
    "2+2+1+1+1": [
        ("2g", "1g", "1g", "2g", "1g"),
        ("1g", "1g", "2g", "2g", "1g"),
    ],
    "2+1+1+1+1+1": [
        ("2g", "1g", "1g", "1g", "1g", "1g"),
        ("1g", "1g", "2g", "1g", "1g", "1g"),
        ("1g", "1g", "1g", "1g", "2g", "1g"),
        ("1g", "1g", "1g", "1g", "1g", "2g"),
    ],
    "1+1+1+1+1+1+1": [("1g", "1g", "1g", "1g", "1g", "1g", "1g")],
}

def _all_unique_physical_realizations(template_name: str) -> List[Tuple[str, List[Tuple[int, int, str]]]]:
    if template_name not in ABSTRACT_TO_PHYSICAL:
        raise KeyError(f"Unknown abstract template: {template_name}")
    out = []
    for profiles in ABSTRACT_TO_PHYSICAL[template_name]:
        out.append((_physical_profiles_to_string(profiles), _physical_profiles_to_intervals(profiles)))
    return out

@dataclass
class MigInstance:
    start: int
    end: int
    profile: str
    workload: Optional[str] = None
    batch: Optional[int] = None
    mu: float = 0.0
    preserved: bool = False

@dataclass
class GPUState:
    gpu_id: int
    source: str = "real"
    instances: List[Any] = field(default_factory=list)

    def sort_instances(self):
        self.instances = sorted(self.instances, key=lambda x: (x.start, x.end, x.profile))

    def template_str(self):
        self.sort_instances()
        return "+".join(str(PROFILE_SIZE[inst.profile]) for inst in self.instances if inst.profile != "void")

@dataclass
class ClusterState:
    gpus: List[Any] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def real_gpus(self):
        return [g for g in self.gpus if getattr(g, "source", "real") == "real"]


### Code Block 2: Validation / pretty-print / verification


In [183]:

# ============================================================
# [Transition Cell 2] Validation / pretty-print / verification
# Role:
#   - validate target state structure
#   - pretty-print built target state
#   - verify built target against MILP multiset
# Note:
#   verification is relaxed to allow upward placement
#   (e.g., a MILP 3g demand may be realized on a 4g slot)
# ============================================================

def assert_valid_cluster_state(state: "ClusterState"):
    for g in state.real_gpus():
        g.sort_instances()
        cur = 0
        for inst in g.instances:
            if inst.start != cur:
                raise ValueError(f"GPU {g.gpu_id}: expected start={cur}, got {inst.start}")
            if inst.end <= inst.start:
                raise ValueError(f"GPU {g.gpu_id}: bad interval ({inst.start},{inst.end})")
            if inst.profile != "void":
                if PROFILE_SIZE[inst.profile] != (inst.end - inst.start):
                    raise ValueError(f"GPU {g.gpu_id}: profile-size mismatch on {inst.profile}")
            cur = inst.end
        if cur != 7:
            raise ValueError(f"GPU {g.gpu_id}: total covered slices={cur}, expected 7")

def summarize_state(state: "ClusterState", title="[STATE]"):
    print("=" * 90)
    print(title)
    print("=" * 90)
    display_map = state.metadata.get("display_id_map", {})
    for g in state.real_gpus():
        row = []
        for inst in sorted(g.instances, key=lambda x: (x.start, x.end)):
            if inst.workload is None:
                row.append(f"[{inst.start},{inst.end}) {inst.profile}: free")
            else:
                row.append(
                    f"[{inst.start},{inst.end}) {inst.profile}: "
                    f"{inst.workload} / B{inst.batch} / mu={inst.mu:.4f}"
                )
        disp = display_map.get(g.gpu_id, g.gpu_id)
        if disp == g.gpu_id:
            print(f"GPU {g.gpu_id}: " + " | ".join(row))
        else:
            print(f"GPU {g.gpu_id} (display {disp}): " + " | ".join(row))

def print_target_build_result(target: "ClusterState", title="[CHECK] TARGET STATE"):
    """Pretty-print a built target state and its build metrics."""
    summarize_state(target, title=title)
    metrics = target.metadata.get("build_metrics", {})
    if metrics:
        print("\nSearch metrics:")
        for k in [
            "elapsed_time_sec",
            "ordered_abstract_templates",
            "ordered_physical_templates",
            "layout_preserve_score",
            "exact_preserve",
            "same_gpu_preserve",
            "spread",
            "collocate_pairs",
            "mixed_gpu_count",
            "template_match_count",
            "score_tuple",
        ]:
            if k in metrics:
                print(f"  {k}: {metrics[k]}")

# Forward declaration for notebook static analyzers; actual implementation is in Transition Cell 3.
def extract_instance_demands_from_milp(milp_res: Dict[str, Any], feasible_option_df_: Optional[Any] = None):
    raise RuntimeError("extract_instance_demands_from_milp is implemented in Transition Cell 3; run that cell before verification.")

def _collect_instance_multiset_from_target(target: "ClusterState") -> Counter:
    cnt = Counter()
    for g in target.real_gpus():
        for inst in g.instances:
            if inst.workload is None:
                continue
            cnt[(inst.workload, inst.profile, int(inst.batch))] += 1
    return cnt

def _collect_instance_multiset_from_milp(milp_res: Dict[str, Any]) -> Counter:
    cnt = Counter()
    for d in extract_instance_demands_from_milp(milp_res):
        cnt[(d["workload"], d["profile"], int(d["batch"]))] += int(d["count"])
    return cnt

def _group_multiset_by_workload_batch(counter_obj: Counter):
    grouped = defaultdict(lambda: Counter())
    for (w, p, b), c in counter_obj.items():
        grouped[(w, int(b))][PROFILE_SIZE[p]] += int(c)
    return grouped

def _relaxed_cover_ok_for_one_group(target_size_cnt: Counter, demand_size_cnt: Counter) -> bool:
    """
    Relaxed verification only for the rewrite rule:
      3g demand may be realized on a 4g slot.

    All other profile sizes must match exactly.
    """
    available = Counter(target_size_cnt)

    # allocate larger demands first; only 3->4 is allowed as an exception
    for req in sorted(demand_size_cnt.keys(), reverse=True):
        need = int(demand_size_cnt[req])
        for _ in range(need):
            chosen = None
            if available[req] > 0:
                chosen = req
            elif req == 3 and available[4] > 0:
                chosen = 4
            else:
                return False
            available[chosen] -= 1
    return True

def verify_target_matches_milp(target: "ClusterState", milp_res: Dict[str, Any], verbose: bool = True):
    target_gpu_count = len(target.real_gpus())
    milp_gpu_count = len(extract_template_list_from_milp(milp_res))
    ok_gpu = (target_gpu_count == milp_gpu_count)

    target_cnt_exact = _collect_instance_multiset_from_target(target)
    milp_cnt = _collect_instance_multiset_from_milp(milp_res)
    ok_exact = (target_cnt_exact == milp_cnt)

    target_grouped = _group_multiset_by_workload_batch(target_cnt_exact)
    milp_grouped = _group_multiset_by_workload_batch(milp_cnt)

    ok_relaxed = True
    if set(target_grouped.keys()) != set(milp_grouped.keys()):
        ok_relaxed = False
    else:
        for key in milp_grouped:
            if not _relaxed_cover_ok_for_one_group(target_grouped[key], milp_grouped[key]):
                ok_relaxed = False
                break

    if verbose:
        print("=" * 80)
        print("[VERIFY] target vs milp")
        print("=" * 80)
        print(f"GPU count match              : {ok_gpu}  (target={target_gpu_count}, milp={milp_gpu_count})")
        print(f"Instance multiset exact      : {ok_exact}")
        print(f"Instance multiset relaxed    : {ok_relaxed}")
        if not ok_exact:
            print("\nTarget multiset:", target_cnt_exact)
            print("MILP multiset  :", milp_cnt)

    return ok_gpu and ok_relaxed


### Code Block 3: MILP extraction helpers

In [184]:
# ============================================================
# [Transition Cell 3] MILP extraction helpers
# Role:
#   - convert MILP result into template list / instance demands / arrivals
#   - expand aggregated counts into per-instance demand objects
# ============================================================

def milp_expand_template_list(y_sol: Dict[int, int], templates=TEMPLATES) -> List[str]:
    out = []
    for t_idx, cnt in sorted(y_sol.items()):
        if int(cnt) <= 0:
            continue
        tpl_name = templates[int(t_idx)][0]
        out.extend([tpl_name] * int(cnt))
    return out

def extract_template_list_from_milp(milp_res: Dict[str, Any]) -> List[str]:
    if "chosen_templates" in milp_res and milp_res["chosen_templates"] is not None:
        return list(milp_res["chosen_templates"])
    if "y_sol" in milp_res and milp_res["y_sol"] is not None:
        return milp_expand_template_list(milp_res["y_sol"], TEMPLATES)
    raise ValueError("Cannot extract templates from milp_res")

def extract_instance_demands_from_milp(
    milp_res: Dict[str, Any],
    feasible_option_df_: Optional[Any] = None,
) -> List[Dict[str, Any]]:
    if feasible_option_df_ is None:
        feasible_option_df_ = feasible_option_df

    if "x_sol" not in milp_res or milp_res["x_sol"] is None:
        raise ValueError("milp_res does not contain x_sol")

    x_sol = milp_res["x_sol"]
    chosen = feasible_option_df_[feasible_option_df_["opt_idx"].isin(list(x_sol.keys()))].copy()

    agg = defaultdict(lambda: {"count": 0, "mu": None})
    for _, row in chosen.iterrows():
        opt_idx = int(row["opt_idx"])
        cnt = int(x_sol.get(opt_idx, 0))
        if cnt <= 0:
            continue
        key = (str(row["workload"]), str(row["profile"]), int(row["batch"]))
        agg[key]["count"] += cnt
        agg[key]["mu"] = float(row["mu"])

    out = []
    for (w, p, b), info in sorted(agg.items()):
        out.append({
            "workload": w,
            "profile": p,
            "batch": int(b),
            "count": int(info["count"]),
            "mu": float(info["mu"]),
        })
    return out

def _arrival_dict_from_milp(milp_res: Dict[str, Any]) -> Dict[str, float]:
    arr = {}
    if "arrival_rate" in milp_res and milp_res["arrival_rate"] is not None:
        vec = milp_res["arrival_rate"]
    else:
        vec = arrival_rate
    for i, w in enumerate(WORKLOAD_NAMES):
        arr[w] = float(vec[i])
    return arr

def _profile_need_from_instance_demands(instance_demands: List[Dict[str, Any]]) -> Dict[str, int]:
    c = Counter({p: 0 for p in PROFILE_ORDER})
    for d in instance_demands:
        c[d["profile"]] += int(d["count"])
    return dict(c)

def _expand_demands_with_ids(instance_demands: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    did = 0
    for d in instance_demands:
        for _ in range(int(d["count"])):
            out.append({
                "demand_id": did,
                "workload": d["workload"],
                "profile": d["profile"],
                "batch": int(d["batch"]),
                "mu": float(d["mu"]),
            })
            did += 1

    # harder-first ordering
    out.sort(key=lambda x: (-PROFILE_SIZE[x["profile"]], x["workload"], -x["batch"], x["demand_id"]))
    return out

### Code Block 4: Template-set search (abstract level)


In [185]:
# ============================================================
# [Transition Cell 4] Template-set search (abstract level)
# Role:
#   - generate candidate abstract template multisets satisfying profile_need
#   - assign chosen templates to concrete GPU ids
# ============================================================

def _prev_real_template_list(prev_state: Optional["ClusterState"]) -> List[str]:
    if prev_state is None:
        return []
    return [g.template_str() for g in prev_state.real_gpus()]

def _template_usefulness_score(
    tpl: str,
    profile_need: Dict[str, int],
    prev_templates_counter: Counter,
    milp_templates_counter: Counter,
):
    cap = TEMPLATE_NAME_TO_K[tpl]
    cover = sum(min(cap[p], profile_need.get(p, 0)) * PROFILE_SIZE[p] for p in PROFILE_ORDER)
    return (
        1000 * (prev_templates_counter[tpl] > 0),
        500 * (milp_templates_counter[tpl] > 0),
        cover,
        -len(_template_to_parts(tpl)),
        tpl,
    )

def _dominates_need(cur_cap: Dict[str, int], profile_need: Dict[str, int]) -> bool:
    return all(cur_cap.get(p, 0) >= profile_need.get(p, 0) for p in PROFILE_ORDER)

def _enumerate_candidate_abstract_template_sets(
    gpu_count: int,
    profile_need: Dict[str, int],
    milp_template_ref: List[str],
    prev_state: Optional["ClusterState"],
    max_candidates: int = 64,
) -> List[List[str]]:
    all_templates = _template_name_list_from_globals()
    prev_templates = _prev_real_template_list(prev_state)
    prev_counter = Counter(prev_templates)
    milp_counter = Counter(milp_template_ref)

    pool = []
    seen = set()
    for tpl in prev_templates + milp_template_ref + all_templates:
        if tpl in TEMPLATE_NAME_TO_K and tpl not in seen:
            seen.add(tpl)
            pool.append(tpl)

    pool.sort(
        key=lambda tpl: _template_usefulness_score(
            tpl, profile_need, prev_counter, milp_counter
        ),
        reverse=True,
    )

    max_per_gpu = {p: 0 for p in PROFILE_ORDER}
    for tpl in pool:
        cap = TEMPLATE_NAME_TO_K[tpl]
        for p in PROFILE_ORDER:
            max_per_gpu[p] = max(max_per_gpu[p], cap[p])

    candidates = []

    def optimistic_reachable(cur_cap: Dict[str, int], remaining_gpus: int) -> bool:
        for p in PROFILE_ORDER:
            if cur_cap[p] + remaining_gpus * max_per_gpu[p] < profile_need[p]:
                return False
        return True

    def dfs(pos: int, chosen: List[str], cur_cap: Dict[str, int], start_idx: int):
        if len(candidates) >= max_candidates * 8:
            return
        if pos == gpu_count:
            if _dominates_need(cur_cap, profile_need):
                candidates.append(list(chosen))
            return
        if not optimistic_reachable(cur_cap, gpu_count - pos):
            return

        for idx in range(start_idx, len(pool)):
            tpl = pool[idx]
            cap = TEMPLATE_NAME_TO_K[tpl]
            nxt = dict(cur_cap)
            for p in PROFILE_ORDER:
                nxt[p] += cap[p]
            chosen.append(tpl)
            dfs(pos + 1, chosen, nxt, idx)
            chosen.pop()

    dfs(0, [], {p: 0 for p in PROFILE_ORDER}, 0)

    if not candidates:
        raise RuntimeError("No abstract template multiset can satisfy profile_need")

    def candidate_rank(cand):
        c = Counter(cand)
        over = sum(max(0, sum(TEMPLATE_NAME_TO_K[t][p] * c[t] for t in c) - profile_need[p]) for p in PROFILE_ORDER)
        return (
            sum(min(c[t], prev_counter[t]) for t in c),
            sum(min(c[t], milp_counter[t]) for t in c),
            -over,
            sorted(cand),
        )

    candidates.sort(key=candidate_rank, reverse=True)

    out = []
    seen2 = set()
    for cand in candidates:
        key = tuple(sorted(cand))
        if key in seen2:
            continue
        seen2.add(key)
        out.append(list(cand))
        if len(out) >= max_candidates:
            break
    return out

# Front-move the rewrite families that change abstract template semantics.
UPGRADE_REWRITE_TEMPLATE_MAP = {
    "3+3": "4+3",
    "3+2+1": "4+2+1",
    "3+1+1+1": "4+1+1+1",
}

def _augment_candidate_abstract_template_sets(
    candidate_sets: List[List[str]],
    milp_template_ref: List[str],
    prev_state: Optional["ClusterState"],
    max_candidates: int = 64,
) -> List[List[str]]:
    prev_templates = _prev_real_template_list(prev_state)
    prev_counter = Counter(prev_templates)
    milp_counter = Counter(milp_template_ref)

    augmented = []
    seen = set()

    def add(cand: List[str]):
        key = tuple(sorted(cand))
        if key in seen:
            return
        seen.add(key)
        augmented.append(list(cand))

    for cand in candidate_sets:
        add(cand)

    # For each native candidate, derive a few upgrade-aware candidates that can
    # preserve existing 4g-bearing templates across transitions.
    for cand in candidate_sets:
        cnt = Counter(cand)
        for base_tpl, up_tpl in UPGRADE_REWRITE_TEMPLATE_MAP.items():
            anchor = max(prev_counter.get(up_tpl, 0), milp_counter.get(up_tpl, 0))
            if cnt.get(base_tpl, 0) <= 0 or anchor <= 0:
                continue
            limit = min(cnt[base_tpl], anchor)
            for k in range(1, limit + 1):
                new_cand = list(cand)
                changed = 0
                for i, tpl in enumerate(new_cand):
                    if tpl == base_tpl and changed < k:
                        new_cand[i] = up_tpl
                        changed += 1
                add(new_cand)
                if len(augmented) >= max_candidates * 3:
                    break
            if len(augmented) >= max_candidates * 3:
                break
        if len(augmented) >= max_candidates * 3:
            break

    def aug_rank(cand: List[str]):
        c = Counter(cand)
        upgrade_cnt = sum(c.get(up_tpl, 0) for up_tpl in UPGRADE_REWRITE_TEMPLATE_MAP.values())
        return (
            sum(min(c[t], prev_counter[t]) for t in c),
            sum(min(c[t], milp_counter[t]) for t in c),
            upgrade_cnt,
            -sum(len(_template_to_parts(t)) for t in cand),
            tuple(sorted(cand)),
        )

    augmented.sort(key=aug_rank, reverse=True)
    return augmented[:max_candidates]

def _order_candidate_templates_for_gpu_ids(
    candidate_templates: List[str],
    gpu_count: int,
    prev_state: Optional["ClusterState"],
    milp_template_ref: List[str],
) -> List[str]:
    """
    Greedy O(G^2) assignment:
      - prefer matching old GPU template first
      - then matching MILP ref template on the same index
      - else use largest / more useful template
    """
    remain = list(candidate_templates)
    prev_templates = _prev_real_template_list(prev_state)
    ordered = []

    for gid in range(gpu_count):
        pick = None
        if gid < len(prev_templates) and prev_templates[gid] in remain:
            pick = prev_templates[gid]
        elif gid < len(milp_template_ref) and milp_template_ref[gid] in remain:
            pick = milp_template_ref[gid]
        else:
            remain.sort(key=lambda t: (len(_template_to_parts(t)), t))
            pick = remain[0]
        ordered.append(pick)
        remain.remove(pick)

    return ordered


### Code Block 5: Abstract -> physical expansion


In [186]:
# ============================================================
# [Transition Cell 5] Abstract -> physical expansion
# Role:
#   - enumerate physical layouts for each abstract template
#   - score physical layouts against prev_state
#   - enumerate top physical-layout combinations for whole cluster
# ============================================================

def _get_prev_gpu_intervals(prev_state: Optional["ClusterState"], gpu_id: int):
    if prev_state is None:
        return []
    for g in prev_state.real_gpus():
        if g.gpu_id == gpu_id:
            return [(inst.start, inst.end, inst.profile) for inst in sorted(g.instances, key=lambda x: (x.start, x.end))]
    return []

def _layout_score_vs_prev(intervals: List[Tuple[int, int, str]], prev_intervals: List[Tuple[int, int, str]]):
    """
    4-tuple proxy:
      0) exact interval-profile matches
      1) same number of non-void intervals
      2) prefix matches by position
      3) total overlapping same-profile length
    """
    exact = 0
    prefix = 0
    overlap_same_profile = 0

    new_nv = [(s, e, p) for (s, e, p) in intervals if p != "void"]
    old_nv = [(s, e, p) for (s, e, p) in prev_intervals if p != "void"]

    for a in new_nv:
        for b in old_nv:
            if a == b:
                exact += 1

    for i in range(min(len(new_nv), len(old_nv))):
        if new_nv[i] == old_nv[i]:
            prefix += 1

    for s1, e1, p1 in new_nv:
        for s2, e2, p2 in old_nv:
            if p1 != p2:
                continue
            overlap = max(0, min(e1, e2) - max(s1, s2))
            overlap_same_profile += overlap

    return (
        int(exact),
        int(len(new_nv) == len(old_nv)),
        int(prefix),
        int(overlap_same_profile),
    )

def _get_physical_layout_candidates_for_gpu(
    abstract_template: str,
    gpu_id: int,
    prev_state: Optional["ClusterState"],
    topk: int = 4,
):
    prev_intervals = _get_prev_gpu_intervals(prev_state, gpu_id)
    cands = []
    for pstr, intervals in _all_unique_physical_realizations(abstract_template):
        score = _layout_score_vs_prev(intervals, prev_intervals)
        cands.append((pstr, intervals, score))
    cands.sort(key=lambda x: (x[2], x[0]), reverse=True)
    return cands[:topk]

def _enumerate_physical_layout_combinations(
    ordered_abstract_templates: List[str],
    prev_state: Optional["ClusterState"],
    milp_template_ref: List[str],
    max_combos: int = 32,
    per_gpu_topk: int = 4,
):
    per_gpu_candidates = []
    for gid, atpl in enumerate(ordered_abstract_templates):
        per_gpu_candidates.append(
            _get_physical_layout_candidates_for_gpu(
                abstract_template=atpl,
                gpu_id=gid,
                prev_state=prev_state,
                topk=per_gpu_topk,
            )
        )

    combos = []

    def dfs(gid, chosen_pstr, chosen_intervals, score_acc):
        if len(combos) >= max_combos * 8:
            return
        if gid == len(ordered_abstract_templates):
            combos.append((score_acc, list(chosen_pstr), list(chosen_intervals)))
            return
        for pstr, intervals, lscore in per_gpu_candidates[gid]:
            nxt = (
                score_acc[0] + lscore[0],
                score_acc[1] + lscore[1],
                score_acc[2] + lscore[2],
                score_acc[3] + lscore[3],
            )
            chosen_pstr.append(pstr)
            chosen_intervals.append(intervals)
            dfs(gid + 1, chosen_pstr, chosen_intervals, nxt)
            chosen_pstr.pop()
            chosen_intervals.pop()

    dfs(0, [], [], (0, 0, 0, 0))
    combos.sort(key=lambda x: (x[0], x[1]), reverse=True)

    out = []
    seen = set()
    for score_acc, pstrs, intervals in combos:
        key = tuple(pstrs)
        if key in seen:
            continue
        seen.add(key)
        out.append({
            "physical_template_strs": list(pstrs),
            "intervals_list": list(intervals),
            "layout_score": tuple(score_acc),
        })
        if len(out) >= max_combos:
            break
    return out

def _make_slot_list_from_intervals_list(intervals_list: List[List[Tuple[int, int, str]]]) -> List[Dict[str, Any]]:
    slots = []
    sid = 0
    for gid, intervals in enumerate(intervals_list):
        for local_idx, (s, e, p) in enumerate(intervals):
            if p == "void":
                continue
            slots.append({
                "slot_id": sid,
                "gpu_id": gid,
                "slot_idx": local_idx,
                "start": s,
                "end": e,
                "profile": p,
            })
            sid += 1
    return slots


### Code Block 6: Common scoring / exact-preserve helpers


In [187]:
# ============================================================
# [Transition Cell 6] Common scoring / preserve helpers
# Role:
#   - compute preserve / spread / collocation / mixed-GPU metrics
#   - build ClusterState from slot assignments
#   - provide unified score tuple used by both solvers
# Preserve definition:
#   exact preserve:
#       same slot + same workload + same profile
#       (batch may differ)
#   upgrade preserve:
#       native 3g demand stays on an old 4g slot with the same workload
#       when the candidate has surplus 4g over native 4g need
# Extra:
#   final rewrite is still kept as a legalization fallback:
#     3+3       -> 4+3
#     3+2+1     -> choose among 4+2+1 / 1+1+2+3 / 2+1+1+3
#     3+1+1+1   -> choose among 4+1+1+1 / 1+1+1+1+3
# ============================================================

from functools import lru_cache

_OLD_SLOT_MAP_CACHE = {}
_PREV_INTERVALS_CACHE = {}

def _has_prev(prev_state: Optional["ClusterState"]):
    return prev_state is not None and len(prev_state.real_gpus()) > 0

def _old_exact_slot_map(prev_state: Optional["ClusterState"]):
    if prev_state is None:
        return {}
    key = id(prev_state)
    if key in _OLD_SLOT_MAP_CACHE:
        return _OLD_SLOT_MAP_CACHE[key]
    mp = {}
    for g in prev_state.real_gpus():
        for inst in g.instances:
            if inst.profile == "void":
                continue
            mp[(g.gpu_id, inst.start, inst.end, inst.profile)] = inst
    _OLD_SLOT_MAP_CACHE[key] = mp
    return mp

def _template_match_count(ordered_abstract_templates, ordered_physical_templates, prev_state):
    prev_templates = _prev_real_template_list(prev_state)
    if len(prev_templates) != len(ordered_abstract_templates):
        return 0
    c = 0
    for i in range(len(ordered_abstract_templates)):
        if i < len(prev_templates) and (
            prev_templates[i] == ordered_abstract_templates[i] or prev_templates[i] == ordered_physical_templates[i]
        ):
            c += 1
    return c

def _slot_preserve_match(slot: Dict[str, Any], demand: Dict[str, Any], prev_state: Optional["ClusterState"], old_map=None) -> bool:
    if old_map is None:
        old_map = _old_exact_slot_map(prev_state)
    old = old_map.get((slot["gpu_id"], slot["start"], slot["end"], slot["profile"]))
    if old is None:
        return False
    return old.workload == demand["workload"] and old.profile == demand["profile"]

def _slot_upgrade_preserve_match(slot: Dict[str, Any], demand: Dict[str, Any], prev_state: Optional["ClusterState"], old_map=None) -> bool:
    if demand["profile"] != "3g" or slot["profile"] != "4g":
        return False
    if old_map is None:
        old_map = _old_exact_slot_map(prev_state)
    old = old_map.get((slot["gpu_id"], slot["start"], slot["end"], slot["profile"]))
    if old is None:
        return False
    return old.workload == demand["workload"] and old.profile == "4g"

def _inst_preserve_match(inst: "MigInstance", gpu_id: int, prev_state: Optional["ClusterState"], old_map=None) -> bool:
    if old_map is None:
        old_map = _old_exact_slot_map(prev_state)
    old = old_map.get((gpu_id, inst.start, inst.end, inst.profile))
    if old is None or inst.workload is None:
        return False
    return old.workload == inst.workload and old.profile == inst.profile

def _assignments_to_metrics(
    slots: List[Dict[str, Any]],
    assigned: Dict[int, Optional[Dict[str, Any]]],
    ordered_abstract_templates: List[str],
    ordered_physical_templates: List[str],
    layout_preserve_score: Tuple,
    prev_state: Optional["ClusterState"],
    old_map=None,
):
    if old_map is None:
        old_map = _old_exact_slot_map(prev_state)

    exact_preserve = 0
    upgrade_preserve = 0
    workload_gpus = defaultdict(set)
    per_gpu_workload_count = defaultdict(Counter)

    for sl in slots:
        info = assigned.get(sl["slot_id"], None)
        if info is None:
            continue
        d = info["demand"]
        workload_gpus[d["workload"]].add(sl["gpu_id"])
        per_gpu_workload_count[sl["gpu_id"]][d["workload"]] += 1

        if info.get("placement_mode") == "upgrade_preserve":
            upgrade_preserve += 1
            continue

        old = old_map.get((sl["gpu_id"], sl["start"], sl["end"], sl["profile"]))
        if old is not None and old.workload == d["workload"] and old.profile == d["profile"]:
            exact_preserve += 1

    spread = sum(len(v) for v in workload_gpus.values())

    collocate_pairs = 0
    mixed_gpu_count = 0
    for gid, cnt in per_gpu_workload_count.items():
        kinds = sum(1 for _, v in cnt.items() if v > 0)
        if kinds >= 2:
            mixed_gpu_count += 1
        for _, n in cnt.items():
            collocate_pairs += n * (n - 1) // 2

    return {
        "ordered_abstract_templates": list(ordered_abstract_templates),
        "ordered_physical_templates": list(ordered_physical_templates),
        "layout_preserve_score": tuple(layout_preserve_score),
        "exact_preserve": int(exact_preserve),
        "upgrade_preserve": int(upgrade_preserve),
        "same_gpu_preserve": 0,
        "spread": int(spread),
        "collocate_pairs": int(collocate_pairs),
        "mixed_gpu_count": int(mixed_gpu_count),
        "template_match_count": int(_template_match_count(
            ordered_abstract_templates,
            ordered_physical_templates,
            prev_state,
        )),
    }

def _score_tuple(metrics: Dict[str, Any], prev_mode: bool):
    if prev_mode:
        return (
            metrics["exact_preserve"],
            metrics["upgrade_preserve"],
            -metrics["spread"],
            metrics["collocate_pairs"],
            -metrics["mixed_gpu_count"],
            metrics["layout_preserve_score"][0],
            metrics["layout_preserve_score"][1],
            metrics["template_match_count"],
        )
    else:
        return (
            -metrics["spread"],
            metrics["collocate_pairs"],
            -metrics["mixed_gpu_count"],
            metrics["layout_preserve_score"][0],
            metrics["template_match_count"],
        )

# ---------- layout rewrite helpers ----------

VOID_LIKE_REWRITE_CANDIDATES = {
    # keep upgrade-style rewrites as legalization fallback as well.
    # They are front-moved into candidate augmentation for scoring/search,
    # but if a surviving candidate still materializes a 6-slice template,
    # we must legalize it to a full 7-slice layout before validation.
    "3+3": [("4g", "3g")],
    "3+2+1": [("4g", "2g", "1g"), ("1g", "1g", "2g", "3g"), ("2g", "1g", "1g", "3g")],
    "3+1+1+1": [("4g", "1g", "1g", "1g"), ("1g", "1g", "1g", "1g", "3g")],
}

def _profiles_string_from_instances(insts: List["MigInstance"]) -> str:
    return "+".join(str(PROFILE_SIZE[inst.profile]) for inst in sorted(insts, key=lambda x: (x.start, x.end)) if inst.profile != "void")

def _candidate_priority_no_prev(current_t: str, profiles: Tuple[str, ...]) -> Tuple[int, ...]:
    s = _physical_profiles_to_string(profiles)
    if current_t == "3+2+1":
        order = {"1+1+2+3": 0, "2+1+1+3": 1}
        return (order.get(s, 99),)
    if current_t == "3+1+1+1":
        order = {"1+1+1+1+3": 0}
        return (order.get(s, 99),)
    return (99,)

def _build_intervals_from_profiles(profiles: Tuple[str, ...]) -> List[Tuple[int, int, str]]:
    return _physical_profiles_to_intervals(profiles)

def _get_prev_gpu_intervals_cached(prev_state: Optional["ClusterState"], gpu_id: int):
    key = (id(prev_state), gpu_id)
    if key in _PREV_INTERVALS_CACHE:
        return _PREV_INTERVALS_CACHE[key]
    val = _get_prev_gpu_intervals(prev_state, gpu_id)
    _PREV_INTERVALS_CACHE[key] = val
    return val

def _assign_instances_to_candidate_layout(
    gpu_id: int,
    old_assigned_insts: List["MigInstance"],
    candidate_profiles: Tuple[str, ...],
    prev_state: Optional["ClusterState"],
    prefer_first_1g_idle: bool,
    old_map=None,
):
    """
    Remap currently assigned workloads on this GPU to a rewritten legal layout.
    This final step only handles order-style legalization among already-selected layouts.
    """
    if old_map is None:
        old_map = _old_exact_slot_map(prev_state)
    intervals = _build_intervals_from_profiles(candidate_profiles)
    candidate_slots = []
    oneg_rank = 0
    for (s, e, p) in intervals:
        if p == "void":
            continue
        if p == "1g":
            oneg_rank += 1
            rank = oneg_rank
        else:
            rank = 0
        candidate_slots.append({
            "start": s, "end": e, "profile": p, "size": PROFILE_SIZE[p], "oneg_rank": rank
        })

    assigned_work = [inst for inst in sorted(old_assigned_insts, key=lambda x: (-PROFILE_SIZE[x.profile], x.start, x.end)) if inst.workload is not None]
    slot_used = [False] * len(candidate_slots)
    new_insts = []

    for inst in assigned_work:
        compat = []
        for idx, sl in enumerate(candidate_slots):
            if slot_used[idx]:
                continue
            if sl["size"] < PROFILE_SIZE[inst.profile]:
                continue

            preserve = 0
            if prev_state is not None:
                old = old_map.get((gpu_id, sl["start"], sl["end"], sl["profile"]))
                if old is not None and old.workload == inst.workload and old.profile == inst.profile:
                    preserve = 1

            exact_profile_fit = int(sl["profile"] == inst.profile)
            size_waste = sl["size"] - PROFILE_SIZE[inst.profile]

            first_1g_penalty = 0
            if prev_state is None and prefer_first_1g_idle and inst.profile == "1g" and sl["profile"] == "1g":
                if sl["oneg_rank"] == 1:
                    first_1g_penalty = 1

            compat.append((
                preserve,
                exact_profile_fit,
                -size_waste,
                -first_1g_penalty,
                -sl["start"],
                -idx,
                idx,
            ))

        if not compat:
            return None

        compat.sort(reverse=True)
        chosen_idx = compat[0][-1]
        sl = candidate_slots[chosen_idx]
        slot_used[chosen_idx] = True

        new_insts.append(
            MigInstance(
                start=sl["start"],
                end=sl["end"],
                profile=sl["profile"],
                workload=inst.workload,
                batch=inst.batch,
                mu=inst.mu,
                preserved=False,
            )
        )

    for idx, sl in enumerate(candidate_slots):
        if not slot_used[idx]:
            new_insts.append(
                MigInstance(
                    start=sl["start"],
                    end=sl["end"],
                    profile=sl["profile"],
                    workload=None,
                    batch=None,
                    mu=0.0,
                    preserved=False,
                )
            )

    new_insts = sorted(new_insts, key=lambda x: (x.start, x.end))
    cur = 0
    completed = []
    for inst in new_insts:
        if inst.start > cur:
            completed.append(MigInstance(start=cur, end=inst.start, profile="void", workload=None, batch=None, mu=0.0, preserved=False))
        completed.append(inst)
        cur = inst.end
    if cur < 7:
        completed.append(MigInstance(start=cur, end=7, profile="void", workload=None, batch=None, mu=0.0, preserved=False))

    for inst in completed:
        if inst.workload is None or inst.profile == "void":
            inst.preserved = False
        else:
            inst.preserved = _inst_preserve_match(inst, gpu_id, prev_state, old_map=old_map)
    return completed

def _rewrite_void_like_layout_for_gpu(
    gpu_id: int,
    insts: List["MigInstance"],
    prev_state: Optional["ClusterState"],
):
    old_map = _old_exact_slot_map(prev_state)
    current_t = _profiles_string_from_instances(insts)
    if current_t not in VOID_LIKE_REWRITE_CANDIDATES:
        return sorted(insts, key=lambda x: (x.start, x.end))

    candidates = VOID_LIKE_REWRITE_CANDIDATES[current_t]
    prev_intervals = _get_prev_gpu_intervals_cached(prev_state, gpu_id)

    best = None
    best_key = None

    for cand_profiles in candidates:
        prefer_first_1g_idle = (prev_state is None and current_t in {"3+2+1", "3+1+1+1"})
        rewritten = _assign_instances_to_candidate_layout(
            gpu_id=gpu_id,
            old_assigned_insts=insts,
            candidate_profiles=cand_profiles,
            prev_state=prev_state,
            prefer_first_1g_idle=prefer_first_1g_idle,
            old_map=old_map,
        )
        if rewritten is None:
            continue

        preserve_cnt = sum(
            1 for inst in rewritten
            if inst.workload is not None and inst.profile != "void" and _inst_preserve_match(inst, gpu_id, prev_state, old_map=old_map)
        )

        intervals = [(inst.start, inst.end, inst.profile) for inst in rewritten]
        layout_score = _layout_score_vs_prev(intervals, prev_intervals)

        if prev_state is None:
            key = (0,) + tuple(-x for x in _candidate_priority_no_prev(current_t, cand_profiles))
        else:
            key = (
                preserve_cnt,
                layout_score[0],
                layout_score[1],
                layout_score[2],
                layout_score[3],
            )

        if best is None or key > best_key:
            best = rewritten
            best_key = key

    if best is None:
        return sorted(insts, key=lambda x: (x.start, x.end))
    return sorted(best, key=lambda x: (x.start, x.end))

def _assignments_to_target(
    slots: List[Dict[str, Any]],
    assigned: Dict[int, Optional[Dict[str, Any]]],
    ordered_abstract_templates: List[str],
    ordered_physical_templates: List[str],
    layout_preserve_score: Tuple,
    prev_state: Optional["ClusterState"],
):
    old_map = _old_exact_slot_map(prev_state)
    by_gpu = defaultdict(list)

    for sl in slots:
        info = assigned.get(sl["slot_id"], None)
        workload, batch, mu = None, None, 0.0
        preserved = False

        if info is not None:
            d = info["demand"]
            workload = d["workload"]
            batch = int(d["batch"])
            mu = float(d["mu"])
            preserved = _slot_preserve_match(sl, d, prev_state, old_map=old_map) or (info.get("placement_mode") == "upgrade_preserve")

        by_gpu[sl["gpu_id"]].append(
            MigInstance(
                start=sl["start"], end=sl["end"], profile=sl["profile"],
                workload=workload, batch=batch, mu=mu, preserved=preserved
            )
        )

    gpus = []
    gpu_ids = sorted({sl["gpu_id"] for sl in slots})
    actual_physical_templates = []

    for gid in gpu_ids:
        insts = sorted(by_gpu[gid], key=lambda x: (x.start, x.end))
        rewritten = _rewrite_void_like_layout_for_gpu(gid, insts, prev_state)
        gpus.append(GPUState(gpu_id=gid, source="real", instances=rewritten))
        actual_physical_templates.append(_profiles_string_from_instances(rewritten))

    target = ClusterState(gpus=gpus, metadata={})
    assert_valid_cluster_state(target)

    metrics = _assignments_to_metrics(
        slots=slots,
        assigned=assigned,
        ordered_abstract_templates=ordered_abstract_templates,
        ordered_physical_templates=actual_physical_templates,
        layout_preserve_score=layout_preserve_score,
        prev_state=prev_state,
        old_map=old_map,
    )
    metrics["ordered_physical_templates"] = list(actual_physical_templates)
    return target, metrics

def _exact_preserve_preassign(
    demands: List[Dict[str, Any]],
    slots: List[Dict[str, Any]],
    prev_state: Optional["ClusterState"],
    native_profile_need: Dict[str, int],
):
    """
    Hard-fix preserve in two phases:
      (1) exact preserve using same slot + same workload + same profile
      (2) upgrade-preserve allowing a native 3g demand to stay on an old 4g slot
          when current candidate has surplus 4g over native 4g need
    Batch does not matter.
    """
    old_map = _old_exact_slot_map(prev_state)

    demand_buckets = defaultdict(list)
    for d in demands:
        demand_buckets[(d["workload"], d["profile"])].append(d)

    for k in demand_buckets:
        demand_buckets[k].sort(key=lambda d: (int(d["batch"]), int(d["demand_id"])))

    assigned = {sl["slot_id"]: None for sl in slots}
    preserved_ids = set()

    # phase 1: exact preserve
    for sl in slots:
        old = old_map.get((sl["gpu_id"], sl["start"], sl["end"], sl["profile"]))
        if old is None or old.workload is None:
            continue
        key = (old.workload, old.profile)
        if len(demand_buckets[key]) == 0:
            continue
        d = demand_buckets[key].pop(0)
        assigned[sl["slot_id"]] = {"slot": sl, "demand": d, "placement_mode": "exact_preserve"}
        preserved_ids.add(d["demand_id"])

    # phase 2: native 3g -> old 4g exact-slot preserve, limited by surplus 4g.
    total_4g_slots = sum(1 for sl in slots if sl["profile"] == "4g")
    native_need_4g = int(native_profile_need.get("4g", 0))
    upgrade_budget = max(0, total_4g_slots - native_need_4g)

    if upgrade_budget > 0 and _has_prev(prev_state):
        for sl in slots:
            if upgrade_budget <= 0:
                break
            if sl["profile"] != "4g" or assigned[sl["slot_id"]] is not None:
                continue
            old = old_map.get((sl["gpu_id"], sl["start"], sl["end"], sl["profile"]))
            if old is None or old.workload is None or old.profile != "4g":
                continue
            key = (old.workload, "3g")
            if len(demand_buckets[key]) == 0:
                continue
            d = demand_buckets[key].pop(0)
            assigned[sl["slot_id"]] = {"slot": sl, "demand": d, "placement_mode": "upgrade_preserve"}
            preserved_ids.add(d["demand_id"])
            upgrade_budget -= 1

    residual_demands = [d for d in demands if d["demand_id"] not in preserved_ids]
    return assigned, residual_demands

def _group_demands_by_workload_profile(demands):
    buckets = defaultdict(list)
    for d in demands:
        buckets[(d["workload"], d["profile"])].append(d)
    return buckets


########################################################################################################################


### Code Block 8: Greedy + move/swap local-repair solver


In [188]:
# ============================================================
# [Transition Cell 8] Greedy + move/swap local-repair solver
# Role:
#   - preserve preprocessing
#   - workload-aware greedy assignment
#   - local repair with two neighborhoods:
#       (1) move, (2) swap
# Greedy policy:
#   - with prev_state: first preserve same slot+workload+profile,
#     then try to place remaining instances onto GPUs that already
#     carry the same workload
#   - additionally, when the chosen candidate has surplus 4g over native
#     4g need, a native 3g demand may stay on an old 4g slot of the same
#     workload (upgrade-preserve)
#   - without prev_state: directly pack same workload together
# ============================================================

def _order_residual_demands_for_greedy(demands, slots, prev_state):
    old_map = _old_exact_slot_map(prev_state)

    def preserve_candidates(d):
        exact_cnt = 0
        upgrade_cnt = 0
        for sl in slots:
            if _slot_preserve_match(sl, d, prev_state, old_map=old_map):
                exact_cnt += 1
            elif _slot_upgrade_preserve_match(sl, d, prev_state, old_map=old_map):
                upgrade_cnt += 1
        return (exact_cnt, upgrade_cnt)

    ordered = list(demands)
    if _has_prev(prev_state):
        ordered.sort(key=lambda d: (-preserve_candidates(d)[0], -preserve_candidates(d)[1], d["workload"], -PROFILE_SIZE[d["profile"]], int(d["batch"]), int(d["demand_id"])))
    else:
        ordered.sort(key=lambda d: (d["workload"], -PROFILE_SIZE[d["profile"]], int(d["batch"]), int(d["demand_id"])))
    return ordered

def _template_combo_bonus(demand, slot, per_gpu_profiles):
    gpu_id = slot["gpu_id"]
    workload = demand["workload"]
    existing_profiles = per_gpu_profiles[gpu_id].get(workload, set())
    if len(existing_profiles) == 0:
        return 0
    prof = demand["profile"]
    return 1 if prof not in existing_profiles else 0

def _build_assignment_stats(assigned):
    per_gpu_workload_count = defaultdict(Counter)
    workload_gpus = defaultdict(set)
    per_gpu_profiles = defaultdict(lambda: defaultdict(set))
    for _, info in assigned.items():
        if info is None:
            continue
        sl = info["slot"]
        d = info["demand"]
        gid = sl["gpu_id"]
        w = d["workload"]
        per_gpu_workload_count[gid][w] += 1
        workload_gpus[w].add(gid)
        per_gpu_profiles[gid][w].add(d["profile"])
    return per_gpu_workload_count, workload_gpus, per_gpu_profiles

def _greedy_incremental_rank(demand, slot, assigned, prev_state, stats=None, old_map=None):
    if stats is None:
        per_gpu_workload_count, workload_gpus, per_gpu_profiles = _build_assignment_stats(assigned)
    else:
        per_gpu_workload_count, workload_gpus, per_gpu_profiles = stats
    if old_map is None:
        old_map = _old_exact_slot_map(prev_state)

    gpu_id = slot["gpu_id"]
    workload = demand["workload"]
    same_workload = per_gpu_workload_count[gpu_id][workload]
    new_touch = 1 if gpu_id not in workload_gpus[workload] else 0
    distinct_before = len(per_gpu_workload_count[gpu_id])
    mixed_penalty = 1 if (distinct_before >= 1 and same_workload == 0) else 0
    preserve_bonus = 1 if _slot_preserve_match(slot, demand, prev_state, old_map=old_map) else 0
    upgrade_bonus = 1 if _slot_upgrade_preserve_match(slot, demand, prev_state, old_map=old_map) else 0
    combo_bonus = _template_combo_bonus(demand, slot, per_gpu_profiles)

    if _has_prev(prev_state):
        return (
            preserve_bonus,
            upgrade_bonus,
            same_workload,
            -new_touch,
            combo_bonus,
            -mixed_penalty,
            -slot["slot_idx"],
        )
    else:
        return (
            same_workload,
            -new_touch,
            combo_bonus,
            -mixed_penalty,
            -slot["slot_idx"],
        )

def _solve_target_with_greedy_repair(
    demands: List[Dict[str, Any]],
    ordered_abstract_templates: List[str],
    ordered_physical_templates: List[str],
    intervals_list: List[List[Tuple[int, int, str]]],
    prev_state: Optional["ClusterState"],
    native_profile_need: Dict[str, int],
    layout_preserve_score: Tuple = (0, 0, 0, 0),
    repair_rounds: int = 8,
):
    slots = _make_slot_list_from_intervals_list(intervals_list)
    assigned, residual = _exact_preserve_preassign(demands, slots, prev_state, native_profile_need)
    old_map = _old_exact_slot_map(prev_state)

    free_by_profile = defaultdict(list)
    for sl in slots:
        free_by_profile[sl["profile"]].append(sl)

    # ----- greedy fill -----
    residual = _order_residual_demands_for_greedy(residual, slots, prev_state)
    per_gpu_workload_count, workload_gpus, per_gpu_profiles = _build_assignment_stats(assigned)

    total_4g_slots = len(free_by_profile["4g"])
    used_upgrade = sum(1 for _, info in assigned.items() if info is not None and info.get("placement_mode") == "upgrade_preserve")
    upgrade_budget_remaining = max(0, total_4g_slots - int(native_profile_need.get("4g", 0)) - used_upgrade)

    for d in residual:
        cand = [sl for sl in free_by_profile[d["profile"]] if assigned[sl["slot_id"]] is None]

        if d["profile"] == "3g" and upgrade_budget_remaining > 0 and _has_prev(prev_state):
            for sl in free_by_profile["4g"]:
                if assigned[sl["slot_id"]] is not None:
                    continue
                if _slot_upgrade_preserve_match(sl, d, prev_state, old_map=old_map):
                    cand.append(sl)

        # deduplicate by slot id
        dedup = {}
        for sl in cand:
            dedup[sl["slot_id"]] = sl
        cand = list(dedup.values())

        if not cand:
            raise RuntimeError(f"Greedy failed: no free slot for ({d['workload']}, {d['profile']}, B{d['batch']})")

        stats = (per_gpu_workload_count, workload_gpus, per_gpu_profiles)
        cand.sort(key=lambda sl: _greedy_incremental_rank(d, sl, assigned, prev_state, stats=stats, old_map=old_map), reverse=True)
        best = cand[0]

        placement_mode = "greedy"
        if d["profile"] == "3g" and best["profile"] == "4g":
            placement_mode = "upgrade_preserve"
            upgrade_budget_remaining -= 1

        assigned[best["slot_id"]] = {"slot": best, "demand": d, "placement_mode": placement_mode}
        gid = best["gpu_id"]
        w = d["workload"]
        per_gpu_workload_count[gid][w] += 1
        workload_gpus[w].add(gid)
        per_gpu_profiles[gid][w].add(d["profile"])

    # ----- local repair: move + swap -----
    prev_mode = _has_prev(prev_state)

    def is_preserved(sid, info):
        if info is None:
            return False
        return info.get("placement_mode") == "upgrade_preserve" or _slot_preserve_match(info["slot"], info["demand"], prev_state, old_map=old_map)

    cur_metrics = _assignments_to_metrics(
        slots=slots,
        assigned=assigned,
        ordered_abstract_templates=ordered_abstract_templates,
        ordered_physical_templates=ordered_physical_templates,
        layout_preserve_score=layout_preserve_score,
        prev_state=prev_state,
        old_map=old_map,
    )
    cur_score = _score_tuple(cur_metrics, prev_mode)

    slot_map = {sl["slot_id"]: sl for sl in slots}
    slot_ids = [sl["slot_id"] for sl in slots]

    for _ in range(repair_rounds):
        improved = False
        best_assign = dict(assigned)
        best_metrics = dict(cur_metrics)
        best_score = cur_score

        # (1) move neighborhood
        for src in slot_ids:
            src_info = assigned[src]
            if src_info is None or is_preserved(src, src_info):
                continue
            src_profile = src_info["slot"]["profile"]

            for dst in slot_ids:
                if assigned[dst] is not None:
                    continue
                if slot_map[dst]["profile"] != src_profile:
                    continue

                trial = dict(assigned)
                trial[dst] = {"slot": slot_map[dst], "demand": trial[src]["demand"], "placement_mode": trial[src].get("placement_mode", "greedy")}
                trial[src] = None

                m = _assignments_to_metrics(
                    slots=slots,
                    assigned=trial,
                    ordered_abstract_templates=ordered_abstract_templates,
                    ordered_physical_templates=ordered_physical_templates,
                    layout_preserve_score=layout_preserve_score,
                    prev_state=prev_state,
                    old_map=old_map,
                )
                s = _score_tuple(m, prev_mode)
                if s > best_score:
                    best_score = s
                    best_metrics = m
                    best_assign = trial
                    improved = True

        # (2) swap neighborhood
        for i in range(len(slot_ids)):
            sid1 = slot_ids[i]
            info1 = assigned[sid1]
            if info1 is None or is_preserved(sid1, info1):
                continue

            for j in range(i + 1, len(slot_ids)):
                sid2 = slot_ids[j]
                info2 = assigned[sid2]
                if info2 is None or is_preserved(sid2, info2):
                    continue
                if info1["slot"]["profile"] != info2["slot"]["profile"]:
                    continue

                trial = dict(assigned)
                slot1 = info1["slot"]
                slot2 = info2["slot"]

                trial[sid1] = {"slot": slot1, "demand": info2["demand"], "placement_mode": "greedy"}
                trial[sid2] = {"slot": slot2, "demand": info1["demand"], "placement_mode": "greedy"}

                m = _assignments_to_metrics(
                    slots=slots,
                    assigned=trial,
                    ordered_abstract_templates=ordered_abstract_templates,
                    ordered_physical_templates=ordered_physical_templates,
                    layout_preserve_score=layout_preserve_score,
                    prev_state=prev_state,
                    old_map=old_map,
                )
                s = _score_tuple(m, prev_mode)
                if s > best_score:
                    best_score = s
                    best_metrics = m
                    best_assign = trial
                    improved = True

        if not improved:
            break

        assigned = best_assign
        cur_metrics = best_metrics
        cur_score = best_score

    # Final materialization + rewrite only once
    return _assignments_to_target(
        slots=slots,
        assigned=assigned,
        ordered_abstract_templates=ordered_abstract_templates,
        ordered_physical_templates=ordered_physical_templates,
        layout_preserve_score=layout_preserve_score,
        prev_state=prev_state,
    )




### Code Block 7: Public API (greedy only)

In [189]:
# ============================================================
# [Transition Cell 7] Public API (greedy only)
# Role:
#   - expose one unified entrypoint:
#       build_target_state_from_milp(...)
#   - use greedy+repair only
# ============================================================



def _gpu_logical_template(g: "GPUState"):
    arr = [inst.profile for inst in sorted(g.instances, key=lambda x: (x.start, x.end)) if inst.profile != "void"]
    arr = sorted(arr, key=lambda p: (-PROFILE_SIZE[p], p))
    return tuple(arr)

def _gpu_physical_template(g: "GPUState"):
    return tuple(inst.profile for inst in sorted(g.instances, key=lambda x: (x.start, x.end)) if inst.profile != "void")

def _gpu_interval_profile_list(g: "GPUState"):
    return [(inst.start, inst.end, inst.profile) for inst in sorted(g.instances, key=lambda x: (x.start, x.end)) if inst.profile != "void"]

def _gpu_match_score(old_g: "GPUState", new_g: "GPUState"):
    old_log = _gpu_logical_template(old_g)
    new_log = _gpu_logical_template(new_g)
    old_phy = _gpu_physical_template(old_g)
    new_phy = _gpu_physical_template(new_g)

    score = 0

    # 1) logical template match (highest priority)
    if old_log == new_log:
        score += 1000

    # 2) physical template match (slot order / physical layout)
    if old_phy == new_phy:
        score += 200

    # 3) exact interval-profile matches + same-profile overlap
    old_iv = _gpu_interval_profile_list(old_g)
    new_iv = _gpu_interval_profile_list(new_g)

    exact = 0
    overlap_same_profile = 0
    for a in old_iv:
        for b in new_iv:
            if a == b:
                exact += 1
            if a[2] == b[2]:
                overlap_same_profile += max(0, min(a[1], b[1]) - max(a[0], b[0]))

    score += 20 * exact + overlap_same_profile

    # 4) profile-workload similarity (multiset-based, order-invariant)
    #    compare (profile, workload) instead of slot-aligned workload,
    #    because physical order can be fixed later by _apply_same_logical_template_order_fix().
    old_pw = Counter(
        (inst.profile, getattr(inst, "workload", None))
        for inst in old_g.instances
        if getattr(inst, "profile", None) not in (None, "void")
    )
    new_pw = Counter(
        (inst.profile, getattr(inst, "workload", None))
        for inst in new_g.instances
        if getattr(inst, "profile", None) not in (None, "void")
    )

    common_pw = 0
    all_keys = set(old_pw.keys()) | set(new_pw.keys())
    for k in all_keys:
        common_pw += min(old_pw.get(k, 0), new_pw.get(k, 0))

    # each matched (profile, workload)
    score += 30 * common_pw

    # full multiset match bonus
    if old_pw == new_pw:
        score += 300

    return score

def _reassign_gpu_ids_by_matching(target: "ClusterState", prev_state: Optional["ClusterState"]):
    if prev_state is None:
        target.metadata["display_id_map"] = {g.gpu_id: i for i, g in enumerate(sorted(target.real_gpus(), key=lambda x: x.gpu_id))}
        return target

    old_gpus = list(sorted(prev_state.real_gpus(), key=lambda x: x.gpu_id))
    new_gpus = list(sorted(target.real_gpus(), key=lambda x: x.gpu_id))
    if not new_gpus:
        target.metadata["display_id_map"] = {}
        return target

    pairs = []
    for oi, og in enumerate(old_gpus):
        for nj, ng in enumerate(new_gpus):
            pairs.append((_gpu_match_score(og, ng), oi, nj))
    pairs.sort(reverse=True)

    matched_old = set()
    matched_new = set()
    assignment = {}
    for sc, oi, nj in pairs:
        if oi in matched_old or nj in matched_new:
            continue
        matched_old.add(oi)
        matched_new.add(nj)
        assignment[nj] = old_gpus[oi].gpu_id

    used_ids = set(assignment.values())
    next_id = max([g.gpu_id for g in old_gpus], default=-1) + 1
    for nj, ng in enumerate(new_gpus):
        if nj not in assignment:
            while next_id in used_ids:
                next_id += 1
            assignment[nj] = next_id
            used_ids.add(next_id)
            next_id += 1

    old_to_new = {}
    for nj, ng in enumerate(new_gpus):
        old_to_new[ng.gpu_id] = assignment[nj]

    for g in target.real_gpus():
        new_id = old_to_new[g.gpu_id]
        g.gpu_id = new_id
        for inst in g.instances:
            inst.gpu_id = new_id

    target.gpus = sorted(target.gpus, key=lambda x: (getattr(x, "source", "real"), x.gpu_id))
    target.metadata["display_id_map"] = {g.gpu_id: i for i, g in enumerate(sorted(target.real_gpus(), key=lambda x: x.gpu_id))}
    return target

def _apply_same_logical_template_order_fix(target: "ClusterState", prev_state: Optional["ClusterState"]):
    if prev_state is None:
        return target
    prev_by_id = {g.gpu_id: g for g in prev_state.real_gpus()}
    for g in target.real_gpus():
        old = prev_by_id.get(g.gpu_id)
        if old is None:
            continue
        if _gpu_logical_template(old) != _gpu_logical_template(g):
            continue

        old_order = [inst.profile for inst in sorted(old.instances, key=lambda x: (x.start, x.end)) if inst.profile != "void"]
        new_instances = [copy.deepcopy(inst) for inst in sorted(g.instances, key=lambda x: (x.start, x.end)) if inst.profile != "void"]

        by_profile = defaultdict(list)
        for inst in new_instances:
            by_profile[inst.profile].append(inst)

        rebuilt = []
        cur = 0
        ok = True
        for prof in old_order:
            if not by_profile[prof]:
                ok = False
                break
            inst = by_profile[prof].pop(0)
            ln = PROFILE_SIZE[prof]
            inst.start = cur
            inst.end = cur + ln
            inst.gpu_id = g.gpu_id
            rebuilt.append(inst)
            cur += ln
        if ok and cur == 7:
            g.instances = rebuilt
            g.sort_instances()
    return target

def build_target_state_from_milp(
    milp_res: Dict[str, Any],
    prev_state: Optional["ClusterState"] = None,
    abstract_template_topk: int = 64,
    physical_layout_topk: int = 32,
    per_gpu_layout_topk: int = 4,
    verbose: bool = True,
) -> "ClusterState":
    t0 = time.time()

    milp_template_ref = extract_template_list_from_milp(milp_res)
    instance_demands = extract_instance_demands_from_milp(milp_res)
    arrivals = _arrival_dict_from_milp(milp_res)

    gpu_count = len(milp_template_ref)
    profile_need = _profile_need_from_instance_demands(instance_demands)
    demands = _expand_demands_with_ids(instance_demands)

    candidate_abstract_sets = _enumerate_candidate_abstract_template_sets(
        gpu_count=gpu_count,
        profile_need=profile_need,
        milp_template_ref=milp_template_ref,
        prev_state=prev_state,
        max_candidates=abstract_template_topk,
    )
    candidate_abstract_sets = _augment_candidate_abstract_template_sets(
        candidate_sets=candidate_abstract_sets,
        milp_template_ref=milp_template_ref,
        prev_state=prev_state,
        max_candidates=abstract_template_topk,
    )

    prev_mode = _has_prev(prev_state)
    best_target, best_metrics, best_score = None, None, None

    for cand_abs in candidate_abstract_sets:
        ordered_abs = _order_candidate_templates_for_gpu_ids(
            candidate_templates=cand_abs,
            gpu_count=gpu_count,
            prev_state=prev_state,
            milp_template_ref=milp_template_ref,
        )

        physical_layout_combos = _enumerate_physical_layout_combinations(
            ordered_abstract_templates=ordered_abs,
            prev_state=prev_state,
            milp_template_ref=milp_template_ref,
            max_combos=physical_layout_topk,
            per_gpu_topk=per_gpu_layout_topk,
        )

        for combo in physical_layout_combos:
            try:
                target, metrics = _solve_target_with_greedy_repair(
                    demands=demands,
                    ordered_abstract_templates=ordered_abs,
                    ordered_physical_templates=combo["physical_template_strs"],
                    intervals_list=combo["intervals_list"],
                    prev_state=prev_state,
                    native_profile_need=profile_need,
                    layout_preserve_score=combo["layout_score"],
                    repair_rounds=8,
                )
            except RuntimeError:
                continue

            score = _score_tuple(metrics, prev_mode)

            if best_score is None or score > best_score:
                best_score = score
                best_target = target
                best_metrics = metrics
                best_target.metadata["arrivals"] = dict(arrivals)

    if best_target is None:
        raise RuntimeError("Target-state build failed: no feasible candidate found")

    # Post-process 1: re-match target GPUs to old GPU identities by minimum-change preference
    best_target = _reassign_gpu_ids_by_matching(best_target, prev_state)

    # Post-process 2: if logical template is unchanged for a matched GPU, keep old physical order
    best_target = _apply_same_logical_template_order_fix(best_target, prev_state)

    elapsed = time.time() - t0
    best_target.metadata["build_metrics"] = dict(best_metrics)
    best_target.metadata["build_metrics"]["score_tuple"] = best_score
    best_target.metadata["build_metrics"]["elapsed_time_sec"] = elapsed
    best_target.metadata["build_method"] = "greedy"

    if verbose:
        print("=" * 90)
        print("[TARGET-STATE BUILDER] BEST RESULT (greedy)")
        print("=" * 90)
        print(f"GPU count fixed from MILP    : {gpu_count}")
        print(f"Profile need                : {profile_need}")
        print(f"MILP abstract template ref  : {milp_template_ref}")
        print(f"Chosen abstract templates   : {best_metrics['ordered_abstract_templates']}")
        print(f"Chosen physical templates   : {best_metrics['ordered_physical_templates']}")
        print(f"Layout preserve score       : {best_metrics['layout_preserve_score']}")
        print(f"Build score                 : {best_score}")
        print(f"Build time (s)              : {elapsed:.4f}")
        print(f"exact_preserve              : {best_metrics['exact_preserve']}")
        print(f"upgrade_preserve            : {best_metrics['upgrade_preserve']}")
        print(f"same_gpu_preserve           : {best_metrics['same_gpu_preserve']}")
        print(f"spread                      : {best_metrics['spread']}")
        print(f"collocate_pairs             : {best_metrics['collocate_pairs']}")
        print(f"mixed_gpu_count             : {best_metrics['mixed_gpu_count']}")
        print(f"template_match_count        : {best_metrics['template_match_count']}")
        print("=" * 90)

    return best_target


### Greedy result


In [190]:

# ============================================================
# Helper 1: solve MILP under a new arrival-rate vector
# ============================================================

def _solve_milp_with_new_arrival_rate(
    new_arrival_rate,
    verbose=False,
    time_limit_s=None,
    mip_gap=None,
    threads=None,
    warm_start_res=None,
    apply_option_pruning=True,
):
    """
    Re-solve the same MILP with a new arrival-rate vector.
    Compatible with the current notebook's MILP setup.
    """
    arr = np.array(new_arrival_rate, dtype=float).copy()

    if arr.ndim != 1:
        raise ValueError("new_arrival_rate must be a 1-D array-like")
    if len(arr) != N_WORKLOADS:
        raise ValueError(f"new_arrival_rate length must be {N_WORKLOADS}, got {len(arr)}")

    res = solve_milp_gurobi_batch_unified(
        feasible_option_df=feasible_option_df,
        arrival_rate=arr,
        templates=TEMPLATES,
        template_K=TEMPLATE_K,
        profile_order=PROFILE_ORDER,
        n_workloads=N_WORKLOADS,
        time_limit_s=time_limit_s,
        mip_gap=mip_gap,
        threads=threads,
        verbose=verbose,
        apply_option_pruning=apply_option_pruning,
        warm_start_res=warm_start_res,
    )
    res["arrival_rate_used"] = arr.copy()
    return res


# ============================================================
# Helper 2: compare two arrival-rate vectors
# ============================================================

def print_arrival_rate_compare(old_arrival_rate, new_arrival_rate):
    old_arr = np.array(old_arrival_rate, dtype=float).copy()
    new_arr = np.array(new_arrival_rate, dtype=float).copy()

    if old_arr.ndim != 1 or new_arr.ndim != 1:
        raise ValueError("arrival-rate inputs must be 1-D array-like")
    if len(old_arr) != N_WORKLOADS or len(new_arr) != N_WORKLOADS:
        raise ValueError(f"arrival-rate length must be {N_WORKLOADS}")

    rows = []
    for i, w in enumerate(WORKLOAD_NAMES):
        old_v = float(old_arr[i])
        new_v = float(new_arr[i])
        ratio = (new_v / old_v) if old_v > 0 else np.nan
        delta = new_v - old_v
        pct = ((new_v - old_v) / old_v * 100.0) if old_v > 0 else np.nan
        rows.append([w, old_v, new_v, delta, ratio, pct])

    print("=" * 90)
    print("Arrival-rate comparison")
    print("=" * 90)
    print(tabulate(
        rows,
        headers=["Workload", "Old", "New", "Delta", "New/Old", "Delta %"],
        tablefmt="github",
        floatfmt=".4f"
    ))


In [191]:
target0 = build_target_state_from_milp(
    milp_res=milp_res,
    prev_state=None,
    abstract_template_topk=64,
            physical_layout_topk=32,
    per_gpu_layout_topk=4,
    verbose=True,
)

print_target_build_result(target0, title="[CHECK] TARGET STATE (no prev_state)")
verify_target_matches_milp(target0, milp_res)
print("target0 templates:", [g.template_str() for g in target0.real_gpus()])

[TARGET-STATE BUILDER] BEST RESULT (greedy)
GPU count fixed from MILP    : 6
Profile need                : {'7g': 0, '4g': 3, '3g': 3, '2g': 1, '1g': 19}
MILP abstract template ref  : ['4+3', '4+1+1+1', '4+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '2+1+1+1+1+1']
Chosen abstract templates   : ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
Chosen physical templates   : ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
Layout preserve score       : (0, 0, 0, 0)
Build score                 : (-8, 51, -1, 0, 0)
Build time (s)              : 0.1037
exact_preserve              : 0
upgrade_preserve            : 0
same_gpu_preserve           : 0
spread                      : 8
collocate_pairs             : 51
mixed_gpu_count             : 1
template_match_count        : 0
[CHECK] TARGET STATE (no prev_state)
GPU 0: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GP

In [192]:
arrival_rate_up_manual = np.array(arrival_rate, dtype=float).copy()

name_to_idx = {name: i for i, name in enumerate(WORKLOAD_NAMES)}

arrival_rate_up_manual[name_to_idx["llama"]]    *= 1.15
arrival_rate_up_manual[name_to_idx["gpt2"]]     *= 1.12
arrival_rate_up_manual[name_to_idx["vgg16"]]    *= 1.08
arrival_rate_up_manual[name_to_idx["resnet50"]] *= 1.08
arrival_rate_up_manual[name_to_idx["vit_base"]] *= 3 #1.05

milp_res_up_manual = _solve_milp_with_new_arrival_rate(
    arrival_rate_up_manual,
    verbose=False,
    warm_start_res=milp_res,
)

print_arrival_rate_compare(arrival_rate, arrival_rate_up_manual)
print_milp_gurobi_batch_result(milp_res_up_manual)

target1 = build_target_state_from_milp(
    milp_res=milp_res_up_manual,
    prev_state=target0,
    abstract_template_topk=64,
    physical_layout_topk=32,
    per_gpu_layout_topk=4,
    verbose=True,
)

print_target_build_result(target1, title="[CHECK] TARGET STATE (prev_state = target0)")
verify_target_matches_milp(target1, milp_res_up_manual)
print("target1 templates:", [g.template_str() for g in target1.real_gpus()])

Arrival-rate comparison
| Workload   |       Old |       New |     Delta |   New/Old |   Delta % |
|------------|-----------|-----------|-----------|-----------|-----------|
| llama      |    3.0000 |    3.4500 |    0.4500 |    1.1500 |   15.0000 |
| gpt2       |   20.0000 |   22.4000 |    2.4000 |    1.1200 |   12.0000 |
| vgg16      |  300.0000 |  324.0000 |   24.0000 |    1.0800 |    8.0000 |
| resnet50   |  300.0000 |  324.0000 |   24.0000 |    1.0800 |    8.0000 |
| vit_base   | 3000.0000 | 9000.0000 | 6000.0000 |    3.0000 |  200.0000 |
Method        : MILP-Gurobi(batch)
Elapsed (s)   : 0.0450
Feasible      : True
GPU count     : 9
Objective     : 9.0000
Templates     : ['4+3', '4+3', '2+2+3', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1']
K_total       : {'7g': 0, '4g': 2, '3g': 9, '2g': 2, '1g': 24}

Per-workload allocation / rate summary:
| Workload   |   Arrival |   Provided |   Provided/Arrival |    Slack | Placement (batch,profile,count)   |
|

In [193]:
arrival_rate_t2_manual = np.array(arrival_rate_up_manual, dtype=float).copy()

name_to_idx = {name: i for i, name in enumerate(WORKLOAD_NAMES)}

# manually decrease load from target1 setting
arrival_rate_t2_manual[name_to_idx["llama"]]    *= 0.72
arrival_rate_t2_manual[name_to_idx["gpt2"]]     *= 0.78
arrival_rate_t2_manual[name_to_idx["vgg16"]]    *= 0.3
arrival_rate_t2_manual[name_to_idx["resnet50"]] *= 0.3
arrival_rate_t2_manual[name_to_idx["vit_base"]] *= 0.3

milp_res_t2_manual = _solve_milp_with_new_arrival_rate(
    arrival_rate_t2_manual,
    verbose=False,
    warm_start_res=milp_res_up_manual,
)

print_arrival_rate_compare(arrival_rate_up_manual, arrival_rate_t2_manual)
print_milp_gurobi_batch_result(milp_res_t2_manual)


target2 = build_target_state_from_milp(
    milp_res=milp_res_t2_manual,
    prev_state=target1,   # important: target1 is now the old state
    abstract_template_topk=64,
            physical_layout_topk=32,
    per_gpu_layout_topk=4,
    verbose=True,
)

print_target_build_result(target2, title="[CHECK] TARGET STATE (target2 from prev_state=target1)")
verify_target_matches_milp(target2, milp_res_t2_manual)
print("target2 templates:", [g.template_str() for g in target2.real_gpus()])

Arrival-rate comparison
| Workload   |       Old |       New |      Delta |   New/Old |   Delta % |
|------------|-----------|-----------|------------|-----------|-----------|
| llama      |    3.4500 |    2.4840 |    -0.9660 |    0.7200 |  -28.0000 |
| gpt2       |   22.4000 |   17.4720 |    -4.9280 |    0.7800 |  -22.0000 |
| vgg16      |  324.0000 |   97.2000 |  -226.8000 |    0.3000 |  -70.0000 |
| resnet50   |  324.0000 |   97.2000 |  -226.8000 |    0.3000 |  -70.0000 |
| vit_base   | 9000.0000 | 2700.0000 | -6300.0000 |    0.3000 |  -70.0000 |
Method        : MILP-Gurobi(batch)
Elapsed (s)   : 0.0429
Feasible      : True
GPU count     : 6
Objective     : 6.0000
Templates     : ['3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1']
K_total       : {'7g': 0, '4g': 0, '3g': 6, '2g': 0, '1g': 24}

Per-workload allocation / rate summary:
| Workload   |   Arrival |   Provided |   Provided/Arrival |    Slack | Placement (batch,profile,count)   |
|------------|---

In [194]:
arrival_rate_up_3 = np.array(arrival_rate_t2_manual, dtype=float).copy()

name_to_idx = {name: i for i, name in enumerate(WORKLOAD_NAMES)}

arrival_rate_up_3[name_to_idx["llama"]]    *= 1.15
arrival_rate_up_3[name_to_idx["gpt2"]]     *= 1.12
arrival_rate_up_3[name_to_idx["vgg16"]]    *= 1.08
arrival_rate_up_3[name_to_idx["resnet50"]] *= 1.08
arrival_rate_up_3[name_to_idx["vit_base"]] *= 3 #1.05

milp_res_up_3 = _solve_milp_with_new_arrival_rate(
    arrival_rate_up_3,
    verbose=False,
    warm_start_res=milp_res_t2_manual,
)

print_arrival_rate_compare(arrival_rate_t2_manual, arrival_rate_up_3)
print_milp_gurobi_batch_result(milp_res_up_3)

target3 = build_target_state_from_milp(
    milp_res=milp_res_up_3,
    prev_state=target2,
    abstract_template_topk=64,
    physical_layout_topk=32,
    per_gpu_layout_topk=4,
    verbose=True,
)

print_target_build_result(target3, title="[CHECK] TARGET STATE (prev_state = target0)")
verify_target_matches_milp(target1, milp_res_up_manual)
print("target3 templates:", [g.template_str() for g in target3.real_gpus()])



Arrival-rate comparison
| Workload   |       Old |       New |     Delta |   New/Old |   Delta % |
|------------|-----------|-----------|-----------|-----------|-----------|
| llama      |    2.4840 |    2.8566 |    0.3726 |    1.1500 |   15.0000 |
| gpt2       |   17.4720 |   19.5686 |    2.0966 |    1.1200 |   12.0000 |
| vgg16      |   97.2000 |  104.9760 |    7.7760 |    1.0800 |    8.0000 |
| resnet50   |   97.2000 |  104.9760 |    7.7760 |    1.0800 |    8.0000 |
| vit_base   | 2700.0000 | 8100.0000 | 5400.0000 |    3.0000 |  200.0000 |
Method        : MILP-Gurobi(batch)
Elapsed (s)   : 0.0408
Feasible      : True
GPU count     : 8
Objective     : 8.0000
Templates     : ['4+3', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1']
K_total       : {'7g': 0, '4g': 1, '3g': 8, '2g': 0, '1g': 28}

Per-workload allocation / rate summary:
| Workload   |   Arrival |   Provided |   Provided/Arrival |    Slack | Placement (batch,profile,count)       |


In [195]:
print("target0 templates:", [g.template_str() for g in target0.real_gpus()])
print("target1 templates:", [g.template_str() for g in target1.real_gpus()])
print("target2 templates:", [g.template_str() for g in target2.real_gpus()])
print("target3 templates:", [g.template_str() for g in target3.real_gpus()])

target0 templates: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
target1 templates: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3', '1+1+1+1+3', '4+3']
target2 templates: ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3']
target3 templates: ['4+3', '4+3', '1+1+1+1+3', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3', '1+1+1+1+3', '4+2+1']


# Action

## Shared

下面的公共代码供两个 planner 共用：

- GPU-level / instance-level diff
- throughput / feasibility 检查
- physical GPU id 管理（当前限制为 A-K）
- fine-action execution simulator
- planner 输出打印与下一轮 MILP 前的 canonicalize


In [124]:

# ============================================================
# [Action Code Block 1] Imports / helpers / physical-id utilities
# ============================================================
from typing import Dict, Any, List, Optional, Tuple
from collections import defaultdict
import copy
import time
import pandas as pd

PHYSICAL_ID_POOL = [chr(ord("A") + i) for i in range(11)]  # A-K

def _deepcopy_state(state: "ClusterState") -> "ClusterState":
    return copy.deepcopy(state)

def _gpu_map_by_id(state: "ClusterState") -> Dict[int, "GPUState"]:
    return {g.gpu_id: g for g in state.real_gpus()}

def _ensure_state_metadata(state: "ClusterState"):
    if not hasattr(state, "metadata") or state.metadata is None:
        state.metadata = {}
    state.metadata.setdefault("physical_id_map", {})
    state.metadata.setdefault("next_physical_idx", 0)

def _set_physical_id(state: "ClusterState", gpu_id: int, pid: str):
    _ensure_state_metadata(state)
    state.metadata["physical_id_map"][int(gpu_id)] = pid

def _get_physical_id(state: "ClusterState", gpu_id: int) -> Optional[str]:
    _ensure_state_metadata(state)
    return state.metadata["physical_id_map"].get(int(gpu_id))

def _alloc_new_physical_id(state: "ClusterState") -> str:
    _ensure_state_metadata(state)
    idx = int(state.metadata.get("next_physical_idx", 0))
    if idx >= len(PHYSICAL_ID_POOL):
        raise RuntimeError("Out of physical GPU ids (A-Z)")
    pid = PHYSICAL_ID_POOL[idx]
    state.metadata["next_physical_idx"] = idx + 1
    return pid

def _bootstrap_physical_ids_for_state(state: "ClusterState"):
    _ensure_state_metadata(state)
    for g in sorted(state.real_gpus(), key=lambda x: x.gpu_id):
        if _get_physical_id(state, g.gpu_id) is None:
            _set_physical_id(state, g.gpu_id, _alloc_new_physical_id(state))

def print_state_templates_with_physical(state: "ClusterState", title="[STATE TEMPLATES]"):
    _ensure_state_metadata(state)
    print("=" * 90)
    print(title)
    print("=" * 90)
    for g in sorted(state.real_gpus(), key=lambda x: x.gpu_id):
        pid = _get_physical_id(state, g.gpu_id)
        print(f"GPU {g.gpu_id} [{pid}] -> {g.template_str()}")

def _format_gpu_detail_line(g: "GPUState", pid: Optional[str] = None) -> str:
    parts = []
    for inst in sorted(g.instances, key=lambda x: (x.start, x.end)):
        if inst.workload is None:
            parts.append(f"[{inst.start},{inst.end}) {inst.profile}: free")
        else:
            parts.append(f"[{inst.start},{inst.end}) {inst.profile}: {inst.workload} / B{inst.batch} / mu={inst.mu:.4f}")
    if pid is None:
        return f"GPU {g.gpu_id}: " + " | ".join(parts)
    return f"GPU {g.gpu_id} [{pid}]: " + " | ".join(parts)

def print_state_detail(state: "ClusterState", title="[STATE]", include_physical: bool = False, only_gpu_ids: Optional[List[int]] = None, template_label: Optional[str] = None):
    _ensure_state_metadata(state)
    print("=" * 90)
    print(title)
    print("=" * 90)
    gpu_ids = None if only_gpu_ids is None else {int(x) for x in only_gpu_ids}
    rows = []
    for g in sorted(state.real_gpus(), key=lambda x: x.gpu_id):
        if gpu_ids is not None and int(g.gpu_id) not in gpu_ids:
            continue
        pid = _get_physical_id(state, g.gpu_id) if include_physical else None
        rows.append(_format_gpu_detail_line(g, pid=pid))
    if rows:
        for row in rows:
            print(row)
    else:
        print("(empty)")
    if template_label is not None:
        print(f"{template_label}: {[g.template_str() for g in sorted(state.real_gpus(), key=lambda x: x.gpu_id)]}")

def print_stage_current_and_target(current_state: "ClusterState", target_state: "ClusterState", title_prefix: str):
    print(f"GPU count fixed from MILP    : {len(target_state.real_gpus())}")
    print_state_detail(current_state, title=f"{title_prefix} CURRENT STATE", include_physical=True, template_label="current templates")
    print_state_detail(target_state, title=f"{title_prefix} TARGET STATE", include_physical=False, template_label="target templates")


In [125]:

# ============================================================
# [Action Code Block 2] Template / instance diff
# ============================================================
def _gpu_template_signature(g: "GPUState") -> Tuple[int, ...]:
    g.sort_instances()
    return tuple(sorted(PROFILE_SIZE[inst.profile] for inst in g.instances if inst.profile != "void"))

def same_template(src_gpu: Optional["GPUState"], tgt_gpu: Optional["GPUState"]) -> bool:
    if src_gpu is None or tgt_gpu is None:
        return False
    return _gpu_template_signature(src_gpu) == _gpu_template_signature(tgt_gpu)

def _slot_key(inst: "MigInstance") -> Tuple[int, int, str]:
    return (inst.start, inst.end, inst.profile)

def _instance_payload(inst: "MigInstance") -> Tuple[Optional[str], Optional[int]]:
    return (inst.workload, inst.batch)

def classify_gpu_change(src_gpu: Optional["GPUState"], tgt_gpu: Optional["GPUState"]) -> str:
    if src_gpu is None and tgt_gpu is not None:
        return "create_gpu"
    if src_gpu is not None and tgt_gpu is None:
        return "remove_gpu"
    if src_gpu is None and tgt_gpu is None:
        return "none"
    if same_template(src_gpu, tgt_gpu):
        src_slots = {_slot_key(x): _instance_payload(x) for x in src_gpu.instances}
        tgt_slots = {_slot_key(x): _instance_payload(x) for x in tgt_gpu.instances}
        if src_slots == tgt_slots:
            return "keep_gpu"
        return "instance_diff"
    return "reconfiguration"

def diff_instances_within_same_template(src_gpu: "GPUState", tgt_gpu: "GPUState") -> List[Dict[str, Any]]:
    src_by_slot = {_slot_key(x): x for x in src_gpu.instances}
    tgt_by_slot = {_slot_key(x): x for x in tgt_gpu.instances}
    all_slots = sorted(set(src_by_slot) | set(tgt_by_slot))
    out = []
    for sk in all_slots:
        s = src_by_slot.get(sk)
        t = tgt_by_slot.get(sk)
        if s is None and t is not None:
            # same-template path usually should not hit this, but keep for completeness
            out.append({"type": "instance_create", "slot": sk, "src": None, "tgt": t})
            continue
        if s is not None and t is None:
            out.append({"type": "instance_remove", "slot": sk, "src": s, "tgt": None})
            continue
        if s.workload == t.workload and s.batch == t.batch:
            out.append({"type": "keep", "slot": sk, "src": s, "tgt": t})
        elif s.workload == t.workload and s.workload is not None and s.batch != t.batch:
            out.append({"type": "batch_change", "slot": sk, "src": s, "tgt": t})
        elif t.workload is None:
            out.append({"type": "safe_remove_instance", "slot": sk, "src": s, "tgt": t})
        elif s.workload is None:
            out.append({"type": "place_instance", "slot": sk, "src": s, "tgt": t})
        else:
            out.append({"type": "workload_change", "slot": sk, "src": s, "tgt": t})
    return out


In [126]:

# ============================================================
# [Action Code Block 3] Throughput accounting / feasibility checks
# ============================================================
def _arrival_dict_from_vector(arr_vec) -> Dict[str, float]:
    arr = np.array(arr_vec, dtype=float)
    return {WORKLOAD_NAMES[i]: float(arr[i]) for i in range(len(WORKLOAD_NAMES))}

def _required_arrival_dict(src_arr, tgt_arr) -> Dict[str, float]:
    src_d = _arrival_dict_from_vector(src_arr)
    tgt_d = _arrival_dict_from_vector(tgt_arr)
    return {w: min(src_d.get(w, 0.0), tgt_d.get(w, 0.0)) for w in WORKLOAD_NAMES}

def _provided_by_workload(state: "ClusterState") -> Dict[str, float]:
    prov = defaultdict(float)
    for g in state.real_gpus():
        for inst in g.instances:
            if inst.workload is not None:
                prov[inst.workload] += float(inst.mu)
    return dict(prov)

def _safe_after_removing_instance(state: "ClusterState", inst: "MigInstance", required: Dict[str, float]) -> bool:
    if inst.workload is None:
        return True
    prov = _provided_by_workload(state)
    prov[inst.workload] = prov.get(inst.workload, 0.0) - float(inst.mu)
    return prov.get(inst.workload, 0.0) + 1e-9 >= required.get(inst.workload, 0.0)

def _safe_after_removing_gpu(state: "ClusterState", gpu: "GPUState", required: Dict[str, float]) -> bool:
    prov = _provided_by_workload(state)
    for inst in gpu.instances:
        if inst.workload is not None:
            prov[inst.workload] = prov.get(inst.workload, 0.0) - float(inst.mu)
    for w, req in required.items():
        if prov.get(w, 0.0) + 1e-9 < req:
            return False
    return True

def _find_free_profile_slots(state: "ClusterState") -> List[Dict[str, Any]]:
    slots = []
    for g in state.real_gpus():
        for inst in g.instances:
            if inst.workload is None:
                slots.append({
                    "gpu_id": g.gpu_id,
                    "slot": (inst.start, inst.end, inst.profile),
                    "profile": inst.profile,
                    "size": PROFILE_SIZE[inst.profile],
                    "inst": inst,
                })
    return slots

def _find_active_bridge_slot(source_state: "ClusterState",
                             target_state: "ClusterState",
                             profile: str,
                             avoid_gpu_id: Optional[int] = None,
                             avoid_slot: Optional[Tuple[int, int, str]] = None) -> Optional[Dict[str, Any]]:
    tgt_map = _gpu_map_by_id(target_state)
    for cand in _find_free_profile_slots(source_state):
        if cand["profile"] != profile:
            continue
        if avoid_gpu_id is not None and int(cand["gpu_id"]) == int(avoid_gpu_id) and cand["slot"] == avoid_slot:
            continue
        tgt_gpu = tgt_map.get(int(cand["gpu_id"]))
        tgt_inst = _get_inst_by_slot(tgt_gpu, cand["slot"]) if tgt_gpu is not None else None
        if tgt_inst is None:
            continue
        if tgt_inst.workload is not None:
            continue
        return cand
    return None


In [127]:

# ============================================================
# [Action Code Block 4] Planner core (coarse/fine hierarchy)
# ============================================================
def _canonicalize_state_for_next_round(executed_state: "ClusterState") -> "ClusterState":
    out = _deepcopy_state(executed_state)
    _ensure_state_metadata(out)
    old_gpus = sorted(out.real_gpus(), key=lambda x: x.gpu_id)
    old_pid_map = dict(out.metadata.get("physical_id_map", {}))
    new_gpus = []
    new_pid_map = {}
    for new_id, g in enumerate(old_gpus):
        g2 = copy.deepcopy(g)
        old_id = g2.gpu_id
        g2.gpu_id = new_id
        new_gpus.append(g2)
        if old_id in old_pid_map:
            new_pid_map[new_id] = old_pid_map[old_id]
    out.gpus = new_gpus
    out.metadata["physical_id_map"] = new_pid_map
    return out

def _get_gpu_by_id_mut(state: "ClusterState", gpu_id: int) -> Optional["GPUState"]:
    for g in state.real_gpus():
        if int(g.gpu_id) == int(gpu_id):
            return g
    return None

def _get_inst_by_slot(gpu: Optional["GPUState"], slot: Tuple[int, int, str]):
    if gpu is None:
        return None
    s0, e0, p0 = slot
    for inst in gpu.instances:
        if inst.start == s0 and inst.end == e0 and inst.profile == p0:
            return inst
    return None

def _copy_inst_payload(dst_inst: "MigInstance", src_inst: Optional["MigInstance"]):
    if src_inst is None:
        dst_inst.workload = None
        dst_inst.batch = None
        dst_inst.mu = 0.0
        dst_inst.preserved = False
        return
    dst_inst.workload = src_inst.workload
    dst_inst.batch = src_inst.batch
    dst_inst.mu = float(src_inst.mu)
    dst_inst.preserved = bool(getattr(src_inst, "preserved", False))

def _replace_or_append_gpu(state: "ClusterState", gpu: "GPUState"):
    for idx, old_gpu in enumerate(state.gpus):
        if int(old_gpu.gpu_id) == int(gpu.gpu_id):
            state.gpus[idx] = gpu
            return
    state.gpus.append(gpu)

def _remove_gpu_if_bound_to_pid(state: "ClusterState", gpu_id: int, physical_gpu_id: str):
    cur_gpu = _get_gpu_by_id_mut(state, gpu_id)
    if cur_gpu is None:
        return
    cur_pid = _get_physical_id(state, gpu_id)
    if cur_pid != physical_gpu_id:
        return
    state.gpus = [g for g in state.gpus if int(g.gpu_id) != int(gpu_id)]
    state.metadata.get("physical_id_map", {}).pop(int(gpu_id), None)

def _simulate_fine_actions(source_state: "ClusterState",
                           target_state: "ClusterState",
                           fine_actions: List[Dict[str, Any]],
                           next_physical_idx: int) -> "ClusterState":
    executed_state = _deepcopy_state(source_state)
    _ensure_state_metadata(executed_state)
    _bootstrap_physical_ids_for_state(executed_state)
    _ensure_state_metadata(target_state)
    target_map = _gpu_map_by_id(target_state)

    for action in fine_actions:
        action_type = action["type"]

        if action_type in {
            "allocate_gpu",
            "configure_full_template",
            "configure_partial_profile",
            "place_target_layout",
            "defer_remove_gpu",
            "defer_remove_instance",
            "defer_workload_change",
        }:
            continue

        if action_type == "bind_target_gpu":
            gid = int(action["gpu_id"])
            pid = action["physical_gpu_id"]
            tgt_gpu = target_map.get(gid)
            if tgt_gpu is None:
                continue
            _replace_or_append_gpu(executed_state, copy.deepcopy(tgt_gpu))
            _set_physical_id(executed_state, gid, pid)
            continue

        if action_type == "clear_gpu":
            _remove_gpu_if_bound_to_pid(executed_state, int(action["gpu_id"]), action["physical_gpu_id"])
            continue

        if action_type == "clear_template":
            continue

        gid = int(action["gpu_id"])
        pid = action["physical_gpu_id"]
        cur_gpu = _get_gpu_by_id_mut(executed_state, gid)
        if cur_gpu is None or _get_physical_id(executed_state, gid) != pid:
            continue

        slot = tuple(action.get("slot", ()))
        cur_inst = _get_inst_by_slot(cur_gpu, slot) if slot else None
        tgt_gpu = target_map.get(gid)
        tgt_inst = _get_inst_by_slot(tgt_gpu, slot) if slot else None

        if action_type == "bridge_place_instance":
            if cur_inst is not None:
                cur_inst.workload = action.get("workload")
                cur_inst.batch = action.get("batch")
                cur_inst.mu = float(action.get("mu", 0.0))
                cur_inst.preserved = False
            continue

        if action_type in {"update_batch", "place_instance", "workload_change"}:
            if cur_inst is not None and tgt_inst is not None:
                _copy_inst_payload(cur_inst, tgt_inst)
            continue

        if action_type == "remove_instance":
            if cur_inst is not None:
                _copy_inst_payload(cur_inst, None)
            continue

    executed_state.gpus = sorted(executed_state.real_gpus(), key=lambda x: x.gpu_id)
    executed_state.metadata["next_physical_idx"] = int(next_physical_idx)
    return executed_state

def plan_naive_action(source_state: "ClusterState",
                      target_state: "ClusterState",
                      src_arrival,
                      tgt_arrival,
                      stage_name="stage") -> Dict[str, Any]:
    source_state = _deepcopy_state(source_state)
    target_state = _deepcopy_state(target_state)
    _bootstrap_physical_ids_for_state(source_state)
    _ensure_state_metadata(target_state)
    required = _required_arrival_dict(src_arrival, tgt_arrival)

    src_map = _gpu_map_by_id(source_state)
    tgt_map = _gpu_map_by_id(target_state)
    all_gpu_ids = sorted(set(src_map) | set(tgt_map))

    coarse_actions = []
    fine_actions = []
    target_pid_map = {}
    available_old_gpu_ids = set(src_map.keys())

    for gid in all_gpu_ids:
        s = src_map.get(gid)
        t = tgt_map.get(gid)
        change = classify_gpu_change(s, t)

        if change == "keep_gpu":
            pid = _get_physical_id(source_state, gid)
            target_pid_map[gid] = pid
            coarse_actions.append({
                "type": "keep_gpu",
                "gpu_id": gid,
                "physical_gpu_id": pid,
            })
            fine_actions.append({
                "type": "bind_target_gpu",
                "gpu_id": gid,
                "physical_gpu_id": pid,
            })
            available_old_gpu_ids.discard(gid)
            continue

        if change == "create_gpu":
            pid = _alloc_new_physical_id(source_state)
            target_pid_map[gid] = pid
            coarse_actions.append({
                "type": "create_gpu",
                "gpu_id": gid,
                "new_physical_gpu_id": pid,
                "template": t.template_str(),
            })
            fine_actions.extend([
                {"type": "allocate_gpu", "physical_gpu_id": pid},
                {"type": "configure_full_template", "physical_gpu_id": pid, "template": t.template_str()},
                {"type": "place_target_layout", "gpu_id": gid, "physical_gpu_id": pid},
                {"type": "bind_target_gpu", "gpu_id": gid, "physical_gpu_id": pid},
            ])
            continue

        if change == "remove_gpu":
            pid = _get_physical_id(source_state, gid)
            safe = _safe_after_removing_gpu(source_state, s, required)
            coarse_actions.append({
                "type": "remove_gpu",
                "gpu_id": gid,
                "physical_gpu_id": pid,
                "safe_now": safe,
            })
            if safe:
                fine_actions.extend([
                    {"type": "clear_gpu", "gpu_id": gid, "physical_gpu_id": pid},
                    {"type": "clear_template", "gpu_id": gid, "physical_gpu_id": pid},
                ])
            else:
                fine_actions.append({
                    "type": "defer_remove_gpu",
                    "gpu_id": gid,
                    "physical_gpu_id": pid,
                })
            available_old_gpu_ids.discard(gid)
            continue

        if change == "reconfiguration":
            old_pid = _get_physical_id(source_state, gid)
            new_pid = _alloc_new_physical_id(source_state)
            target_pid_map[gid] = new_pid
            coarse_actions.append({
                "type": "reconfiguration",
                "gpu_id": gid,
                "source_physical_gpu_id": old_pid,
                "new_physical_gpu_id": new_pid,
                "src_template": s.template_str(),
                "tgt_template": t.template_str(),
            })
            fine_actions.extend([
                {"type": "allocate_gpu", "physical_gpu_id": new_pid},
                {"type": "configure_full_template", "physical_gpu_id": new_pid, "template": t.template_str()},
                {"type": "place_target_layout", "gpu_id": gid, "physical_gpu_id": new_pid},
                {"type": "bind_target_gpu", "gpu_id": gid, "physical_gpu_id": new_pid},
                {"type": "clear_gpu", "gpu_id": gid, "physical_gpu_id": old_pid},
                {"type": "clear_template", "gpu_id": gid, "physical_gpu_id": old_pid},
            ])
            available_old_gpu_ids.discard(gid)
            continue

        if change == "instance_diff":
            pid = _get_physical_id(source_state, gid)
            target_pid_map[gid] = pid
            available_old_gpu_ids.discard(gid)
            inst_actions = diff_instances_within_same_template(s, t)
            coarse_actions.append({
                "type": "instance_diff",
                "gpu_id": gid,
                "physical_gpu_id": pid,
                "template": t.template_str(),
                "instance_changes": [a["type"] for a in inst_actions],
            })
            for ia in inst_actions:
                if ia["type"] == "keep":
                    continue
                elif ia["type"] == "batch_change":
                    fine_actions.append({
                        "type": "update_batch",
                        "gpu_id": gid,
                        "physical_gpu_id": pid,
                        "slot": ia["slot"],
                        "old_batch": ia["src"].batch,
                        "new_batch": ia["tgt"].batch,
                        "workload": ia["src"].workload,
                    })
                elif ia["type"] == "safe_remove_instance":
                    safe = _safe_after_removing_instance(source_state, ia["src"], required)
                    fine_actions.append({
                        "type": "remove_instance",
                        "gpu_id": gid,
                        "physical_gpu_id": pid,
                        "slot": ia["slot"],
                        "workload": ia["src"].workload,
                        "safe_now": safe,
                    })
                elif ia["type"] == "place_instance":
                    fine_actions.append({
                        "type": "place_instance",
                        "gpu_id": gid,
                        "physical_gpu_id": pid,
                        "slot": ia["slot"],
                        "workload": ia["tgt"].workload,
                        "batch": ia["tgt"].batch,
                    })
                elif ia["type"] == "workload_change":
                    safe = _safe_after_removing_instance(source_state, ia["src"], required)
                    fa = {
                        "type": "workload_change",
                        "gpu_id": gid,
                        "physical_gpu_id": pid,
                        "slot": ia["slot"],
                        "old_workload": ia["src"].workload,
                        "new_workload": ia["tgt"].workload,
                        "old_batch": ia["src"].batch,
                        "new_batch": ia["tgt"].batch,
                        "safe_now": safe,
                    }
                    if not safe:
                        temp_pid = _alloc_new_physical_id(source_state)
                        fa["temp_physical_gpu_id"] = temp_pid
                        fa["fallback"] = "create_temp_capacity"
                    fine_actions.append(fa)
                elif ia["type"] == "instance_remove":
                    fine_actions.append({
                        "type": "remove_instance",
                        "gpu_id": gid,
                        "physical_gpu_id": pid,
                        "slot": ia["slot"],
                        "workload": ia["src"].workload if ia["src"] else None,
                        "safe_now": True,
                    })
                elif ia["type"] == "instance_create":
                    fine_actions.append({
                        "type": "place_instance",
                        "gpu_id": gid,
                        "physical_gpu_id": pid,
                        "slot": ia["slot"],
                        "workload": ia["tgt"].workload if ia["tgt"] else None,
                        "batch": ia["tgt"].batch if ia["tgt"] else None,
                    })
            continue

    planned_state = _deepcopy_state(target_state)
    _ensure_state_metadata(planned_state)
    planned_state.metadata["physical_id_map"] = {int(gid): pid for gid, pid in target_pid_map.items()}
    planned_state.metadata["next_physical_idx"] = source_state.metadata.get("next_physical_idx", 0)

    executed_state = _simulate_fine_actions(
        source_state=source_state,
        target_state=planned_state,
        fine_actions=fine_actions,
        next_physical_idx=source_state.metadata.get("next_physical_idx", 0),
    )

    return {
        "stage_name": stage_name,
        "required": required,
        "coarse_actions": coarse_actions,
        "fine_actions": fine_actions,
        "planned_state": planned_state,
        "executed_state": executed_state,
    }

def print_plan_result(plan_res: Dict[str, Any], title="[PLAN RESULT]"):
    print("=" * 90)
    print(title)
    print("=" * 90)
    print(f"Stage: {plan_res['stage_name']}")
    print(f"Coarse actions: {len(plan_res['coarse_actions'])}")
    for i, a in enumerate(plan_res["coarse_actions"], 1):
        print(f"{i:02d}. {a}")
    print("-" * 90)
    print(f"Fine actions: {len(plan_res['fine_actions'])}")
    for i, a in enumerate(plan_res["fine_actions"], 1):
        print(f"{i:02d}. {a}")
    deferred = [a for a in plan_res["fine_actions"] if str(a.get("type", "")).startswith("defer_")]
    if deferred:
        print("-" * 90)
        print(f"Deferred actions: {len(deferred)}")


## V0

### Naive v0 experiment — stage 0

从空状态到 `target0` 跑 baseline naive，并记录 planner 用时。

In [128]:

# ============================================================
# [Action Code Block 5] Stage 0: initial -> target0 -> planner -> executed0
# ============================================================
source0 = ClusterState(gpus=[], metadata={})
_ensure_state_metadata(source0)

t_plan0 = time.perf_counter()
plan0 = plan_naive_action(
    source_state=source0,
    target_state=target0,
    src_arrival=np.zeros_like(arrival_rate, dtype=float),
    tgt_arrival=arrival_rate,
    stage_name="s0_to_t0",
)
print_plan_result(plan0, title="[PLAN] source0 -> target0")
plan0_elapsed_sec = time.perf_counter() - t_plan0
executed0 = plan0["executed_state"]

print_state_templates_with_physical(target0, title="[TARGET0] templates")
print_state_templates_with_physical(executed0, title="[EXECUTED0 before canonicalize] templates / physical ids")
source1 = _canonicalize_state_for_next_round(executed0)
print_state_templates_with_physical(source1, title="[SOURCE1 for next MILP/transition] canonicalized templates / physical ids")


[PLAN] source0 -> target0
Stage: s0_to_t0
Coarse actions: 6
01. {'type': 'create_gpu', 'gpu_id': 0, 'new_physical_gpu_id': 'A', 'template': '4+3'}
02. {'type': 'create_gpu', 'gpu_id': 1, 'new_physical_gpu_id': 'B', 'template': '4+3'}
03. {'type': 'create_gpu', 'gpu_id': 2, 'new_physical_gpu_id': 'C', 'template': '4+3'}
04. {'type': 'create_gpu', 'gpu_id': 3, 'new_physical_gpu_id': 'D', 'template': '2+1+1+1+1+1'}
05. {'type': 'create_gpu', 'gpu_id': 4, 'new_physical_gpu_id': 'E', 'template': '1+1+1+1+1+1+1'}
06. {'type': 'create_gpu', 'gpu_id': 5, 'new_physical_gpu_id': 'F', 'template': '1+1+1+1+1+1+1'}
------------------------------------------------------------------------------------------
Fine actions: 24
01. {'type': 'allocate_gpu', 'physical_gpu_id': 'A'}
02. {'type': 'configure_full_template', 'physical_gpu_id': 'A', 'template': '4+3'}
03. {'type': 'place_target_layout', 'gpu_id': 0, 'physical_gpu_id': 'A'}
04. {'type': 'bind_target_gpu', 'gpu_id': 0, 'physical_gpu_id': 'A'}
05. 

### Naive v0 experiment — stage 1

先对 `source1` 重新做 `MILP + transition` 得到 `target1_seq`，再跑 baseline naive，并在进入下一轮前 canonicalize。

In [129]:

# ============================================================
# [Action Code Block 6] Stage 1: source1 -> MILP+transition -> target1_seq
# ============================================================
target1_seq = build_target_state_from_milp(
    milp_res=milp_res_up_manual,
    prev_state=source1,
    abstract_template_topk=64,
    physical_layout_topk=32,
    per_gpu_layout_topk=4,
    verbose=True,
)
print_target_build_result(target1_seq, title="[CHECK] TARGET1_SEQ")
verify_target_matches_milp(target1_seq, milp_res_up_manual)

print_state_templates_with_physical(source1, title="[SOURCE1] templates / physical ids")
print_state_templates_with_physical(target1_seq, title="[TARGET1_SEQ] templates")


[TARGET-STATE BUILDER] BEST RESULT (greedy)
GPU count fixed from MILP    : 9
Profile need                : {'7g': 0, '4g': 2, '3g': 9, '2g': 0, '1g': 23}
MILP abstract template ref  : ['4+3', '4+3', '2+2+3', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1']
Chosen abstract templates   : ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '3+1+1+1+1', '3+3', '4+3']
Chosen physical templates   : ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+3', '4+3', '4+3']
Layout preserve score       : (26, 6, 26, 42)
Build score                 : (23, 2, -12, 55, -2, 26, 6, 0)
Build time (s)              : 0.5132
exact_preserve              : 23
upgrade_preserve            : 2
same_gpu_preserve           : 0
spread                      : 12
collocate_pairs             : 55
mixed_gpu_count             : 2
template_match_count        : 0
[CHECK] TARGET1_SEQ
GPU 0: [0,4) 4g: llama / B1 / mu=0.7410 | [4,7) 3g: llama / B1 / mu

In [130]:

# ============================================================
# [Action Code Block 7] Stage 1 planner / canonicalize
# ============================================================
t_plan1 = time.perf_counter()
plan1 = plan_naive_action(
    source_state=source1,
    target_state=target1_seq,
    src_arrival=arrival_rate,
    tgt_arrival=arrival_rate_up_manual,
    stage_name="t0_to_t1",
)
print_plan_result(plan1, title="[PLAN] target0(executed canonical) -> target1_seq")
plan1_elapsed_sec = time.perf_counter() - t_plan1
executed1 = plan1["executed_state"]
print_state_templates_with_physical(executed1, title="[EXECUTED1 before canonicalize] templates / physical ids")
source2 = _canonicalize_state_for_next_round(executed1)
print_state_templates_with_physical(source2, title="[SOURCE2 for next MILP/transition] canonicalized templates / physical ids")

print(f"[STAGE 1] v0 naive planner elapsed: {plan1_elapsed_sec:.6f} sec")


[PLAN] target0(executed canonical) -> target1_seq
Stage: t0_to_t1
Coarse actions: 9
01. {'type': 'keep_gpu', 'gpu_id': 0, 'physical_gpu_id': 'A'}
02. {'type': 'keep_gpu', 'gpu_id': 1, 'physical_gpu_id': 'B'}
03. {'type': 'keep_gpu', 'gpu_id': 2, 'physical_gpu_id': 'C'}
04. {'type': 'instance_diff', 'gpu_id': 3, 'physical_gpu_id': 'D', 'template': '2+1+1+1+1+1', 'instance_changes': ['safe_remove_instance', 'keep', 'keep', 'keep', 'keep', 'keep']}
05. {'type': 'keep_gpu', 'gpu_id': 4, 'physical_gpu_id': 'E'}
06. {'type': 'keep_gpu', 'gpu_id': 5, 'physical_gpu_id': 'F'}
07. {'type': 'create_gpu', 'gpu_id': 6, 'new_physical_gpu_id': 'G', 'template': '4+3'}
08. {'type': 'create_gpu', 'gpu_id': 7, 'new_physical_gpu_id': 'H', 'template': '1+1+1+1+3'}
09. {'type': 'create_gpu', 'gpu_id': 8, 'new_physical_gpu_id': 'I', 'template': '4+3'}
------------------------------------------------------------------------------------------
Fine actions: 18
01. {'type': 'bind_target_gpu', 'gpu_id': 0, 'physi

### Naive v0 experiment — stage 2

对 `source2` 重复 `MILP + transition -> planner -> canonicalize`，并记录 planner 用时。

In [131]:

# ============================================================
# [Action Code Block 8] Stage 2: source2 -> MILP+transition -> target2_seq
# ============================================================
target2_seq = build_target_state_from_milp(
    milp_res=milp_res_t2_manual,
    prev_state=source2,
    abstract_template_topk=64,
    physical_layout_topk=32,
    per_gpu_layout_topk=4,
    verbose=True,
)
print_target_build_result(target2_seq, title="[CHECK] TARGET2_SEQ")
verify_target_matches_milp(target2_seq, milp_res_t2_manual)

print_state_templates_with_physical(source2, title="[SOURCE2] templates / physical ids")
print_state_templates_with_physical(target2_seq, title="[TARGET2_SEQ] templates")


[TARGET-STATE BUILDER] BEST RESULT (greedy)
GPU count fixed from MILP    : 6
Profile need                : {'7g': 0, '4g': 0, '3g': 6, '2g': 0, '1g': 18}
MILP abstract template ref  : ['3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1']
Chosen abstract templates   : ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
Chosen physical templates   : ['4+3', '4+3', '4+3', '1+1+2+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
Layout preserve score       : (23, 6, 23, 38)
Build score                 : (20, 3, -8, 46, -1, 23, 6, 0)
Build time (s)              : 0.2055
exact_preserve              : 20
upgrade_preserve            : 3
same_gpu_preserve           : 0
spread                      : 8
collocate_pairs             : 46
mixed_gpu_count             : 1
template_match_count        : 0
[CHECK] TARGET2_SEQ
GPU 0: [0,4) 4g: llama / B1 / mu=0.7410 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1: [0,4) 4g: llama / B1 / mu=0.7410 | [4,7) 3g: llama / B1 / mu=0.7

In [132]:

# ============================================================
# [Action Code Block 9] Stage 2 planner / canonicalize
# ============================================================
t_plan2 = time.perf_counter()
plan2 = plan_naive_action(
    source_state=source2,
    target_state=target2_seq,
    src_arrival=arrival_rate_up_manual,
    tgt_arrival=arrival_rate_t2_manual,
    stage_name="t1_to_t2",
)
print_plan_result(plan2, title="[PLAN] target1(executed canonical) -> target2_seq")
plan2_elapsed_sec = time.perf_counter() - t_plan2
executed2 = plan2["executed_state"]
print_state_templates_with_physical(executed2, title="[EXECUTED2 before canonicalize] templates / physical ids")
source3 = _canonicalize_state_for_next_round(executed2)
print_state_templates_with_physical(source3, title="[SOURCE3 for next MILP/transition] canonicalized templates / physical ids")

print(f"[STAGE 2] v0 naive planner elapsed: {plan2_elapsed_sec:.6f} sec")


[PLAN] target1(executed canonical) -> target2_seq
Stage: t1_to_t2
Coarse actions: 9
01. {'type': 'keep_gpu', 'gpu_id': 0, 'physical_gpu_id': 'A'}
02. {'type': 'keep_gpu', 'gpu_id': 1, 'physical_gpu_id': 'B'}
03. {'type': 'remove_gpu', 'gpu_id': 2, 'physical_gpu_id': 'C', 'safe_now': True}
04. {'type': 'instance_diff', 'gpu_id': 3, 'physical_gpu_id': 'D', 'template': '2+1+1+1+1+1', 'instance_changes': ['keep', 'workload_change', 'safe_remove_instance', 'keep', 'keep', 'keep']}
05. {'type': 'keep_gpu', 'gpu_id': 4, 'physical_gpu_id': 'E'}
06. {'type': 'keep_gpu', 'gpu_id': 5, 'physical_gpu_id': 'F'}
07. {'type': 'remove_gpu', 'gpu_id': 6, 'physical_gpu_id': 'G', 'safe_now': True}
08. {'type': 'remove_gpu', 'gpu_id': 7, 'physical_gpu_id': 'H', 'safe_now': False}
09. {'type': 'keep_gpu', 'gpu_id': 8, 'physical_gpu_id': 'I'}
------------------------------------------------------------------------------------------
Fine actions: 12
01. {'type': 'bind_target_gpu', 'gpu_id': 0, 'physical_gpu_i

### Naive v0 experiment — stage 3

对 `source3` 重复 `MILP + transition -> planner -> canonicalize`，并记录 planner 用时。

In [133]:

# ============================================================
# [Action Code Block 10] Stage 3: source3 -> MILP+transition -> target3_seq
# ============================================================
target3_seq = build_target_state_from_milp(
    milp_res=milp_res_up_3,
    prev_state=source3,
    abstract_template_topk=64,
    physical_layout_topk=32,
    per_gpu_layout_topk=4,
    verbose=True,
)
print_target_build_result(target3_seq, title="[CHECK] TARGET3_SEQ")
verify_target_matches_milp(target3_seq, milp_res_up_3)

print_state_templates_with_physical(source3, title="[SOURCE3] templates / physical ids")
print_state_templates_with_physical(target3_seq, title="[TARGET3_SEQ] templates")


[TARGET-STATE BUILDER] BEST RESULT (greedy)
GPU count fixed from MILP    : 8
Profile need                : {'7g': 0, '4g': 1, '3g': 8, '2g': 0, '1g': 22}
MILP abstract template ref  : ['4+3', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1']
Chosen abstract templates   : ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '3+1+1+1+1', '3+3', '3+3']
Chosen physical templates   : ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+3', '4+3', '4+3']
Layout preserve score       : (29, 7, 29, 44)
Build score                 : (23, 1, -13, 48, -3, 29, 7, 0)
Build time (s)              : 0.3453
exact_preserve              : 23
upgrade_preserve            : 1
same_gpu_preserve           : 0
spread                      : 13
collocate_pairs             : 48
mixed_gpu_count             : 3
template_match_count        : 0
[CHECK] TARGET3_SEQ
GPU 0: [0,4) 4g: vit_base / B64 / mu=1718.9838 | [4,7) 3g: llama / B1 / mu=0.7410
GP

In [134]:

# ============================================================
# [Action Code Block 11] Stage 3 planner / canonicalize
# ============================================================
t_plan3 = time.perf_counter()
plan3 = plan_naive_action(
    source_state=source3,
    target_state=target3_seq,
    src_arrival=arrival_rate_t2_manual,
    tgt_arrival=arrival_rate_up_3,
    stage_name="t2_to_t3",
)
print_plan_result(plan3, title="[PLAN] target2(executed canonical) -> target3_seq")
plan3_elapsed_sec = time.perf_counter() - t_plan3
executed3 = plan3["executed_state"]
print_state_templates_with_physical(executed3, title="[EXECUTED3 before canonicalize] templates / physical ids")
source4 = _canonicalize_state_for_next_round(executed3)
print_state_templates_with_physical(source4, title="[SOURCE4 for next MILP] canonicalized templates / physical ids")

print(f"[STAGE 3] v0 naive planner elapsed: {plan3_elapsed_sec:.6f} sec")


[PLAN] target2(executed canonical) -> target3_seq
Stage: t2_to_t3
Coarse actions: 8
01. {'type': 'instance_diff', 'gpu_id': 0, 'physical_gpu_id': 'A', 'template': '4+3', 'instance_changes': ['workload_change', 'keep']}
02. {'type': 'keep_gpu', 'gpu_id': 1, 'physical_gpu_id': 'B'}
03. {'type': 'keep_gpu', 'gpu_id': 2, 'physical_gpu_id': 'D'}
04. {'type': 'keep_gpu', 'gpu_id': 3, 'physical_gpu_id': 'E'}
05. {'type': 'keep_gpu', 'gpu_id': 4, 'physical_gpu_id': 'F'}
06. {'type': 'instance_diff', 'gpu_id': 5, 'physical_gpu_id': 'H', 'template': '1+1+1+1+3', 'instance_changes': ['keep', 'keep', 'workload_change', 'workload_change', 'keep']}
07. {'type': 'keep_gpu', 'gpu_id': 6, 'physical_gpu_id': 'I'}
08. {'type': 'create_gpu', 'gpu_id': 7, 'new_physical_gpu_id': 'J', 'template': '4+3'}
------------------------------------------------------------------------------------------
Fine actions: 12
01. {'type': 'workload_change', 'gpu_id': 0, 'physical_gpu_id': 'A', 'slot': (0, 4, '4g'), 'old_work

### Naive v0 summary

下面给出 baseline naive 的最终状态、模板检查和各阶段 planner 时间。

In [135]:

# ============================================================
# [Action Code Block 12] Summary aliases / quick template checks
# ============================================================
planner_final0 = executed0
planner_final1 = executed1
planner_final2 = executed2
planner_final3 = executed3

print("source1 templates:", [g.template_str() for g in source1.real_gpus()])
print("target1_seq templates:", [g.template_str() for g in target1_seq.real_gpus()])
print("source2 templates:", [g.template_str() for g in source2.real_gpus()])
print("target2_seq templates:", [g.template_str() for g in target2_seq.real_gpus()])
print("source3 templates:", [g.template_str() for g in source3.real_gpus()])
print("target3_seq templates:", [g.template_str() for g in target3_seq.real_gpus()])


v0_time_df = pd.DataFrame([
    {"planner": "naive_v0", "stage": 0, "elapsed_sec": plan0_elapsed_sec},
    {"planner": "naive_v0", "stage": 1, "elapsed_sec": plan1_elapsed_sec},
    {"planner": "naive_v0", "stage": 2, "elapsed_sec": plan2_elapsed_sec},
    {"planner": "naive_v0", "stage": 3, "elapsed_sec": plan3_elapsed_sec},
])
print("\n[v0 planner timing]")
display(v0_time_df)


source1 templates: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
target1_seq templates: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3', '1+1+1+1+3', '4+3']
source2 templates: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3', '1+1+1+1+3', '4+3']
target2_seq templates: ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3']
source3 templates: ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+3', '4+3']
target3_seq templates: ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+3', '4+3', '4+3']

[v0 planner timing]


,planner,stage,elapsed_sec
0,naive_v0,0,0.007044
1,naive_v0,1,0.001676
2,naive_v0,2,0.001302
3,naive_v0,3,0.001296


## V1

### Naive v1 — feasible-first + LIFO free-pool reuse

这一部分在保留 `v0` diff / coarse-fine 结构的基础上，增加两条规则，并在单个 stage 内做多轮迭代直到达到 target 或无进展为止：

1. `safe_now=True` 的动作优先执行；对 unsafe 的 `remove_gpu / remove_instance / workload_change` 不直接执行。
2. `allocate_gpu` 优先从单一 `free_pool` 中按 **LIFO** 复用已释放 GPU；初始空闲池按 `K -> ... -> A` 建立，因此首次分配会先落到 `A`，之后继续遵循 LIFO 复用规则。

下面单独跑完整的 `0 -> t1 -> t2 -> t3` 实验，并统计 planner 时间，方便和 `v0` 对比。

In [136]:

# ============================================================
# [Action Code Block V1-1] v1 planner helpers: single free-pool (LIFO) + feasible-first
# ============================================================
def _build_initial_free_pool(state: "ClusterState") -> List[str]:
    _bootstrap_physical_ids_for_state(state)
    active = {_get_physical_id(state, g.gpu_id) for g in state.real_gpus()}
    # Keep LIFO semantics, but reverse the initial pool so the very first
    # allocation from an empty cluster uses A, then B, then C, ...
    return [pid for pid in reversed(PHYSICAL_ID_POOL) if pid not in active]


def _alloc_from_free_pool(free_pool: List[str]) -> str:
    if not free_pool:
        raise RuntimeError("Out of free physical GPUs in A-K")
    return free_pool.pop()  # LIFO


def _is_deferred_action(action: Dict[str, Any]) -> bool:
    return str(action.get("type", "")).startswith("defer_")


def _split_v1_actions(fine_actions: List[Dict[str, Any]]):
    executed_actions = [copy.deepcopy(a) for a in fine_actions if not _is_deferred_action(a)]
    blocked_actions = [copy.deepcopy(a) for a in fine_actions if _is_deferred_action(a)]
    return executed_actions, blocked_actions


def _state_semantic_signature(state: "ClusterState"):
    rows = []
    for g in sorted(state.real_gpus(), key=lambda x: x.gpu_id):
        inst_rows = []
        for inst in sorted(g.instances, key=lambda x: (x.start, x.end, x.profile)):
            inst_rows.append((
                int(inst.start),
                int(inst.end),
                inst.profile,
                inst.workload,
                None if inst.batch is None else int(inst.batch),
            ))
        rows.append((int(g.gpu_id), tuple(inst_rows)))
    return tuple(rows)


def _matches_target_state(state: "ClusterState", target_state: "ClusterState") -> bool:
    return _state_semantic_signature(state) == _state_semantic_signature(target_state)


def _gpu_semantic_signature(g: Optional["GPUState"]):
    if g is None:
        return None
    inst_rows = []
    for inst in sorted(g.instances, key=lambda x: (x.start, x.end, x.profile)):
        inst_rows.append((
            int(inst.start),
            int(inst.end),
            inst.profile,
            inst.workload,
            None if inst.batch is None else int(inst.batch),
        ))
    return tuple(inst_rows)


def _mismatched_gpu_ids(state: "ClusterState", target_state: "ClusterState") -> List[int]:
    cur_map = _gpu_map_by_id(state)
    tgt_map = _gpu_map_by_id(target_state)
    all_ids = sorted(set(cur_map) | set(tgt_map))
    bad = []
    for gid in all_ids:
        if _gpu_semantic_signature(cur_map.get(gid)) != _gpu_semantic_signature(tgt_map.get(gid)):
            bad.append(int(gid))
    return bad


def _build_canonicalize_actions(before_state: "ClusterState", after_state: "ClusterState") -> List[Dict[str, Any]]:
    _ensure_state_metadata(before_state)
    _ensure_state_metadata(after_state)
    before_rows = []
    for g in sorted(before_state.real_gpus(), key=lambda x: x.gpu_id):
        before_rows.append((int(g.gpu_id), _get_physical_id(before_state, g.gpu_id)))
    after_rows = []
    for g in sorted(after_state.real_gpus(), key=lambda x: x.gpu_id):
        after_rows.append((int(g.gpu_id), _get_physical_id(after_state, g.gpu_id)))
    actions = []
    for (old_gid, old_pid), (new_gid, new_pid) in zip(before_rows, after_rows):
        if old_gid == new_gid and old_pid == new_pid:
            continue
        actions.append({
            "type": "canonicalize_bind",
            "old_gpu_id": old_gid,
            "new_gpu_id": new_gid,
            "physical_gpu_id": new_pid,
        })
    return actions


def print_canonicalize_result(before_state: "ClusterState", after_state: "ClusterState", title="[CANONICALIZE]"):
    print("-" * 90)
    print(title)
    actions = _build_canonicalize_actions(before_state, after_state)
    print(f"Canonicalize actions: {len(actions)}")
    for i, a in enumerate(actions, 1):
        print(f"  K{i:02d}. {a}")
    print_state_detail(after_state, title=f"{title} state after canonicalize", include_physical=True, template_label="canonicalized templates")


def print_v1_stage_result(stage_res: Dict[str, Any], title="[V1 STAGE RESULT]"):
    print("=" * 90)
    print(title)
    print("=" * 90)
    print(f"Stage: {stage_res['stage_name']}")
    print(f"Iterations: {stage_res['iteration_count']}")
    print(f"Reached target: {stage_res['reached_target']}")
    print(f"Elapsed (s): {stage_res['elapsed_sec']:.6f}")
    print(f"Total executed runtime actions: {len(stage_res['executed_actions'])}")
    print(f"Final blocked runtime actions: {len(stage_res['blocked_actions'])}")
    for iter_res in stage_res['iterations']:
        plan = iter_res['plan']
        print("-" * 90)
        print(f"Iteration {iter_res['iteration']}: progress={iter_res['made_progress']}, reached_target={iter_res['reached_target']}")
        print(f"Coarse actions: {len(plan['coarse_actions'])}")
        for i, a in enumerate(plan['coarse_actions'], 1):
            print(f"  C{i:02d}. {a}")
        print(f"Executed runtime actions: {len(iter_res['executed_actions'])}")
        for i, a in enumerate(iter_res['executed_actions'], 1):
            print(f"  X{i:02d}. {a}")
        if iter_res['blocked_actions']:
            print(f"Blocked runtime actions: {len(iter_res['blocked_actions'])}")
            for i, a in enumerate(iter_res['blocked_actions'], 1):
                print(f"  B{i:02d}. {a}")
        remaining_gpu_ids = _mismatched_gpu_ids(iter_res['state_after'], stage_res['target_state'])
        if remaining_gpu_ids:
            print_state_detail(iter_res['state_after'], title=f"[ITER {iter_res['iteration']}] GPUs not yet at target", include_physical=True, only_gpu_ids=remaining_gpu_ids)


def run_v1_stage_iterative(source_state: "ClusterState",
                           target_state: "ClusterState",
                           src_arrival,
                           tgt_arrival,
                           stage_name="stage_v1",
                           max_iters: int = 12) -> Dict[str, Any]:
    current_state = _deepcopy_state(source_state)
    iterations = []
    all_executed_actions = []
    final_blocked_actions = []
    final_plan = None
    reached_target = _matches_target_state(current_state, target_state)

    t0 = time.perf_counter()
    for iter_idx in range(1, max_iters + 1):
        if reached_target:
            break

        iter_plan = plan_naive_action_v1(
            source_state=current_state,
            target_state=target_state,
            src_arrival=src_arrival,
            tgt_arrival=tgt_arrival,
            stage_name=f"{stage_name}_iter{iter_idx}",
        )
        final_plan = iter_plan

        executed_actions = iter_plan["executed_actions"]
        blocked_actions = iter_plan["blocked_actions"]
        next_state = iter_plan["executed_state"]
        made_progress = (_state_semantic_signature(next_state) != _state_semantic_signature(current_state))
        reached_target = _matches_target_state(next_state, target_state)

        iterations.append({
            "iteration": iter_idx,
            "plan": iter_plan,
            "executed_actions": executed_actions,
            "blocked_actions": blocked_actions,
            "made_progress": made_progress,
            "reached_target": reached_target,
            "state_before": _deepcopy_state(current_state),
            "state_after": _deepcopy_state(next_state),
        })

        all_executed_actions.extend(copy.deepcopy(executed_actions))
        final_blocked_actions = copy.deepcopy(blocked_actions)
        current_state = next_state

        if reached_target or not made_progress:
            break

    elapsed_sec = time.perf_counter() - t0
    return {
        "stage_name": stage_name,
        "iterations": iterations,
        "iteration_count": len(iterations),
        "reached_target": reached_target,
        "elapsed_sec": elapsed_sec,
        "executed_actions": all_executed_actions,
        "blocked_actions": final_blocked_actions,
        "final_plan": final_plan,
        "executed_state": current_state,
        "target_state": _deepcopy_state(target_state),
    }


def plan_naive_action_v1(source_state: "ClusterState",
                         target_state: "ClusterState",
                         src_arrival,
                         tgt_arrival,
                         stage_name="stage_v1") -> Dict[str, Any]:
    source_state = _deepcopy_state(source_state)
    target_state = _deepcopy_state(target_state)
    _bootstrap_physical_ids_for_state(source_state)
    _ensure_state_metadata(target_state)
    required = _required_arrival_dict(src_arrival, tgt_arrival)

    src_map = _gpu_map_by_id(source_state)
    tgt_map = _gpu_map_by_id(target_state)
    all_gpu_ids = sorted(set(src_map) | set(tgt_map))

    free_pool = _build_initial_free_pool(source_state)
    coarse_actions = []
    fine_actions = []
    target_pid_map = {}

    for gid in all_gpu_ids:
        s = src_map.get(gid)
        t = tgt_map.get(gid)
        change = classify_gpu_change(s, t)

        if change == "keep_gpu":
            pid = _get_physical_id(source_state, gid)
            target_pid_map[gid] = pid
            coarse_actions.append({"type": "keep_gpu", "gpu_id": gid, "physical_gpu_id": pid})
            continue

        if change == "remove_gpu":
            pid = _get_physical_id(source_state, gid)
            safe = _safe_after_removing_gpu(source_state, s, required)
            coarse_actions.append({"type": "remove_gpu", "gpu_id": gid, "physical_gpu_id": pid, "safe_now": safe})
            if safe:
                fine_actions.extend([
                    {"type": "clear_gpu", "gpu_id": gid, "physical_gpu_id": pid},
                    {"type": "clear_template", "gpu_id": gid, "physical_gpu_id": pid},
                ])
                free_pool.append(pid)
            else:
                fine_actions.append({"type": "defer_remove_gpu", "gpu_id": gid, "physical_gpu_id": pid})
            continue

        if change == "create_gpu":
            pid = _alloc_from_free_pool(free_pool)
            target_pid_map[gid] = pid
            coarse_actions.append({"type": "create_gpu", "gpu_id": gid, "new_physical_gpu_id": pid, "template": t.template_str(), "alloc_policy": "free_pool_lifo"})
            fine_actions.extend([
                {"type": "allocate_gpu", "physical_gpu_id": pid, "policy": "free_pool_lifo"},
                {"type": "configure_full_template", "physical_gpu_id": pid, "template": t.template_str()},
                {"type": "place_target_layout", "gpu_id": gid, "physical_gpu_id": pid},
                {"type": "bind_target_gpu", "gpu_id": gid, "physical_gpu_id": pid},
            ])
            continue

        if change == "reconfiguration":
            old_pid = _get_physical_id(source_state, gid)
            new_pid = _alloc_from_free_pool(free_pool)
            target_pid_map[gid] = new_pid
            coarse_actions.append({
                "type": "reconfiguration",
                "gpu_id": gid,
                "source_physical_gpu_id": old_pid,
                "new_physical_gpu_id": new_pid,
                "src_template": s.template_str(),
                "tgt_template": t.template_str(),
                "alloc_policy": "free_pool_lifo",
            })
            fine_actions.extend([
                {"type": "allocate_gpu", "physical_gpu_id": new_pid, "policy": "free_pool_lifo"},
                {"type": "configure_full_template", "physical_gpu_id": new_pid, "template": t.template_str()},
                {"type": "place_target_layout", "gpu_id": gid, "physical_gpu_id": new_pid},
                {"type": "bind_target_gpu", "gpu_id": gid, "physical_gpu_id": new_pid},
                {"type": "clear_gpu", "gpu_id": gid, "physical_gpu_id": old_pid},
                {"type": "clear_template", "gpu_id": gid, "physical_gpu_id": old_pid},
            ])
            free_pool.append(old_pid)
            continue

        if change == "instance_diff":
            pid = _get_physical_id(source_state, gid)
            target_pid_map[gid] = pid
            inst_actions = diff_instances_within_same_template(s, t)
            coarse_actions.append({
                "type": "instance_diff",
                "gpu_id": gid,
                "physical_gpu_id": pid,
                "template": t.template_str(),
                "instance_changes": [a["type"] for a in inst_actions],
            })
            for ia in inst_actions:
                if ia["type"] == "keep":
                    continue
                elif ia["type"] == "batch_change":
                    fine_actions.append({
                        "type": "update_batch",
                        "gpu_id": gid,
                        "physical_gpu_id": pid,
                        "slot": ia["slot"],
                        "old_batch": ia["src"].batch,
                        "new_batch": ia["tgt"].batch,
                        "workload": ia["src"].workload,
                    })
                elif ia["type"] == "safe_remove_instance":
                    safe = _safe_after_removing_instance(source_state, ia["src"], required)
                    fine_actions.append({
                        "type": "remove_instance" if safe else "defer_remove_instance",
                        "gpu_id": gid,
                        "physical_gpu_id": pid,
                        "slot": ia["slot"],
                        "workload": ia["src"].workload,
                        "safe_now": safe,
                    })
                elif ia["type"] == "place_instance":
                    fine_actions.append({
                        "type": "place_instance",
                        "gpu_id": gid,
                        "physical_gpu_id": pid,
                        "slot": ia["slot"],
                        "workload": ia["tgt"].workload,
                        "batch": ia["tgt"].batch,
                    })
                elif ia["type"] == "workload_change":
                    safe = _safe_after_removing_instance(source_state, ia["src"], required)
                    if safe:
                        fine_actions.extend([
                            {"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": True},
                            {"type": "place_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["tgt"].workload, "batch": ia["tgt"].batch},
                        ])
                    else:
                        bridge_slot = _find_active_bridge_slot(
                            source_state=source_state,
                            target_state=target_state,
                            profile=ia["src"].profile,
                            avoid_gpu_id=gid,
                            avoid_slot=ia["slot"],
                        )
                        if bridge_slot is not None:
                            bridge_pid = _get_physical_id(source_state, bridge_slot["gpu_id"])
                            fine_actions.extend([
                                {
                                    "type": "bridge_place_instance",
                                    "gpu_id": int(bridge_slot["gpu_id"]),
                                    "physical_gpu_id": bridge_pid,
                                    "slot": bridge_slot["slot"],
                                    "workload": ia["src"].workload,
                                    "batch": ia["src"].batch,
                                    "mu": float(ia["src"].mu),
                                    "bridge_for": {
                                        "gpu_id": gid,
                                        "slot": ia["slot"],
                                        "old_workload": ia["src"].workload,
                                        "new_workload": ia["tgt"].workload,
                                    },
                                },
                                {"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": True, "bridged": True},
                                {"type": "place_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["tgt"].workload, "batch": ia["tgt"].batch, "bridged": True},
                            ])
                        else:
                            fine_actions.append({
                                "type": "defer_workload_change",
                                "gpu_id": gid,
                                "physical_gpu_id": pid,
                                "slot": ia["slot"],
                                "old_workload": ia["src"].workload,
                                "new_workload": ia["tgt"].workload,
                                "old_batch": ia["src"].batch,
                                "new_batch": ia["tgt"].batch,
                                "safe_now": False,
                            })
                elif ia["type"] == "instance_remove":
                    fine_actions.append({"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload if ia["src"] else None, "safe_now": True})
                elif ia["type"] == "instance_create":
                    fine_actions.append({"type": "place_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["tgt"].workload if ia["tgt"] else None, "batch": ia["tgt"].batch if ia["tgt"] else None})
            continue

    planned_state = _deepcopy_state(target_state)
    _ensure_state_metadata(planned_state)
    planned_state.metadata["physical_id_map"] = {int(gid): pid for gid, pid in target_pid_map.items()}
    planned_state.metadata["next_physical_idx"] = source_state.metadata.get("next_physical_idx", 0)

    executed_state = _simulate_fine_actions(
        source_state=source_state,
        target_state=planned_state,
        fine_actions=fine_actions,
        next_physical_idx=source_state.metadata.get("next_physical_idx", 0),
    )

    executed_actions, blocked_actions = _split_v1_actions(fine_actions)

    return {
        "stage_name": stage_name,
        "required": required,
        "coarse_actions": coarse_actions,
        "fine_actions": fine_actions,
        "executed_actions": executed_actions,
        "blocked_actions": blocked_actions,
        "planned_state": planned_state,
        "executed_state": executed_state,
        "free_pool_after_plan": list(free_pool),
    }


### Naive v1 experiment — stage 0

从空状态到 `target0` 跑 v1 iterative planner，并记录每轮 executed / blocked actions。

In [137]:

# ============================================================
# [Action Code Block V1-2] Stage 0: initial -> target0 -> iterative v1 planner -> executed0_v1
# ============================================================
source0_v1 = ClusterState(gpus=[], metadata={})
_ensure_state_metadata(source0_v1)

print_stage_current_and_target(source0_v1, target0, title_prefix="[V1 STAGE 0]")

stage0_v1_res = run_v1_stage_iterative(
    source_state=source0_v1,
    target_state=target0,
    src_arrival=np.zeros_like(arrival_rate, dtype=float),
    tgt_arrival=arrival_rate,
    stage_name="stage0_v1",
)
plan0_v1 = stage0_v1_res["final_plan"]
plan0_v1_elapsed_sec = stage0_v1_res["elapsed_sec"]
executed0_v1 = stage0_v1_res["executed_state"]
print_v1_stage_result(stage0_v1_res, title="[V1 STAGE 0] source0 -> target0 -> iterative v1 planner")
print(f"[V1 STAGE 0] iterative v1 planner elapsed: {plan0_v1_elapsed_sec:.6f} sec")
source1_v1 = _canonicalize_state_for_next_round(executed0_v1)
print_canonicalize_result(executed0_v1, source1_v1, title="[V1 STAGE 0] canonicalize for next MILP")


GPU count fixed from MILP    : 6
[V1 STAGE 0] CURRENT STATE
(empty)
current templates: []
[V1 STAGE 0] TARGET STATE
GPU 0: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 2: [0,4) 4g: vit_base / B64 / mu=1718.9838 | [4,7) 3g: vit_base / B64 / mu=1448.9833
GPU 3: [0,2) 2g: vgg16 / B16 / mu=483.8763 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: resnet50 / B16 / mu=351.0342
GPU 4: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu=1.1311
GPU 5: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6)

### Naive v1 experiment — stage 1

使用与 v0 相同的 `MILP + transition` 流程重新生成 `target1_seq_v1`，再跑 stage 内可迭代的 v1 planner，并在进入下一轮前 canonicalize。

In [138]:

# ============================================================
# [Action Code Block V1-3] Stage 1: source1_v1 -> MILP+transition -> target1_seq_v1 -> iterative v1 planner
# ============================================================
target1_seq_v1 = build_target_state_from_milp(
    milp_res=milp_res_up_manual,
    prev_state=source1_v1,
    abstract_template_topk=64,
    physical_layout_topk=32,
    per_gpu_layout_topk=4,
    verbose=False,
)

print_stage_current_and_target(source1_v1, target1_seq_v1, title_prefix="[V1 STAGE 1]")

stage1_v1_res = run_v1_stage_iterative(
    source_state=source1_v1,
    target_state=target1_seq_v1,
    src_arrival=arrival_rate,
    tgt_arrival=arrival_rate_up_manual,
    stage_name="stage1_v1",
)
plan1_v1 = stage1_v1_res["final_plan"]
plan1_v1_elapsed_sec = stage1_v1_res["elapsed_sec"]
executed1_v1 = stage1_v1_res["executed_state"]
print_v1_stage_result(stage1_v1_res, title="[V1 STAGE 1] source1_v1 -> target1_seq_v1 -> iterative v1 planner")
print(f"[V1 STAGE 1] iterative v1 planner elapsed: {plan1_v1_elapsed_sec:.6f} sec")
source2_v1 = _canonicalize_state_for_next_round(executed1_v1)
print_canonicalize_result(executed1_v1, source2_v1, title="[V1 STAGE 1] canonicalize for next MILP")


GPU count fixed from MILP    : 9
[V1 STAGE 1] CURRENT STATE
GPU 0 [A]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1 [B]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 2 [C]: [0,4) 4g: vit_base / B64 / mu=1718.9838 | [4,7) 3g: vit_base / B64 / mu=1448.9833
GPU 3 [D]: [0,2) 2g: vgg16 / B16 / mu=483.8763 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: resnet50 / B16 / mu=351.0342
GPU 4 [E]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu=1.1311
GPU 5 [F]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,

### Naive v1 experiment — stage 2

对 `source2_v1` 重新生成 `target2_seq_v1`，再跑 stage 内可迭代的 v1 planner，并在进入下一轮前 canonicalize。

In [139]:

# ============================================================
# [Action Code Block V1-4] Stage 2: source2_v1 -> MILP+transition -> target2_seq_v1 -> iterative v1 planner
# ============================================================
target2_seq_v1 = build_target_state_from_milp(
    milp_res=milp_res_t2_manual,
    prev_state=source2_v1,
    abstract_template_topk=64,
    physical_layout_topk=32,
    per_gpu_layout_topk=4,
    verbose=False,
)

print_stage_current_and_target(source2_v1, target2_seq_v1, title_prefix="[V1 STAGE 2]")

stage2_v1_res = run_v1_stage_iterative(
    source_state=source2_v1,
    target_state=target2_seq_v1,
    src_arrival=arrival_rate_up_manual,
    tgt_arrival=arrival_rate_t2_manual,
    stage_name="stage2_v1",
)
plan2_v1 = stage2_v1_res["final_plan"]
plan2_v1_elapsed_sec = stage2_v1_res["elapsed_sec"]
executed2_v1 = stage2_v1_res["executed_state"]
print_v1_stage_result(stage2_v1_res, title="[V1 STAGE 2] source2_v1 -> target2_seq_v1 -> iterative v1 planner")
print(f"[V1 STAGE 2] iterative v1 planner elapsed: {plan2_v1_elapsed_sec:.6f} sec")
source3_v1 = _canonicalize_state_for_next_round(executed2_v1)
print_canonicalize_result(executed2_v1, source3_v1, title="[V1 STAGE 2] canonicalize for next MILP")


GPU count fixed from MILP    : 6
[V1 STAGE 2] CURRENT STATE
GPU 0 [A]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1 [B]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 2 [C]: [0,4) 4g: vit_base / B64 / mu=1718.9838 | [4,7) 3g: vit_base / B64 / mu=1448.9833
GPU 3 [D]: [0,2) 2g: free | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: resnet50 / B16 / mu=351.0342
GPU 4 [E]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu=1.1311
GPU 5 [F]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu

### Naive v1 experiment — stage 3

对 `source3_v1` 重新生成 `target3_seq_v1`，再跑 stage 内可迭代的 v1 planner，并输出模板与时间对比。

In [140]:

# ============================================================
# [Action Code Block V1-5] Stage 3: source3_v1 -> MILP+transition -> target3_seq_v1 -> iterative v1 planner
# ============================================================
target3_seq_v1 = build_target_state_from_milp(
    milp_res=milp_res_up_3,
    prev_state=source3_v1,
    abstract_template_topk=64,
    physical_layout_topk=32,
    per_gpu_layout_topk=4,
    verbose=False,
)

print_stage_current_and_target(source3_v1, target3_seq_v1, title_prefix="[V1 STAGE 3]")

stage3_v1_res = run_v1_stage_iterative(
    source_state=source3_v1,
    target_state=target3_seq_v1,
    src_arrival=arrival_rate_t2_manual,
    tgt_arrival=arrival_rate_up_3,
    stage_name="stage3_v1",
)
plan3_v1 = stage3_v1_res["final_plan"]
plan3_v1_elapsed_sec = stage3_v1_res["elapsed_sec"]
executed3_v1 = stage3_v1_res["executed_state"]
print_v1_stage_result(stage3_v1_res, title="[V1 STAGE 3] source3_v1 -> target3_seq_v1 -> iterative v1 planner")
print(f"[V1 STAGE 3] iterative v1 planner elapsed: {plan3_v1_elapsed_sec:.6f} sec")


GPU count fixed from MILP    : 8
[V1 STAGE 3] CURRENT STATE
GPU 0 [A]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1 [B]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 2 [D]: [0,2) 2g: free | [2,3) 1g: vgg16 / B8 / mu=217.9872 | [3,4) 1g: free | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: resnet50 / B16 / mu=351.0342
GPU 3 [E]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu=1.1311
GPU 4 [F]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu=1.1311
GPU 5 [I]: [0,4) 4g: vit_base / B64 / mu=1448.9833 | [4,7) 3g: vit_base / B64 / mu=1448.9833
curren

### Naive v0 vs v1 summary

下面汇总两版 planner 的模板结果和各阶段用时，便于直接对比答案。

In [141]:

# ============================================================
# [Action Code Block V1-6] Summary / comparison
# ============================================================
v1_time_df = pd.DataFrame([
    {"planner": "naive_v1", "stage": 0, "elapsed_sec": plan0_v1_elapsed_sec},
    {"planner": "naive_v1", "stage": 1, "elapsed_sec": plan1_v1_elapsed_sec},
    {"planner": "naive_v1", "stage": 2, "elapsed_sec": plan2_v1_elapsed_sec},
    {"planner": "naive_v1", "stage": 3, "elapsed_sec": plan3_v1_elapsed_sec},
])
comparison_time_df = pd.concat([v0_time_df, v1_time_df], ignore_index=True)
print("[planner timing comparison]")
display(comparison_time_df)

print("\n[v0 templates]")
print("planner_final0:", [g.template_str() for g in planner_final0.real_gpus()])
print("planner_final1:", [g.template_str() for g in planner_final1.real_gpus()])
print("planner_final2:", [g.template_str() for g in planner_final2.real_gpus()])
print("planner_final3:", [g.template_str() for g in planner_final3.real_gpus()])

print("\n[v1 templates]")
print("executed0_v1:", [g.template_str() for g in executed0_v1.real_gpus()])
print("executed1_v1:", [g.template_str() for g in executed1_v1.real_gpus()])
print("executed2_v1:", [g.template_str() for g in executed2_v1.real_gpus()])
print("executed3_v1:", [g.template_str() for g in executed3_v1.real_gpus()])

print("\n[v0 physical bindings]")
for label, st in [("v0_stage0", planner_final0), ("v0_stage1", planner_final1), ("v0_stage2", planner_final2), ("v0_stage3", planner_final3)]:
    print(label, dict(sorted(st.metadata.get("physical_id_map", {}).items())))

print("\n[v1 physical bindings]")
for label, st in [("v1_stage0", executed0_v1), ("v1_stage1", executed1_v1), ("v1_stage2", executed2_v1), ("v1_stage3", executed3_v1)]:
    print(label, dict(sorted(st.metadata.get("physical_id_map", {}).items())))


[planner timing comparison]


,planner,stage,elapsed_sec
0,naive_v0,0,0.007044
1,naive_v0,1,0.001676
2,naive_v0,2,0.001302
3,naive_v0,3,0.001296
4,naive_v1,0,0.000796
5,naive_v1,1,0.003034
6,naive_v1,2,0.002861
7,naive_v1,3,0.001213



[v0 templates]
planner_final0: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
planner_final1: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3', '1+1+1+1+3', '4+3']
planner_final2: ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+3', '4+3']
planner_final3: ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+3', '4+3', '4+3']

[v1 templates]
executed0_v1: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
executed1_v1: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3', '1+1+1+1+3', '4+3']
executed2_v1: ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3']
executed3_v1: ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3', '4+3', '1+1+1+1+3']

[v0 physical bindings]
v0_stage0 {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F'}
v0_stage1 {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 8: 'I'}
v0_stag

### V1 bridge minimal demo

下面构造一个最小场景，专门触发 `bridge_place_instance`：

- `GPU 0` 上一个 `1g` 的 `gpt2` 需要改成 `resnet50`
- 直接删掉这个 `gpt2` 当前不安全
- `GPU 1` 上有一个 `target` 中仍保持空闲的 `1g` slot，可作为临时 bridge
- 同时 `target` 还要求把 `gpt2` 正式放到 `GPU 1` 的另一个 `1g` slot

因此预期行为是：

1. 第 1 轮先把 `gpt2` 临时放到 bridge slot，再完成原位 workload change，并把目标里的 `gpt2` 正式放到新位置
2. 第 2 轮再把 bridge slot 上那份临时 `gpt2` 清掉


In [142]:
# ============================================================
# [Action Code Block V1-7] Minimal bridge-trigger demo
# ============================================================
def _demo_lookup_mu(workload: str, profile: str, batch: int) -> float:
    row = feasible_option_df[
        (feasible_option_df["workload"] == workload)
        & (feasible_option_df["profile"] == profile)
        & (feasible_option_df["batch"] == batch)
    ].iloc[0]
    return float(row["mu"])

def _demo_make_1g_gpu(gpu_id: int, payload_by_slot: Dict[int, Optional[Tuple[str, int, float]]]) -> GPUState:
    insts = []
    for slot_idx in range(7):
        payload = payload_by_slot.get(slot_idx)
        if payload is None:
            insts.append(MigInstance(start=slot_idx, end=slot_idx + 1, profile="1g"))
        else:
            w, b, mu = payload
            insts.append(MigInstance(start=slot_idx, end=slot_idx + 1, profile="1g", workload=w, batch=int(b), mu=float(mu)))
    g = GPUState(gpu_id=gpu_id, source="real", instances=insts)
    g.sort_instances()
    return g

demo_mu_gpt2_1g_b1 = _demo_lookup_mu("gpt2", "1g", 1)
demo_mu_resnet50_1g_b16 = _demo_lookup_mu("resnet50", "1g", 16)

demo_source = ClusterState(gpus=[
    _demo_make_1g_gpu(0, {
        0: ("gpt2", 1, demo_mu_gpt2_1g_b1),
    }),
    _demo_make_1g_gpu(1, {}),
], metadata={})
_ensure_state_metadata(demo_source)
_set_physical_id(demo_source, 0, "A")
_set_physical_id(demo_source, 1, "B")

demo_target = ClusterState(gpus=[
    _demo_make_1g_gpu(0, {
        0: ("resnet50", 16, demo_mu_resnet50_1g_b16),
    }),
    _demo_make_1g_gpu(1, {
        0: ("gpt2", 1, demo_mu_gpt2_1g_b1),
    }),
], metadata={})
_ensure_state_metadata(demo_target)

demo_src_arr = np.zeros_like(arrival_rate, dtype=float)
demo_tgt_arr = np.zeros_like(arrival_rate, dtype=float)
demo_name_to_idx = {name: i for i, name in enumerate(WORKLOAD_NAMES)}
demo_src_arr[demo_name_to_idx["gpt2"]] = 1.0
demo_tgt_arr[demo_name_to_idx["gpt2"]] = 1.0
demo_tgt_arr[demo_name_to_idx["resnet50"]] = 1.0

print_stage_current_and_target(demo_source, demo_target, title_prefix="[V1 BRIDGE DEMO]")

bridge_demo_res = run_v1_stage_iterative(
    source_state=demo_source,
    target_state=demo_target,
    src_arrival=demo_src_arr,
    tgt_arrival=demo_tgt_arr,
    stage_name="bridge_demo_v1",
    max_iters=4,
)

print_v1_stage_result(bridge_demo_res, title="[V1 BRIDGE DEMO] iterative planner")

bridge_demo_bridge_actions = [a for a in bridge_demo_res["executed_actions"] if a.get("type") == "bridge_place_instance"]
print("-" * 90)
print(f"[V1 BRIDGE DEMO] bridge_place_instance count: {len(bridge_demo_bridge_actions)}")
for i, a in enumerate(bridge_demo_bridge_actions, 1):
    print(f"  G{i:02d}. {a}")


GPU count fixed from MILP    : 2
[V1 BRIDGE DEMO] CURRENT STATE
GPU 0 [A]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: free | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
GPU 1 [B]: [0,1) 1g: free | [1,2) 1g: free | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
current templates: ['1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
[V1 BRIDGE DEMO] TARGET STATE
GPU 0: [0,1) 1g: resnet50 / B16 / mu=351.0342 | [1,2) 1g: free | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
GPU 1: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: free | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
target templates: ['1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
[V1 BRIDGE DEMO] iterative planner
Stage: bridge_demo_v1
Iterations: 2
Reached target: True
Elapsed (s): 0.001196
Total executed runtime actions: 5
Final blocked runtime actions: 0
--------------------------------------------------------

## V2

### Naive v2 — drain-aware reorder-first planner

这一部分在 `v1` 的基础上继续加入两点：

1. `workload` 分类：`shared / new / retiring / inactive`
2. 对 `retiring` workload 的删除引入简单 `drain` 语义，并在单个 iteration 内优先执行 capacity-building / placement，再执行 cleanup

当前 notebook 里的 `drain` 先用离散 control iterations 近似：

- 第一次看到 retiring instance 要删除时，先 `mark_draining_instance`
- 下一轮进入 planner 前推进 drain counter
- drain 完成后才真正 remove


In [184]:
from typing import Dict, Any, List, Optional, Tuple
# ============================================================
# [Action Code Block V2-1] workload classification + drain-aware reorder-first helpers
# ============================================================
def classify_workloads_by_arrival(src_arr, tgt_arr) -> Dict[str, List[str]]:
    src_d = _arrival_dict_from_vector(src_arr)
    tgt_d = _arrival_dict_from_vector(tgt_arr)
    out = {"shared": [], "new": [], "retiring": [], "inactive": []}
    for w in WORKLOAD_NAMES:
        s = float(src_d.get(w, 0.0))
        t = float(tgt_d.get(w, 0.0))
        if s > 0 and t > 0:
            out["shared"].append(w)
        elif s <= 0 and t > 0:
            out["new"].append(w)
        elif s > 0 and t <= 0:
            out["retiring"].append(w)
        else:
            out["inactive"].append(w)
    return out

def _workload_class_name(classes: Dict[str, List[str]], workload: Optional[str]) -> str:
    if workload is None:
        return "none"
    for cls in ["shared", "new", "retiring", "inactive"]:
        if workload in classes.get(cls, []):
            return cls
    return "unknown"

def print_workload_classes(classes: Dict[str, List[str]], title="[WORKLOAD CLASSES]"):
    print("=" * 90)
    print(title)
    print("=" * 90)
    for cls in ["shared", "new", "retiring", "inactive"]:
        print(f"{cls:9s}: {classes.get(cls, [])}")

def _v2_slot_token(gpu_id: int, slot: Tuple[int, int, str]) -> str:
    s, e, p = slot
    return f"{int(gpu_id)}:{int(s)}:{int(e)}:{p}"

def _v2_drain_map(state: "ClusterState") -> Dict[str, int]:
    _ensure_state_metadata(state)
    return state.metadata.setdefault("v2_draining_instances", {})

def _v2_get_drain_remaining(state: "ClusterState", gpu_id: int, slot: Tuple[int, int, str]) -> Optional[int]:
    return _v2_drain_map(state).get(_v2_slot_token(gpu_id, slot))

def _v2_start_drain(state: "ClusterState", gpu_id: int, slot: Tuple[int, int, str], rounds: int = 1):
    _v2_drain_map(state)[_v2_slot_token(gpu_id, slot)] = int(rounds)

def _v2_clear_drain(state: "ClusterState", gpu_id: int, slot: Tuple[int, int, str]):
    _v2_drain_map(state).pop(_v2_slot_token(gpu_id, slot), None)

def _v2_advance_drain(state: "ClusterState"):
    dm = _v2_drain_map(state)
    for k in list(dm.keys()):
        dm[k] = max(0, int(dm[k]) - 1)

def _v2_progress_signature(state: "ClusterState"):
    dm = tuple(sorted((k, int(v)) for k, v in _v2_drain_map(state).items()))
    return (_state_semantic_signature(state), dm)

def _simulate_fine_actions_v2(source_state: "ClusterState", target_state: "ClusterState", fine_actions: List[Dict[str, Any]], next_physical_idx: int) -> "ClusterState":
    executed_state = _deepcopy_state(source_state)
    _ensure_state_metadata(executed_state)
    _bootstrap_physical_ids_for_state(executed_state)
    _ensure_state_metadata(target_state)
    target_map = _gpu_map_by_id(target_state)
    executed_state.metadata["v2_draining_instances"] = dict(source_state.metadata.get("v2_draining_instances", {}))

    for action in fine_actions:
        action_type = action["type"]

        if action_type in {"allocate_gpu", "configure_full_template", "configure_partial_profile", "place_target_layout", "defer_remove_gpu", "defer_remove_instance", "defer_workload_change"}:
            continue

        if action_type == "mark_draining_instance":
            _v2_start_drain(executed_state, int(action["gpu_id"]), tuple(action["slot"]), rounds=int(action.get("rounds", 1)))
            continue

        if action_type == "bind_target_gpu":
            gid = int(action["gpu_id"])
            pid = action["physical_gpu_id"]
            tgt_gpu = target_map.get(gid)
            if tgt_gpu is None:
                continue
            _replace_or_append_gpu(executed_state, copy.deepcopy(tgt_gpu))
            _set_physical_id(executed_state, gid, pid)
            continue

        if action_type == "clear_gpu":
            _remove_gpu_if_bound_to_pid(executed_state, int(action["gpu_id"]), action["physical_gpu_id"])
            continue

        if action_type == "clear_template":
            continue

        gid = int(action["gpu_id"])
        pid = action["physical_gpu_id"]
        cur_gpu = _get_gpu_by_id_mut(executed_state, gid)
        if cur_gpu is None or _get_physical_id(executed_state, gid) != pid:
            continue

        slot = tuple(action.get("slot", ()))
        cur_inst = _get_inst_by_slot(cur_gpu, slot) if slot else None
        tgt_gpu = target_map.get(gid)
        tgt_inst = _get_inst_by_slot(tgt_gpu, slot) if slot else None

        if action_type == "bridge_place_instance":
            if cur_inst is not None:
                cur_inst.workload = action.get("workload")
                cur_inst.batch = action.get("batch")
                cur_inst.mu = float(action.get("mu", 0.0))
                cur_inst.preserved = False
            continue

        if action_type in {"update_batch", "place_instance", "workload_change"}:
            if cur_inst is not None and tgt_inst is not None:
                _copy_inst_payload(cur_inst, tgt_inst)
            continue

        if action_type == "remove_instance":
            if cur_inst is not None:
                _copy_inst_payload(cur_inst, None)
            _v2_clear_drain(executed_state, gid, slot)
            continue

    executed_state.gpus = sorted(executed_state.real_gpus(), key=lambda x: x.gpu_id)
    executed_state.metadata["next_physical_idx"] = int(next_physical_idx)
    return executed_state

def print_v2_stage_result(stage_res: Dict[str, Any], title="[V2 STAGE RESULT]"):
    print("=" * 90)
    print(title)
    print("=" * 90)
    print(f"Stage: {stage_res['stage_name']}")
    print(f"Iterations: {stage_res['iteration_count']}")
    print(f"Reached target: {stage_res['reached_target']}")
    print(f"Elapsed (s): {stage_res['elapsed_sec']:.6f}")
    print_workload_classes(stage_res["workload_classes"], title=f"{title} workload classes")
    print(f"Total executed runtime actions: {len(stage_res['executed_actions'])}")
    print(f"Final blocked runtime actions: {len(stage_res['blocked_actions'])}")
    for iter_res in stage_res['iterations']:
        plan = iter_res['plan']
        print("-" * 90)
        print(f"Iteration {iter_res['iteration']}: progress={iter_res['made_progress']}, reached_target={iter_res['reached_target']}")
        print(f"Coarse actions: {len(plan['coarse_actions'])}")
        for i, a in enumerate(plan['coarse_actions'], 1):
            print(f"  C{i:02d}. {a}")
        print(f"Executed runtime actions: {len(iter_res['executed_actions'])}")
        for i, a in enumerate(iter_res['executed_actions'], 1):
            print(f"  X{i:02d}. {a}")
        if iter_res['blocked_actions']:
            print(f"Blocked runtime actions: {len(iter_res['blocked_actions'])}")
            for i, a in enumerate(iter_res['blocked_actions'], 1):
                print(f"  B{i:02d}. {a}")
        remaining_gpu_ids = _mismatched_gpu_ids(iter_res['state_after'], stage_res['target_state'])
        if remaining_gpu_ids:
            print_state_detail(iter_res['state_after'], title=f"[ITER {iter_res['iteration']}] GPUs not yet at target", include_physical=True, only_gpu_ids=remaining_gpu_ids)

def run_v2_stage_iterative(source_state: "ClusterState", target_state: "ClusterState", src_arrival, tgt_arrival, stage_name="stage_v2", max_iters: int = 12) -> Dict[str, Any]:
    current_state = _deepcopy_state(source_state)
    _ensure_state_metadata(current_state)
    iterations = []
    all_executed_actions = []
    final_blocked_actions = []
    final_plan = None
    workload_classes = classify_workloads_by_arrival(src_arrival, tgt_arrival)
    reached_target = _matches_target_state(current_state, target_state) and len(_v2_drain_map(current_state)) == 0

    t0 = time.perf_counter()
    for iter_idx in range(1, max_iters + 1):
        if reached_target:
            break
        _v2_advance_drain(current_state)
        iter_plan = plan_naive_action_v2(source_state=current_state, target_state=target_state, src_arrival=src_arrival, tgt_arrival=tgt_arrival, stage_name=f"{stage_name}_iter{iter_idx}")
        final_plan = iter_plan
        executed_actions = iter_plan["executed_actions"]
        blocked_actions = iter_plan["blocked_actions"]
        next_state = iter_plan["executed_state"]
        made_progress = (_v2_progress_signature(next_state) != _v2_progress_signature(current_state))
        reached_target = _matches_target_state(next_state, target_state) and len(_v2_drain_map(next_state)) == 0
        iterations.append({"iteration": iter_idx, "plan": iter_plan, "executed_actions": executed_actions, "blocked_actions": blocked_actions, "made_progress": made_progress, "reached_target": reached_target, "state_before": _deepcopy_state(current_state), "state_after": _deepcopy_state(next_state)})
        all_executed_actions.extend(copy.deepcopy(executed_actions))
        final_blocked_actions = copy.deepcopy(blocked_actions)
        current_state = next_state
        if reached_target or not made_progress:
            break

    elapsed_sec = time.perf_counter() - t0
    return {"stage_name": stage_name, "iterations": iterations, "iteration_count": len(iterations), "reached_target": reached_target, "elapsed_sec": elapsed_sec, "executed_actions": all_executed_actions, "blocked_actions": final_blocked_actions, "final_plan": final_plan, "executed_state": current_state, "target_state": _deepcopy_state(target_state), "workload_classes": workload_classes}

def plan_naive_action_v2(source_state: "ClusterState", target_state: "ClusterState", src_arrival, tgt_arrival, stage_name="stage_v2") -> Dict[str, Any]:
    source_state = _deepcopy_state(source_state)
    target_state = _deepcopy_state(target_state)
    _bootstrap_physical_ids_for_state(source_state)
    _ensure_state_metadata(target_state)
    _ensure_state_metadata(source_state)
    required = _required_arrival_dict(src_arrival, tgt_arrival)
    workload_classes = classify_workloads_by_arrival(src_arrival, tgt_arrival)

    src_map = _gpu_map_by_id(source_state)
    tgt_map = _gpu_map_by_id(target_state)
    all_gpu_ids = sorted(set(src_map) | set(tgt_map))
    free_pool = _build_initial_free_pool(source_state)

    coarse_actions = []
    drain_actions = []
    capacity_actions = []
    cleanup_actions = []
    blocked_actions = []
    target_pid_map = {}

    for gid in all_gpu_ids:
        s = src_map.get(gid)
        t = tgt_map.get(gid)
        change = classify_gpu_change(s, t)

        if change == "keep_gpu":
            pid = _get_physical_id(source_state, gid)
            target_pid_map[gid] = pid
            coarse_actions.append({"type": "keep_gpu", "gpu_id": gid, "physical_gpu_id": pid})
            continue

        if change == "remove_gpu":
            pid = _get_physical_id(source_state, gid)
            safe = _safe_after_removing_gpu(source_state, s, required)
            coarse_actions.append({"type": "remove_gpu", "gpu_id": gid, "physical_gpu_id": pid, "safe_now": safe})
            if safe:
                cleanup_actions.extend([{"type": "clear_gpu", "gpu_id": gid, "physical_gpu_id": pid}, {"type": "clear_template", "gpu_id": gid, "physical_gpu_id": pid}])
                free_pool.append(pid)
            else:
                blocked_actions.append({"type": "defer_remove_gpu", "gpu_id": gid, "physical_gpu_id": pid})
            continue

        if change == "create_gpu":
            pid = _alloc_from_free_pool(free_pool)
            target_pid_map[gid] = pid
            coarse_actions.append({"type": "create_gpu", "gpu_id": gid, "new_physical_gpu_id": pid, "template": t.template_str(), "alloc_policy": "free_pool_lifo"})
            capacity_actions.extend([{"type": "allocate_gpu", "physical_gpu_id": pid, "policy": "free_pool_lifo"}, {"type": "configure_full_template", "physical_gpu_id": pid, "template": t.template_str()}, {"type": "place_target_layout", "gpu_id": gid, "physical_gpu_id": pid}, {"type": "bind_target_gpu", "gpu_id": gid, "physical_gpu_id": pid}])
            continue

        if change == "reconfiguration":
            old_pid = _get_physical_id(source_state, gid)
            new_pid = _alloc_from_free_pool(free_pool)
            target_pid_map[gid] = new_pid
            coarse_actions.append({"type": "reconfiguration", "gpu_id": gid, "source_physical_gpu_id": old_pid, "new_physical_gpu_id": new_pid, "src_template": s.template_str(), "tgt_template": t.template_str(), "alloc_policy": "free_pool_lifo"})
            capacity_actions.extend([{"type": "allocate_gpu", "physical_gpu_id": new_pid, "policy": "free_pool_lifo"}, {"type": "configure_full_template", "physical_gpu_id": new_pid, "template": t.template_str()}, {"type": "place_target_layout", "gpu_id": gid, "physical_gpu_id": new_pid}, {"type": "bind_target_gpu", "gpu_id": gid, "physical_gpu_id": new_pid}])
            cleanup_actions.extend([{"type": "clear_gpu", "gpu_id": gid, "physical_gpu_id": old_pid}, {"type": "clear_template", "gpu_id": gid, "physical_gpu_id": old_pid}])
            free_pool.append(old_pid)
            continue

        if change == "instance_diff":
            pid = _get_physical_id(source_state, gid)
            target_pid_map[gid] = pid
            inst_actions = diff_instances_within_same_template(s, t)
            coarse_actions.append({"type": "instance_diff", "gpu_id": gid, "physical_gpu_id": pid, "template": t.template_str(), "instance_changes": [a["type"] for a in inst_actions]})
            for ia in inst_actions:
                if ia["type"] == "keep":
                    continue
                if ia["type"] == "batch_change":
                    capacity_actions.append({"type": "update_batch", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "old_batch": ia["src"].batch, "new_batch": ia["tgt"].batch, "workload": ia["src"].workload})
                    continue
                if ia["type"] == "place_instance":
                    capacity_actions.append({"type": "place_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["tgt"].workload, "batch": ia["tgt"].batch})
                    continue
                if ia["type"] == "safe_remove_instance":
                    wcls = _workload_class_name(workload_classes, ia["src"].workload)
                    if wcls == "retiring":
                        rem = _v2_get_drain_remaining(source_state, gid, ia["slot"])
                        if rem is None:
                            drain_actions.append({"type": "mark_draining_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "rounds": 1})
                        elif rem > 0:
                            blocked_actions.append({"type": "defer_remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": False, "drain_remaining": rem})
                        else:
                            cleanup_actions.append({"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": True, "drained": True})
                    else:
                        safe = _safe_after_removing_instance(source_state, ia["src"], required)
                        if safe:
                            cleanup_actions.append({"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": True})
                        else:
                            blocked_actions.append({"type": "defer_remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": False})
                    continue
                if ia["type"] == "workload_change":
                    old_cls = _workload_class_name(workload_classes, ia["src"].workload)
                    safe = _safe_after_removing_instance(source_state, ia["src"], required)
                    if old_cls == "retiring":
                        rem = _v2_get_drain_remaining(source_state, gid, ia["slot"])
                        if rem is None:
                            drain_actions.append({"type": "mark_draining_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "rounds": 1})
                        elif rem > 0:
                            blocked_actions.append({"type": "defer_workload_change", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "old_workload": ia["src"].workload, "new_workload": ia["tgt"].workload, "old_batch": ia["src"].batch, "new_batch": ia["tgt"].batch, "safe_now": False, "drain_remaining": rem})
                        else:
                            cleanup_actions.append({"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": True, "drained": True})
                            capacity_actions.append({"type": "place_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["tgt"].workload, "batch": ia["tgt"].batch, "after_drain": True})
                    elif safe:
                        cleanup_actions.append({"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": True})
                        capacity_actions.append({"type": "place_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["tgt"].workload, "batch": ia["tgt"].batch})
                    else:
                        bridge_slot = _find_active_bridge_slot(source_state=source_state, target_state=target_state, profile=ia["src"].profile, avoid_gpu_id=gid, avoid_slot=ia["slot"])
                        if bridge_slot is not None:
                            bridge_pid = _get_physical_id(source_state, bridge_slot["gpu_id"])
                            capacity_actions.extend([{"type": "bridge_place_instance", "gpu_id": int(bridge_slot["gpu_id"]), "physical_gpu_id": bridge_pid, "slot": bridge_slot["slot"], "workload": ia["src"].workload, "batch": ia["src"].batch, "mu": float(ia["src"].mu), "bridge_for": {"gpu_id": gid, "slot": ia["slot"], "old_workload": ia["src"].workload, "new_workload": ia["tgt"].workload}}, {"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": True, "bridged": True}, {"type": "place_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["tgt"].workload, "batch": ia["tgt"].batch, "bridged": True}])
                        else:
                            blocked_actions.append({"type": "defer_workload_change", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "old_workload": ia["src"].workload, "new_workload": ia["tgt"].workload, "old_batch": ia["src"].batch, "new_batch": ia["tgt"].batch, "safe_now": False})
                    continue
                if ia["type"] == "instance_remove":
                    cleanup_actions.append({"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload if ia["src"] else None, "safe_now": True})
                    continue
                if ia["type"] == "instance_create":
                    capacity_actions.append({"type": "place_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["tgt"].workload if ia["tgt"] else None, "batch": ia["tgt"].batch if ia["tgt"] else None})
                    continue

    fine_actions = drain_actions + capacity_actions + cleanup_actions + blocked_actions
    planned_state = _deepcopy_state(target_state)
    _ensure_state_metadata(planned_state)
    planned_state.metadata["physical_id_map"] = {int(gid): pid for gid, pid in target_pid_map.items()}
    planned_state.metadata["next_physical_idx"] = source_state.metadata.get("next_physical_idx", 0)
    executed_state = _simulate_fine_actions_v2(source_state=source_state, target_state=planned_state, fine_actions=fine_actions, next_physical_idx=source_state.metadata.get("next_physical_idx", 0))
    executed_actions, blocked = _split_v1_actions(fine_actions)
    return {"stage_name": stage_name, "required": required, "coarse_actions": coarse_actions, "fine_actions": fine_actions, "executed_actions": executed_actions, "blocked_actions": blocked, "planned_state": planned_state, "executed_state": executed_state, "workload_classes": workload_classes, "free_pool_after_plan": list(free_pool)}


In [185]:
# ============================================================
# [Action Code Block V2-1B] runtime-state / phased-plan override
# ============================================================
def _v2_runtime_map(state: "ClusterState") -> Dict[str, Dict[str, Any]]:
    _ensure_state_metadata(state)
    return state.metadata.setdefault("v2_runtime_instances", {})

def _v2_get_runtime_entry(state: "ClusterState", gpu_id: int, slot: Tuple[int, int, str]) -> Dict[str, Any]:
    token = _v2_slot_token(gpu_id, slot)
    rm = _v2_runtime_map(state)
    if token not in rm:
        rm[token] = {"queued": 0, "inflight": 0, "accepting_new": True, "rerouted_to": None}
    return rm[token]

def _v2_clear_runtime_entry(state: "ClusterState", gpu_id: int, slot: Tuple[int, int, str]):
    _v2_runtime_map(state).pop(_v2_slot_token(gpu_id, slot), None)

def _v2_nonfree_instances(gpu: "GPUState") -> List["MigInstance"]:
    return [inst for inst in gpu.instances if getattr(inst, "workload", None)]

def _v2_slot_label(gpu_id: int, slot: Tuple[int, int, str], physical_gpu_id: Optional[str] = None) -> str:
    s, e, p = slot
    base = f"gpu{gpu_id}:{p}[{s},{e})"
    return f"{base}@{physical_gpu_id}" if physical_gpu_id is not None else base

def _v2_prepare_source_runtime(source_state: "ClusterState", target_state: "ClusterState", default_queued: int = 2, default_inflight: int = 1) -> "ClusterState":
    st = _deepcopy_state(source_state)
    _ensure_state_metadata(st)
    _bootstrap_physical_ids_for_state(st)
    if st.metadata.get("v2_runtime_instances"):
        return st
    for g in st.real_gpus():
        for inst in _v2_nonfree_instances(g):
            slot = (inst.start, inst.end, inst.profile)
            rt = _v2_get_runtime_entry(st, g.gpu_id, slot)
            rt.update({"queued": 0, "inflight": 0, "accepting_new": True, "rerouted_to": None})
    src_map = _gpu_map_by_id(st)
    tgt_map = _gpu_map_by_id(target_state)
    all_gpu_ids = sorted(set(src_map) | set(tgt_map))
    for gid in all_gpu_ids:
        s = src_map.get(gid)
        t = tgt_map.get(gid)
        change = classify_gpu_change(s, t)
        if change in {"remove_gpu", "reconfiguration"} and s is not None:
            for inst in _v2_nonfree_instances(s):
                slot = (inst.start, inst.end, inst.profile)
                rt = _v2_get_runtime_entry(st, gid, slot)
                rt["queued"] = max(int(rt.get("queued", 0)), int(default_queued))
                rt["inflight"] = max(int(rt.get("inflight", 0)), int(default_inflight))
                rt["accepting_new"] = True
        elif change == "instance_diff" and s is not None and t is not None:
            for ia in diff_instances_within_same_template(s, t):
                if ia["type"] in {"safe_remove_instance", "workload_change"} and ia.get("src") is not None:
                    slot = ia["slot"]
                    rt = _v2_get_runtime_entry(st, gid, slot)
                    rt["queued"] = max(int(rt.get("queued", 0)), int(default_queued))
                    rt["inflight"] = max(int(rt.get("inflight", 0)), int(default_inflight))
                    rt["accepting_new"] = True
    return st

def _v2_runtime_rows(state: "ClusterState") -> List[Dict[str, Any]]:
    rows = []
    for g in state.real_gpus():
        pid = _get_physical_id(state, g.gpu_id)
        for inst in _v2_nonfree_instances(g):
            slot = (inst.start, inst.end, inst.profile)
            rt = _v2_get_runtime_entry(state, g.gpu_id, slot)
            rem = _v2_get_drain_remaining(state, g.gpu_id, slot)
            if int(rt.get("queued", 0)) > 0 or int(rt.get("inflight", 0)) > 0 or not bool(rt.get("accepting_new", True)) or rem is not None:
                rows.append({
                    "gpu_id": g.gpu_id,
                    "physical_gpu_id": pid,
                    "slot": slot,
                    "workload": inst.workload,
                    "queued": int(rt.get("queued", 0)),
                    "inflight": int(rt.get("inflight", 0)),
                    "accepting_new": bool(rt.get("accepting_new", True)),
                    "drain_remaining": rem,
                    "rerouted_to": rt.get("rerouted_to"),
                })
    return rows

def _v2_print_runtime_assumptions(state: "ClusterState", title="[V2 RUNTIME ASSUMPTIONS]"):
    print("=" * 90)
    print(title)
    print("=" * 90)
    rows = _v2_runtime_rows(state)
    if not rows:
        print("(none)")
        return
    for r in rows:
        slot = _v2_slot_label(r["gpu_id"], r["slot"], r["physical_gpu_id"])
        print(f"- {slot}: {r['workload']} | queued={r['queued']} inflight={r['inflight']} accepting_new={r['accepting_new']} drain_remaining={r['drain_remaining']} rerouted_to={r['rerouted_to']}")

def _v2_advance_drain(state: "ClusterState"):
    dm = _v2_drain_map(state)
    rm = _v2_runtime_map(state)
    for k in list(dm.keys()):
        dm[k] = max(0, int(dm[k]) - 1)
        if k in rm:
            rm[k]["inflight"] = int(dm[k])

def _v2_progress_signature(state: "ClusterState"):
    dm = tuple(sorted((k, int(v)) for k, v in _v2_drain_map(state).items()))
    rm = tuple(sorted((k, int(v.get("queued", 0)), int(v.get("inflight", 0)), bool(v.get("accepting_new", True)), str(v.get("rerouted_to"))) for k, v in _v2_runtime_map(state).items()))
    return (_state_semantic_signature(state), dm, rm)

def _v2_reconfig_map(state: "ClusterState") -> Dict[str, Dict[str, Any]]:
    _ensure_state_metadata(state)
    return state.metadata.setdefault("v2_reconfig_targets", {})

def _v2_active_pid_set(state: "ClusterState") -> set:
    out = {pid for _, pid in sorted(state.metadata.get("physical_id_map", {}).items()) if pid is not None}
    for rec in _v2_reconfig_map(state).values():
        pid = rec.get("physical_gpu_id")
        if pid is not None:
            out.add(pid)
    return out

def _v2_peak_from_actions(state_before: "ClusterState", executed_actions: List[Dict[str, Any]]) -> int:
    active = set(_v2_active_pid_set(state_before))
    peak = len(active)
    for a in executed_actions:
        t = a.get("type")
        if t == "allocate_gpu":
            pid = a.get("physical_gpu_id")
            if pid is not None:
                active.add(pid)
        elif t == "clear_gpu":
            pid = a.get("physical_gpu_id")
            if pid is not None and pid in active:
                active.remove(pid)
        peak = max(peak, len(active))
    return peak

def _v2_format_action_short(a: Dict[str, Any]) -> str:
    t = a.get("type")
    if t in {"stop_accepting_new", "mark_draining_instance", "remove_instance", "place_instance", "reroute_queued_tasks"}:
        return f"{t} { _v2_slot_label(int(a['gpu_id']), tuple(a['slot']), a.get('physical_gpu_id')) } {a.get('workload', '')}".strip()
    if t in {"clear_gpu", "clear_template", "bind_target_gpu"}:
        return f"{t} gpu{a.get('gpu_id')}@{a.get('physical_gpu_id')}"
    if t in {"allocate_gpu", "configure_full_template", "place_target_layout"}:
        return str(a)
    return str(a)

def _v2_is_drain_barrier_item(it: Dict[str, Any]) -> bool:
    phase = str(it.get("current_phase", ""))
    blocked_by = str(it.get("blocked_by", ""))
    return ("drain" in phase) or phase in {"stop_accepting_new", "reroute_old_queue", "stop_accepting_old_gpu"} or ("drain" in blocked_by)

def _v2_takeover_candidates(source_state: "ClusterState", target_state: "ClusterState", workload: str, exclude_gpu_id: Optional[int] = None, exclude_slot: Optional[Tuple[int, int, str]] = None) -> List[Dict[str, Any]]:
    out = []
    seen = set()
    tgt_map = _gpu_map_by_id(target_state)
    for g in source_state.real_gpus():
        pid = _get_physical_id(source_state, g.gpu_id)
        tgt_gpu = tgt_map.get(g.gpu_id)
        for inst in _v2_nonfree_instances(g):
            slot = (inst.start, inst.end, inst.profile)
            if g.gpu_id == exclude_gpu_id and slot == exclude_slot:
                continue
            if inst.workload != workload:
                continue
            tgt_inst = _get_inst_by_slot(tgt_gpu, slot) if tgt_gpu is not None else None
            if tgt_inst is not None and tgt_inst.workload == workload:
                key = (g.gpu_id, slot)
                seen.add(key)
                out.append({"kind": "unchanged", "gpu_id": g.gpu_id, "slot": slot, "physical_gpu_id": pid})
    for g in target_state.real_gpus():
        for inst in _v2_nonfree_instances(g):
            slot = (inst.start, inst.end, inst.profile)
            if g.gpu_id == exclude_gpu_id and slot == exclude_slot:
                continue
            if inst.workload != workload:
                continue
            key = (g.gpu_id, slot)
            if key in seen:
                continue
            out.append({"kind": "target-backed", "gpu_id": g.gpu_id, "slot": slot, "physical_gpu_id": None})
    return out

def _v2_takeover_label(cands: List[Dict[str, Any]]) -> Optional[str]:
    if not cands:
        return None
    c = cands[0]
    return f"{c['kind']}[{_v2_slot_label(c['gpu_id'], c['slot'], c.get('physical_gpu_id'))}]"

def _v2_item(item_id: str, item_type: str, current_phase: str, status: str, blocked_by: Optional[str] = None, **kwargs) -> Dict[str, Any]:
    out = {"id": item_id, "type": item_type, "current_phase": current_phase, "status": status, "blocked_by": blocked_by}
    out.update(kwargs)
    return out

def _v2_split_actions(fine_actions: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    executed = []
    blocked = []
    for a in fine_actions:
        if str(a.get("type", "")).startswith("defer_"):
            blocked.append(a)
        else:
            executed.append(a)
    return executed, blocked

def _simulate_fine_actions_v2(source_state: "ClusterState", target_state: "ClusterState", fine_actions: List[Dict[str, Any]], next_physical_idx: int) -> "ClusterState":
    executed_state = _deepcopy_state(source_state)
    _ensure_state_metadata(executed_state)
    _bootstrap_physical_ids_for_state(executed_state)
    _ensure_state_metadata(target_state)
    target_map = _gpu_map_by_id(target_state)
    executed_state.metadata["v2_draining_instances"] = dict(source_state.metadata.get("v2_draining_instances", {}))
    executed_state.metadata["v2_runtime_instances"] = copy.deepcopy(source_state.metadata.get("v2_runtime_instances", {}))
    executed_state.metadata["v2_reconfig_targets"] = copy.deepcopy(source_state.metadata.get("v2_reconfig_targets", {}))

    for action in fine_actions:
        action_type = action["type"]

        if action_type in {"allocate_gpu", "configure_full_template", "configure_partial_profile", "place_target_layout", "defer_remove_gpu", "defer_remove_instance", "defer_workload_change"}:
            continue

        if action_type == "stop_accepting_new":
            rt = _v2_get_runtime_entry(executed_state, int(action["gpu_id"]), tuple(action["slot"]))
            rt["accepting_new"] = False
            continue

        if action_type == "reroute_queued_tasks":
            rt = _v2_get_runtime_entry(executed_state, int(action["gpu_id"]), tuple(action["slot"]))
            rt["queued"] = 0
            rt["rerouted_to"] = action.get("to")
            continue

        if action_type == "mark_draining_instance":
            gid = int(action["gpu_id"])
            slot = tuple(action["slot"])
            rt = _v2_get_runtime_entry(executed_state, gid, slot)
            rt["accepting_new"] = False
            rounds = max(int(action.get("rounds", 1)), int(rt.get("inflight", 0)))
            _v2_start_drain(executed_state, gid, slot, rounds=rounds)
            rt["inflight"] = rounds
            continue

        if action_type == "mark_reconfig_target_prepared":
            _v2_reconfig_map(executed_state)[str(int(action["gpu_id"]))] = {"physical_gpu_id": action["physical_gpu_id"], "template": action.get("template")}
            continue

        if action_type == "bind_target_gpu":
            gid = int(action["gpu_id"])
            pid = action["physical_gpu_id"]
            tgt_gpu = target_map.get(gid)
            if tgt_gpu is None:
                continue
            _replace_or_append_gpu(executed_state, copy.deepcopy(tgt_gpu))
            _set_physical_id(executed_state, gid, pid)
            for inst in _v2_nonfree_instances(tgt_gpu):
                slot = (inst.start, inst.end, inst.profile)
                rt = _v2_get_runtime_entry(executed_state, gid, slot)
                rt.setdefault("queued", 0)
                rt.setdefault("inflight", 0)
                rt["accepting_new"] = True
            _v2_reconfig_map(executed_state).pop(str(gid), None)
            continue

        if action_type == "clear_gpu":
            gid = int(action["gpu_id"])
            cur_gpu = _get_gpu_by_id_mut(executed_state, gid)
            if cur_gpu is not None:
                for inst in list(cur_gpu.instances):
                    slot = (inst.start, inst.end, inst.profile)
                    _v2_clear_runtime_entry(executed_state, gid, slot)
                    _v2_clear_drain(executed_state, gid, slot)
            _remove_gpu_if_bound_to_pid(executed_state, gid, action["physical_gpu_id"])
            continue

        if action_type == "clear_template":
            continue

        gid = int(action["gpu_id"])
        pid = action["physical_gpu_id"]
        cur_gpu = _get_gpu_by_id_mut(executed_state, gid)
        if cur_gpu is None or _get_physical_id(executed_state, gid) != pid:
            continue

        slot = tuple(action.get("slot", ()))
        cur_inst = _get_inst_by_slot(cur_gpu, slot) if slot else None
        tgt_gpu = target_map.get(gid)
        tgt_inst = _get_inst_by_slot(tgt_gpu, slot) if slot else None

        if action_type == "bridge_place_instance":
            if cur_inst is not None:
                cur_inst.workload = action.get("workload")
                cur_inst.batch = action.get("batch")
                cur_inst.mu = float(action.get("mu", 0.0))
                cur_inst.preserved = False
                rt = _v2_get_runtime_entry(executed_state, gid, slot)
                rt["accepting_new"] = True
            continue

        if action_type in {"update_batch", "place_instance", "workload_change"}:
            if cur_inst is not None and tgt_inst is not None:
                _copy_inst_payload(cur_inst, tgt_inst)
                rt = _v2_get_runtime_entry(executed_state, gid, slot)
                rt.setdefault("queued", 0)
                rt.setdefault("inflight", 0)
                rt["accepting_new"] = True
            continue

        if action_type == "remove_instance":
            if cur_inst is not None:
                _copy_inst_payload(cur_inst, None)
            _v2_clear_drain(executed_state, gid, slot)
            _v2_clear_runtime_entry(executed_state, gid, slot)
            continue

    executed_state.gpus = sorted(executed_state.real_gpus(), key=lambda x: x.gpu_id)
    executed_state.metadata["next_physical_idx"] = int(next_physical_idx)
    return executed_state

def plan_naive_action_v2(source_state: "ClusterState", target_state: "ClusterState", src_arrival, tgt_arrival, stage_name="stage_v2") -> Dict[str, Any]:
    source_state = _v2_prepare_source_runtime(source_state, target_state)
    target_state = _deepcopy_state(target_state)
    _bootstrap_physical_ids_for_state(source_state)
    _ensure_state_metadata(target_state)
    _ensure_state_metadata(source_state)
    required = _required_arrival_dict(src_arrival, tgt_arrival)
    workload_classes = classify_workloads_by_arrival(src_arrival, tgt_arrival)
    src_map = _gpu_map_by_id(source_state)
    tgt_map = _gpu_map_by_id(target_state)
    all_gpu_ids = sorted(set(src_map) | set(tgt_map))
    free_pool = _build_initial_free_pool(source_state)
    coarse_actions = []
    phase_actions = []
    capacity_actions = []
    cleanup_actions = []
    finalize_actions = []
    blocked_actions = []
    target_pid_map = {}
    plan_items = []

    def add_slot_barrier(gid, pid, slot, workload, item_type, item_id, safe_after_drain=True, reroute_title="reroute_old_queue"):
        rt = _v2_get_runtime_entry(source_state, gid, slot)
        rem = _v2_get_drain_remaining(source_state, gid, slot)
        cands = _v2_takeover_candidates(source_state, target_state, workload, exclude_gpu_id=gid, exclude_slot=slot)
        takeover = _v2_takeover_label(cands)
        local_actions = []
        phase = "post_drain_ready"
        status = "ready"
        blocked_by = None
        if (int(rt.get("queued", 0)) > 0 or int(rt.get("inflight", 0)) > 0 or bool(rt.get("accepting_new", True))) and takeover is None:
            phase = "takeover_ready"
            status = "blocked"
            blocked_by = "no_takeover_capacity"
        else:
            if bool(rt.get("accepting_new", True)):
                local_actions.append({"type": "stop_accepting_new", "gpu_id": gid, "physical_gpu_id": pid, "slot": slot, "workload": workload})
                phase = "stop_accepting_new"
                status = "in_progress"
            if int(rt.get("queued", 0)) > 0:
                local_actions.append({"type": "reroute_queued_tasks", "gpu_id": gid, "physical_gpu_id": pid, "slot": slot, "workload": workload, "queued": int(rt.get("queued", 0)), "to": takeover})
                phase = reroute_title
                status = "in_progress"
            if rem is None and int(rt.get("inflight", 0)) > 0:
                local_actions.append({"type": "mark_draining_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": slot, "workload": workload, "rounds": int(rt.get("inflight", 0))})
                phase = "drain_old"
                status = "blocked"
                blocked_by = "drain_started"
            elif rem is not None and int(rem) > 0:
                phase = "drain_old"
                status = "blocked"
                blocked_by = "inflight_tasks_not_zero"
        plan_items.append(_v2_item(item_id=item_id, item_type=item_type, current_phase=phase, status=status, blocked_by=blocked_by, gpu_id=gid, physical_gpu_id=pid, slot=slot, workload=workload, takeover=takeover, queued=int(rt.get("queued", 0)), inflight=int(rt.get("inflight", 0)), drain_remaining=rem, capacity_safe=bool(safe_after_drain)))
        return local_actions, phase, status, blocked_by

    for gid in all_gpu_ids:
        s = src_map.get(gid)
        t = tgt_map.get(gid)
        change = classify_gpu_change(s, t)

        if change == "keep_gpu":
            pid = _get_physical_id(source_state, gid)
            target_pid_map[gid] = pid
            coarse_actions.append({"type": "keep_gpu", "gpu_id": gid, "physical_gpu_id": pid})
            continue

        if change == "create_gpu":
            pid = _alloc_from_free_pool(free_pool)
            target_pid_map[gid] = pid
            coarse_actions.append({"type": "create_gpu", "gpu_id": gid, "new_physical_gpu_id": pid, "template": t.template_str(), "alloc_policy": "free_pool_lifo"})
            capacity_actions.extend([{"type": "allocate_gpu", "physical_gpu_id": pid, "policy": "free_pool_lifo"}, {"type": "configure_full_template", "physical_gpu_id": pid, "template": t.template_str()}, {"type": "place_target_layout", "gpu_id": gid, "physical_gpu_id": pid}, {"type": "bind_target_gpu", "gpu_id": gid, "physical_gpu_id": pid}])
            plan_items.append(_v2_item(item_id=f"CREATE_gpu{gid}", item_type="create_gpu", current_phase="prepare_target_side", status="ready", gpu_id=gid, physical_gpu_id=pid, template=t.template_str()))
            continue

        if change == "remove_gpu":
            pid = _get_physical_id(source_state, gid)
            safe = _safe_after_removing_gpu(source_state, s, required)
            coarse_actions.append({"type": "remove_gpu", "gpu_id": gid, "physical_gpu_id": pid, "safe_now": safe})
            slot_blocked = False
            any_stop = False
            any_reroute = False
            any_drain = False
            for inst in _v2_nonfree_instances(s):
                slot = (inst.start, inst.end, inst.profile)
                acts, phase, status, blocked_by = add_slot_barrier(gid, pid, slot, inst.workload, "remove_gpu_slot", f"REMOVE_gpu{gid}_{slot[0]}_{slot[1]}_{slot[2]}", safe_after_drain=safe)
                phase_actions.extend(acts)
                any_stop = any_stop or any(a["type"] == "stop_accepting_new" for a in acts)
                any_reroute = any_reroute or any(a["type"] == "reroute_queued_tasks" for a in acts)
                any_drain = any_drain or (phase == "drain_old")
                slot_blocked = slot_blocked or (status == "blocked")
            if slot_blocked or any_stop or any_reroute or any_drain:
                phase = "drain_old_gpu" if any_drain else ("reroute_old_queue" if any_reroute else ("stop_accepting_old_gpu" if any_stop else "takeover_ready"))
                blocked_actions.append({"type": "defer_remove_gpu", "gpu_id": gid, "physical_gpu_id": pid, "phase": phase})
                plan_items.append(_v2_item(item_id=f"REMOVE_gpu{gid}", item_type="remove_gpu", current_phase=phase, status="blocked" if slot_blocked else "in_progress", blocked_by="waiting_for_slot_drain" if slot_blocked else None, gpu_id=gid, physical_gpu_id=pid))
            else:
                cleanup_actions.extend([{"type": "clear_gpu", "gpu_id": gid, "physical_gpu_id": pid}, {"type": "clear_template", "gpu_id": gid, "physical_gpu_id": pid}])
                free_pool.append(pid)
                plan_items.append(_v2_item(item_id=f"REMOVE_gpu{gid}", item_type="remove_gpu", current_phase="clear_old_gpu", status="ready", gpu_id=gid, physical_gpu_id=pid))
            continue

        if change == "reconfiguration":
            old_pid = _get_physical_id(source_state, gid)
            in_place_ok = True
            for inst in _v2_nonfree_instances(s):
                slot = (inst.start, inst.end, inst.profile)
                cands = _v2_takeover_candidates(source_state, target_state, inst.workload, exclude_gpu_id=gid, exclude_slot=slot)
                if not any(c.get("kind") == "unchanged" for c in cands):
                    in_place_ok = False
                    break
            if in_place_ok:
                new_pid = old_pid
                prepared = True
                target_pid_map[gid] = old_pid
                coarse_actions.append({"type": "reconfiguration", "gpu_id": gid, "source_physical_gpu_id": old_pid, "new_physical_gpu_id": old_pid, "src_template": s.template_str(), "tgt_template": t.template_str(), "mode": "in_place_old_first"})
            else:
                rec = _v2_reconfig_map(source_state).get(str(gid))
                if rec is None:
                    new_pid = _alloc_from_free_pool(free_pool)
                    prepared = False
                else:
                    new_pid = rec.get("physical_gpu_id")
                    prepared = True
                target_pid_map[gid] = new_pid
                coarse_actions.append({"type": "reconfiguration", "gpu_id": gid, "source_physical_gpu_id": old_pid, "new_physical_gpu_id": new_pid, "src_template": s.template_str(), "tgt_template": t.template_str(), "alloc_policy": "free_pool_lifo", "mode": "target_first"})
                if not prepared:
                    capacity_actions.extend([{"type": "allocate_gpu", "physical_gpu_id": new_pid, "policy": "free_pool_lifo"}, {"type": "configure_full_template", "physical_gpu_id": new_pid, "template": t.template_str()}, {"type": "place_target_layout", "gpu_id": gid, "physical_gpu_id": new_pid}, {"type": "mark_reconfig_target_prepared", "gpu_id": gid, "physical_gpu_id": new_pid, "template": t.template_str()}])
            slot_blocked = False
            any_stop = False
            any_reroute = False
            any_drain = False
            for inst in _v2_nonfree_instances(s):
                slot = (inst.start, inst.end, inst.profile)
                acts, phase, status, blocked_by = add_slot_barrier(gid, old_pid, slot, inst.workload, "reconfiguration_slot", f"RECONF_gpu{gid}_{slot[0]}_{slot[1]}_{slot[2]}", safe_after_drain=True)
                phase_actions.extend(acts)
                any_stop = any_stop or any(a["type"] == "stop_accepting_new" for a in acts)
                any_reroute = any_reroute or any(a["type"] == "reroute_queued_tasks" for a in acts)
                any_drain = any_drain or (phase == "drain_old")
                slot_blocked = slot_blocked or (status == "blocked")
            if slot_blocked or any_stop or any_reroute or any_drain:
                if in_place_ok:
                    phase = "drain_old_side" if any_drain else ("reroute_old_queue" if any_reroute else ("shift_routing" if any_stop else "in_place_reconfigure"))
                else:
                    phase = "drain_old_side" if any_drain else ("reroute_old_queue" if any_reroute else ("shift_routing" if any_stop else ("prepare_target_side" if not prepared else "target_side_prepared")))
                blocked_actions.append({"type": "defer_remove_gpu", "gpu_id": gid, "physical_gpu_id": old_pid, "phase": phase})
                plan_items.append(_v2_item(item_id=f"RECONF_gpu{gid}", item_type="reconfiguration", current_phase=phase, status="blocked" if slot_blocked else "in_progress", blocked_by="waiting_for_old_side_drain" if slot_blocked else None, gpu_id=gid, physical_gpu_id=old_pid, target_physical_gpu_id=new_pid))
            else:
                cleanup_actions.extend([{"type": "clear_gpu", "gpu_id": gid, "physical_gpu_id": old_pid}, {"type": "clear_template", "gpu_id": gid, "physical_gpu_id": old_pid}])
                if in_place_ok:
                    finalize_actions.extend([{"type": "configure_full_template", "physical_gpu_id": old_pid, "template": t.template_str()}, {"type": "place_target_layout", "gpu_id": gid, "physical_gpu_id": old_pid}, {"type": "bind_target_gpu", "gpu_id": gid, "physical_gpu_id": old_pid}])
                    plan_items.append(_v2_item(item_id=f"RECONF_gpu{gid}", item_type="reconfiguration", current_phase="in_place_reconfigure", status="ready", gpu_id=gid, physical_gpu_id=old_pid, target_physical_gpu_id=old_pid))
                else:
                    finalize_actions.append({"type": "bind_target_gpu", "gpu_id": gid, "physical_gpu_id": new_pid})
                    free_pool.append(old_pid)
                    plan_items.append(_v2_item(item_id=f"RECONF_gpu{gid}", item_type="reconfiguration", current_phase="finalize_target_side", status="ready", gpu_id=gid, physical_gpu_id=old_pid, target_physical_gpu_id=new_pid))
            continue

        if change == "instance_diff":
            pid = _get_physical_id(source_state, gid)
            target_pid_map[gid] = pid
            inst_actions = diff_instances_within_same_template(s, t)
            coarse_actions.append({"type": "instance_diff", "gpu_id": gid, "physical_gpu_id": pid, "template": t.template_str(), "instance_changes": [a["type"] for a in inst_actions]})
            for ia in inst_actions:
                if ia["type"] == "keep":
                    continue
                if ia["type"] == "batch_change":
                    capacity_actions.append({"type": "update_batch", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "old_batch": ia["src"].batch, "new_batch": ia["tgt"].batch, "workload": ia["src"].workload})
                    continue
                if ia["type"] == "place_instance":
                    capacity_actions.append({"type": "place_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["tgt"].workload, "batch": ia["tgt"].batch})
                    plan_items.append(_v2_item(item_id=f"PLACE_gpu{gid}_{ia['slot'][0]}_{ia['slot'][1]}_{ia['slot'][2]}", item_type="place_instance", current_phase="prepare_target_side", status="ready", gpu_id=gid, physical_gpu_id=pid, slot=ia["slot"], workload=ia["tgt"].workload))
                    continue
                if ia["type"] == "safe_remove_instance":
                    safe = _safe_after_removing_instance(source_state, ia["src"], required)
                    acts, phase, status, blocked_by = add_slot_barrier(gid, pid, ia["slot"], ia["src"].workload, "remove_instance", f"RM_gpu{gid}_{ia['slot'][0]}_{ia['slot'][1]}_{ia['slot'][2]}", safe_after_drain=safe)
                    phase_actions.extend(acts)
                    if status == "ready":
                        cleanup_actions.append({"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": True, "drained": True})
                        plan_items[-1]["current_phase"] = "remove_old_instance"
                    else:
                        blocked_actions.append({"type": "defer_remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "phase": phase})
                    continue
                if ia["type"] == "workload_change":
                    safe = _safe_after_removing_instance(source_state, ia["src"], required)
                    acts, phase, status, blocked_by = add_slot_barrier(gid, pid, ia["slot"], ia["src"].workload, "workload_change", f"WC_gpu{gid}_{ia['slot'][0]}_{ia['slot'][1]}_{ia['slot'][2]}", safe_after_drain=safe)
                    phase_actions.extend(acts)
                    if status == "ready":
                        cleanup_actions.append({"type": "remove_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["src"].workload, "safe_now": True, "drained": True})
                        capacity_actions.append({"type": "place_instance", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "workload": ia["tgt"].workload, "batch": ia["tgt"].batch, "after_drain": True})
                        plan_items[-1]["current_phase"] = "replace_slot"
                    else:
                        blocked_actions.append({"type": "defer_workload_change", "gpu_id": gid, "physical_gpu_id": pid, "slot": ia["slot"], "old_workload": ia["src"].workload, "new_workload": ia["tgt"].workload, "phase": phase})
                    continue

    fine_actions = capacity_actions + phase_actions + cleanup_actions + finalize_actions + blocked_actions
    planned_state = _deepcopy_state(target_state)
    _ensure_state_metadata(planned_state)
    planned_state.metadata["physical_id_map"] = {int(gid): pid for gid, pid in target_pid_map.items()}
    planned_state.metadata["next_physical_idx"] = source_state.metadata.get("next_physical_idx", 0)
    executed_state = _simulate_fine_actions_v2(source_state=source_state, target_state=planned_state, fine_actions=fine_actions, next_physical_idx=source_state.metadata.get("next_physical_idx", 0))
    executed_actions, blocked = _v2_split_actions(fine_actions)
    return {"stage_name": stage_name, "required": required, "coarse_actions": coarse_actions, "fine_actions": fine_actions, "executed_actions": executed_actions, "blocked_actions": blocked, "planned_state": planned_state, "executed_state": executed_state, "workload_classes": workload_classes, "free_pool_after_plan": list(free_pool), "plan_items": plan_items}

def run_v2_stage_iterative(source_state: "ClusterState", target_state: "ClusterState", src_arrival, tgt_arrival, stage_name="stage_v2", max_iters: int = 12) -> Dict[str, Any]:
    current_state = _v2_prepare_source_runtime(source_state, target_state)
    _ensure_state_metadata(current_state)
    initial_runtime_state = _deepcopy_state(current_state)
    iterations = []
    all_executed_actions = []
    final_blocked_actions = []
    final_plan = None
    workload_classes = classify_workloads_by_arrival(src_arrival, tgt_arrival)
    reached_target = _matches_target_state(current_state, target_state) and len(_v2_drain_map(current_state)) == 0
    t0 = time.perf_counter()
    peak_active_gpu = len(_v2_active_pid_set(current_state))
    for iter_idx in range(1, max_iters + 1):
        if reached_target:
            break
        _v2_advance_drain(current_state)
        iter_plan = plan_naive_action_v2(source_state=current_state, target_state=target_state, src_arrival=src_arrival, tgt_arrival=tgt_arrival, stage_name=f"{stage_name}_iter{iter_idx}")
        final_plan = iter_plan
        executed_actions = iter_plan["executed_actions"]
        blocked_actions = iter_plan["blocked_actions"]
        next_state = iter_plan["executed_state"]
        made_progress = (_v2_progress_signature(next_state) != _v2_progress_signature(current_state))
        reached_target = _matches_target_state(next_state, target_state) and len(_v2_drain_map(next_state)) == 0
        iter_peak = _v2_peak_from_actions(current_state, executed_actions)
        peak_active_gpu = max(peak_active_gpu, iter_peak, len(_v2_active_pid_set(next_state)))
        iterations.append({"iteration": iter_idx, "plan": iter_plan, "executed_actions": executed_actions, "blocked_actions": blocked_actions, "made_progress": made_progress, "reached_target": reached_target, "state_before": _deepcopy_state(current_state), "state_after": _deepcopy_state(next_state), "iter_peak_active_gpu": iter_peak, "active_gpu_after": len(_v2_active_pid_set(next_state))})
        all_executed_actions.extend(copy.deepcopy(executed_actions))
        final_blocked_actions = copy.deepcopy(blocked_actions)
        current_state = next_state
        if reached_target or not made_progress:
            break
    elapsed_sec = time.perf_counter() - t0
    return {"stage_name": stage_name, "iterations": iterations, "iteration_count": len(iterations), "reached_target": reached_target, "elapsed_sec": elapsed_sec, "executed_actions": all_executed_actions, "blocked_actions": final_blocked_actions, "final_plan": final_plan, "executed_state": current_state, "target_state": _deepcopy_state(target_state), "workload_classes": workload_classes, "initial_runtime_state": initial_runtime_state, "peak_active_gpu": peak_active_gpu, "source_active_gpu": len(_v2_active_pid_set(initial_runtime_state)), "final_active_gpu": len(_v2_active_pid_set(current_state))}

def _v2_changed_slot_summary_lines(state: "ClusterState") -> List[str]:
    rows = _v2_runtime_rows(state)
    out = []
    for r in rows:
        slot = _v2_slot_label(r['gpu_id'], r['slot'], r['physical_gpu_id'])
        parts = [f"{slot}", str(r['workload'])]
        if int(r.get('queued', 0)) > 0:
            parts.append(f"q={int(r['queued'])}")
        if int(r.get('inflight', 0)) > 0:
            parts.append(f"inflight={int(r['inflight'])}")
        if not bool(r.get('accepting_new', True)):
            parts.append('accepting_new=False')
        if r.get('rerouted_to'):
            parts.append(f"to={r['rerouted_to']}")
        out.append(' | '.join(parts))
    return out

def print_v2_stage_result(stage_res: Dict[str, Any], title="[V2 STAGE RESULT]"):
    print("=" * 90)
    print(title)
    print("=" * 90)
    print(f"Stage: {stage_res['stage_name']}")
    print(f"Iterations: {stage_res['iteration_count']} | Reached target: {stage_res['reached_target']} | Elapsed (s): {stage_res['elapsed_sec']:.6f}")
    print(f"Active GPU: source={stage_res['source_active_gpu']} final={stage_res['final_active_gpu']} peak={stage_res['peak_active_gpu']}")
    lines = _v2_changed_slot_summary_lines(stage_res["initial_runtime_state"])
    if lines:
        print("Changed slots summary:")
        for line in lines:
            print(f"  - {line}")
    prev_items = {}
    print(f"Executed runtime actions(total)={len(stage_res['executed_actions'])} | Final blocked={len(stage_res['blocked_actions'])}")
    for iter_res in stage_res['iterations']:
        plan = iter_res['plan']
        items = {it['id']: it for it in plan.get('plan_items', [])}
        drain_items = [it for it in plan.get('plan_items', []) if _v2_is_drain_barrier_item(it)]
        blocked_items = [it for it in plan.get('plan_items', []) if it.get('status') == 'blocked' and not _v2_is_drain_barrier_item(it)]
        print("-" * 90)
        print(f"Iter {iter_res['iteration']}: progress={iter_res['made_progress']} reached_target={iter_res['reached_target']} active_after={iter_res['active_gpu_after']} iter_peak={iter_res['iter_peak_active_gpu']} drain_barriers={len(drain_items)} other_blocked={len(blocked_items)}")
        transitions = []
        for item_id, it in items.items():
            prev = prev_items.get(item_id)
            if prev is None:
                transitions.append(f"{item_id}: init -> {it['current_phase']} ({it['status']})")
            elif prev.get('current_phase') != it.get('current_phase') or prev.get('status') != it.get('status'):
                transitions.append(f"{item_id}: {prev.get('current_phase')} ({prev.get('status')}) -> {it.get('current_phase')} ({it.get('status')})")
        if transitions:
            print("Phase transitions:")
            for line in transitions:
                print(f"  - {line}")
        if drain_items:
            print("Drain barriers:")
            for it in drain_items:
                msg = f"  - {it['id']} | phase={it['current_phase']} | status={it['status']}"
                if it.get('workload') is not None:
                    msg += f" | workload={it['workload']}"
                if it.get('takeover') is not None:
                    msg += f" | takeover={it['takeover']}"
                if it.get('drain_remaining') is not None:
                    msg += f" | drain_remaining={it['drain_remaining']}"
                if it.get('blocked_by'):
                    msg += f" | blocked_by={it['blocked_by']}"
                print(msg)
        if blocked_items:
            print("Other blocked items:")
            for it in blocked_items:
                print(f"  - {it['id']} | phase={it['current_phase']} | blocked_by={it.get('blocked_by')}")
        if iter_res['executed_actions']:
            print("Executed runtime actions:")
            for i, a in enumerate(iter_res['executed_actions'], 1):
                print(f"  X{i:02d}. {_v2_format_action_short(a)}")
        prev_items = {k: copy.deepcopy(v) for k, v in items.items()}


### Naive v2 experiment — stage 0

从空状态到 `target0` 跑 v2 planner，并打印 workload 分类。

In [186]:
# ============================================================
# [Action Code Block V2-2] Stage 0: initial -> target0 -> v2 planner -> executed0_v2
# ============================================================
source0_v2 = ClusterState(gpus=[], metadata={})
_ensure_state_metadata(source0_v2)
print_stage_current_and_target(source0_v2, target0, title_prefix="[V2 STAGE 0]")
stage0_v2_res = run_v2_stage_iterative(source_state=source0_v2, target_state=target0, src_arrival=np.zeros_like(arrival_rate, dtype=float), tgt_arrival=arrival_rate, stage_name="stage0_v2")
plan0_v2 = stage0_v2_res["final_plan"]
plan0_v2_elapsed_sec = stage0_v2_res["elapsed_sec"]
executed0_v2 = stage0_v2_res["executed_state"]
print_v2_stage_result(stage0_v2_res, title="[V2 STAGE 0] source0 -> target0 -> iterative v2 planner")
print(f"[V2 STAGE 0] iterative v2 planner elapsed: {plan0_v2_elapsed_sec:.6f} sec")
source1_v2 = _canonicalize_state_for_next_round(executed0_v2)
print_canonicalize_result(executed0_v2, source1_v2, title="[V2 STAGE 0] canonicalize for next MILP")


GPU count fixed from MILP    : 6
[V2 STAGE 0] CURRENT STATE
(empty)
current templates: []
[V2 STAGE 0] TARGET STATE
GPU 0: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 2: [0,4) 4g: vit_base / B64 / mu=1718.9838 | [4,7) 3g: vit_base / B64 / mu=1448.9833
GPU 3: [0,2) 2g: vgg16 / B16 / mu=483.8763 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: resnet50 / B16 / mu=351.0342
GPU 4: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu=1.1311
GPU 5: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6)

### Naive v2 experiment — stage 1

对 `source1_v2` 重新生成 `target1_seq_v2`，再跑 stage 内可迭代的 v2 planner。

In [187]:
# ============================================================
# [Action Code Block V2-3] Stage 1: source1_v2 -> target1_seq_v2 -> iterative v2 planner
# ============================================================
target1_seq_v2 = build_target_state_from_milp(milp_res=milp_res_up_manual, prev_state=source1_v2, abstract_template_topk=64, physical_layout_topk=32, per_gpu_layout_topk=4, verbose=False)
print_stage_current_and_target(source1_v2, target1_seq_v2, title_prefix="[V2 STAGE 1]")
stage1_v2_res = run_v2_stage_iterative(source_state=source1_v2, target_state=target1_seq_v2, src_arrival=arrival_rate, tgt_arrival=arrival_rate_up_manual, stage_name="stage1_v2")
plan1_v2 = stage1_v2_res["final_plan"]
plan1_v2_elapsed_sec = stage1_v2_res["elapsed_sec"]
executed1_v2 = stage1_v2_res["executed_state"]
print_v2_stage_result(stage1_v2_res, title="[V2 STAGE 1] source1_v2 -> target1_seq_v2 -> iterative v2 planner")
print(f"[V2 STAGE 1] iterative v2 planner elapsed: {plan1_v2_elapsed_sec:.6f} sec")
source2_v2 = _canonicalize_state_for_next_round(executed1_v2)
print_canonicalize_result(executed1_v2, source2_v2, title="[V2 STAGE 1] canonicalize for next MILP")


GPU count fixed from MILP    : 9
[V2 STAGE 1] CURRENT STATE
GPU 0 [A]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1 [B]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 2 [C]: [0,4) 4g: vit_base / B64 / mu=1718.9838 | [4,7) 3g: vit_base / B64 / mu=1448.9833
GPU 3 [D]: [0,2) 2g: vgg16 / B16 / mu=483.8763 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: resnet50 / B16 / mu=351.0342
GPU 4 [E]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu=1.1311
GPU 5 [F]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,

### Naive v2 experiment — stage 2

对 `source2_v2` 重新生成 `target2_seq_v2`，再跑 stage 内可迭代的 v2 planner。

In [188]:
# ============================================================
# [Action Code Block V2-4] Stage 2: source2_v2 -> target2_seq_v2 -> iterative v2 planner
# ============================================================
target2_seq_v2 = build_target_state_from_milp(milp_res=milp_res_t2_manual, prev_state=source2_v2, abstract_template_topk=64, physical_layout_topk=32, per_gpu_layout_topk=4, verbose=False)
print_stage_current_and_target(source2_v2, target2_seq_v2, title_prefix="[V2 STAGE 2]")
stage2_v2_res = run_v2_stage_iterative(source_state=source2_v2, target_state=target2_seq_v2, src_arrival=arrival_rate_up_manual, tgt_arrival=arrival_rate_t2_manual, stage_name="stage2_v2")
plan2_v2 = stage2_v2_res["final_plan"]
plan2_v2_elapsed_sec = stage2_v2_res["elapsed_sec"]
executed2_v2 = stage2_v2_res["executed_state"]
print_v2_stage_result(stage2_v2_res, title="[V2 STAGE 2] source2_v2 -> target2_seq_v2 -> iterative v2 planner")
print(f"[V2 STAGE 2] iterative v2 planner elapsed: {plan2_v2_elapsed_sec:.6f} sec")
source3_v2 = _canonicalize_state_for_next_round(executed2_v2)
print_canonicalize_result(executed2_v2, source3_v2, title="[V2 STAGE 2] canonicalize for next MILP")


GPU count fixed from MILP    : 6
[V2 STAGE 2] CURRENT STATE
GPU 0 [A]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1 [B]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 2 [C]: [0,4) 4g: vit_base / B64 / mu=1718.9838 | [4,7) 3g: vit_base / B64 / mu=1448.9833
GPU 3 [D]: [0,2) 2g: free | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: resnet50 / B16 / mu=351.0342
GPU 4 [E]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu=1.1311
GPU 5 [F]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu

### Naive v2 experiment — stage 3

对 `source3_v2` 重新生成 `target3_seq_v2`，再跑 stage 内可迭代的 v2 planner。

In [189]:
# ============================================================
# [Action Code Block V2-5] Stage 3: source3_v2 -> target3_seq_v2 -> iterative v2 planner
# ============================================================
target3_seq_v2 = build_target_state_from_milp(milp_res=milp_res_up_3, prev_state=source3_v2, abstract_template_topk=64, physical_layout_topk=32, per_gpu_layout_topk=4, verbose=False)
print_stage_current_and_target(source3_v2, target3_seq_v2, title_prefix="[V2 STAGE 3]")
stage3_v2_res = run_v2_stage_iterative(source_state=source3_v2, target_state=target3_seq_v2, src_arrival=arrival_rate_t2_manual, tgt_arrival=arrival_rate_up_3, stage_name="stage3_v2")
plan3_v2 = stage3_v2_res["final_plan"]
plan3_v2_elapsed_sec = stage3_v2_res["elapsed_sec"]
executed3_v2 = stage3_v2_res["executed_state"]
print_v2_stage_result(stage3_v2_res, title="[V2 STAGE 3] source3_v2 -> target3_seq_v2 -> iterative v2 planner")
print(f"[V2 STAGE 3] iterative v2 planner elapsed: {plan3_v2_elapsed_sec:.6f} sec")


GPU count fixed from MILP    : 8
[V2 STAGE 3] CURRENT STATE
GPU 0 [A]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1 [B]: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 2 [D]: [0,2) 2g: free | [2,3) 1g: vgg16 / B8 / mu=217.9872 | [3,4) 1g: free | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: resnet50 / B16 / mu=351.0342
GPU 3 [E]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu=1.1311
GPU 4 [F]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: gpt2 / B1 / mu=1.1311 | [3,4) 1g: gpt2 / B1 / mu=1.1311 | [4,5) 1g: gpt2 / B1 / mu=1.1311 | [5,6) 1g: gpt2 / B1 / mu=1.1311 | [6,7) 1g: gpt2 / B1 / mu=1.1311
GPU 5 [I]: [0,4) 4g: vit_base / B64 / mu=1448.9833 | [4,7) 3g: vit_base / B64 / mu=1448.9833
curren

### Naive v0 vs v1 vs v2 summary

下面汇总三版 planner 的模板结果和各阶段用时。

In [190]:
# ============================================================
# [Action Code Block V2-6] Summary / comparison
# ============================================================
v2_time_df = pd.DataFrame([
    {"planner": "naive_v2", "stage": 0, "elapsed_sec": plan0_v2_elapsed_sec},
    {"planner": "naive_v2", "stage": 1, "elapsed_sec": plan1_v2_elapsed_sec},
    {"planner": "naive_v2", "stage": 2, "elapsed_sec": plan2_v2_elapsed_sec},
    {"planner": "naive_v2", "stage": 3, "elapsed_sec": plan3_v2_elapsed_sec},
])
comparison_time_df_v2 = pd.concat([v0_time_df, v1_time_df, v2_time_df], ignore_index=True)
print("[planner timing comparison: v0 vs v1 vs v2]")
display(comparison_time_df_v2)
print("\n[v2 templates]")
print("executed0_v2:", [g.template_str() for g in executed0_v2.real_gpus()])
print("executed1_v2:", [g.template_str() for g in executed1_v2.real_gpus()])
print("executed2_v2:", [g.template_str() for g in executed2_v2.real_gpus()])
print("executed3_v2:", [g.template_str() for g in executed3_v2.real_gpus()])
print("\n[v2 physical bindings]")
for label, st in [("v2_stage0", executed0_v2), ("v2_stage1", executed1_v2), ("v2_stage2", executed2_v2), ("v2_stage3", executed3_v2)]:
    print(label, dict(sorted(st.metadata.get("physical_id_map", {}).items())))


[planner timing comparison: v0 vs v1 vs v2]


,planner,stage,elapsed_sec
0,naive_v0,0,0.007044
1,naive_v0,1,0.001676
2,naive_v0,2,0.001302
3,naive_v0,3,0.001296
4,naive_v1,0,0.000796
5,naive_v1,1,0.003034
6,naive_v1,2,0.002861
7,naive_v1,3,0.001213
8,naive_v2,0,0.001424
9,naive_v2,1,0.006634



[v2 templates]
executed0_v2: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
executed1_v2: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3', '1+1+1+1+3', '4+3']
executed2_v2: ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3']
executed3_v2: ['4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '4+3', '4+3', '1+1+1+1+3']

[v2 physical bindings]
v2_stage0 {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F'}
v2_stage1 {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 8: 'I'}
v2_stage2 {0: 'A', 1: 'B', 3: 'D', 4: 'E', 5: 'F', 8: 'I'}
v2_stage3 {0: 'A', 1: 'B', 2: 'D', 3: 'E', 4: 'F', 5: 'I', 6: 'C', 7: 'G'}


### V2 drain minimal demo

下面补一个 `v2` 最小实验，专门演示 `stop_accepting_new -> reroute_queued_tasks -> drain_old -> replace_slot` 这条 phase 链：

- `GPU 0` 上一个 `gpt2` 需要改成 `resnet50`
- `GPU 1` 上有一个保持不变的 `gpt2` instance，可承接 reroute 过来的队列/新任务
- 因此 `GPU 0` 不需要 temporary bridge，而是直接依赖 `unchanged capacity + drain`

因此预期行为是：

1. 第 1 轮先停止 `GPU 0` 接新任务，并把 queued tasks reroute 到 `GPU 1`
2. 随后 `GPU 0` 进入 drain，等待 in-flight task 跑完
3. drain 完成后，再执行 `remove old + place new`


In [191]:
# ============================================================
# [Action Code Block V2-7] Minimal drain-trigger demo
# ============================================================
demo2_mu_gpt2_1g_b1 = _demo_lookup_mu("gpt2", "1g", 1)
demo2_mu_resnet50_1g_b16 = _demo_lookup_mu("resnet50", "1g", 16)

demo2_source = ClusterState(gpus=[
    _demo_make_1g_gpu(0, {
        0: ("gpt2", 1, demo2_mu_gpt2_1g_b1),
    }),
    _demo_make_1g_gpu(1, {
        0: ("gpt2", 1, demo2_mu_gpt2_1g_b1),
    }),
], metadata={})
_ensure_state_metadata(demo2_source)
_set_physical_id(demo2_source, 0, "A")
_set_physical_id(demo2_source, 1, "B")

demo2_target = ClusterState(gpus=[
    _demo_make_1g_gpu(0, {
        0: ("resnet50", 16, demo2_mu_resnet50_1g_b16),
    }),
    _demo_make_1g_gpu(1, {
        0: ("gpt2", 1, demo2_mu_gpt2_1g_b1),
    }),
], metadata={})
_ensure_state_metadata(demo2_target)

demo2_src_arr = np.zeros_like(arrival_rate, dtype=float)
demo2_tgt_arr = np.zeros_like(arrival_rate, dtype=float)
demo2_name_to_idx = {name: i for i, name in enumerate(WORKLOAD_NAMES)}
demo2_src_arr[demo2_name_to_idx["gpt2"]] = 2.0
demo2_tgt_arr[demo2_name_to_idx["gpt2"]] = 1.0
demo2_tgt_arr[demo2_name_to_idx["resnet50"]] = 1.0

print_stage_current_and_target(demo2_source, demo2_target, title_prefix="[V2 DRAIN DEMO]")

drain_demo_res = run_v2_stage_iterative(
    source_state=demo2_source,
    target_state=demo2_target,
    src_arrival=demo2_src_arr,
    tgt_arrival=demo2_tgt_arr,
    stage_name="drain_demo_v2",
    max_iters=4,
)

print_v2_stage_result(drain_demo_res, title="[V2 DRAIN DEMO] iterative planner")

drain_marks = [a for a in drain_demo_res["executed_actions"] if a.get("type") == "mark_draining_instance"]
print("-" * 90)
print(f"[V2 DRAIN DEMO] mark_draining_instance count: {len(drain_marks)}")
for i, a in enumerate(drain_marks, 1):
    print(f"  D{i:02d}. {a}")


GPU count fixed from MILP    : 2
[V2 DRAIN DEMO] CURRENT STATE
GPU 0 [A]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: free | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
GPU 1 [B]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: free | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
current templates: ['1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
[V2 DRAIN DEMO] TARGET STATE
GPU 0: [0,1) 1g: resnet50 / B16 / mu=351.0342 | [1,2) 1g: free | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
GPU 1: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: free | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
target templates: ['1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
[V2 DRAIN DEMO] workload classes
shared   : ['gpt2']
new      : ['resnet50']
retiring : []
inactive : ['llama', 'vgg16', 'vit_base']
[V2 DRAIN DEMO] iterative planner
Stage: drain_demo_v2
Iterations: 3 | Reached target: Tr

### V2 reconfiguration minimal demo

下面补一个 `v2` 的 `reconfiguration` 最小实验：

- `GPU 0` 在 source 中是 `1+1+1+1+1+1+1`
- target 中同一个逻辑 `GPU 0` 变成 `4+3`
- 因为 template 改变，这会被 `v2` 识别成 `reconfiguration`
- `GPU 1` 保持不变，用来承接 `GPU 0` 旧 workload 的 reroute / drain

因此预期行为是：

1. 先为 `GPU 0` 提前创建 target side（target-backed capacity）
2. 再让 old side 停接新任务并进入 drain
3. drain 完成后清理 old side


In [192]:
# ============================================================
# [Action Code Block V2-8] Minimal reconfiguration demo
# ============================================================
def _demo_make_4g3g_gpu(gpu_id: int, payload4=None, payload3=None) -> GPUState:
    insts = []
    if payload4 is None:
        insts.append(MigInstance(start=0, end=4, profile="4g"))
    else:
        w, b, mu = payload4
        insts.append(MigInstance(start=0, end=4, profile="4g", workload=w, batch=int(b), mu=float(mu)))
    if payload3 is None:
        insts.append(MigInstance(start=4, end=7, profile="3g"))
    else:
        w, b, mu = payload3
        insts.append(MigInstance(start=4, end=7, profile="3g", workload=w, batch=int(b), mu=float(mu)))
    g = GPUState(gpu_id=gpu_id, source="real", instances=insts)
    g.sort_instances()
    return g

demo3_mu_gpt2_1g_b1 = _demo_lookup_mu("gpt2", "1g", 1)
demo3_mu_vgg16_1g_b8 = _demo_lookup_mu("vgg16", "1g", 8)
demo3_mu_vit_4g_b64 = _demo_lookup_mu("vit_base", "4g", 64)
demo3_mu_llama_3g_b1 = _demo_lookup_mu("llama", "3g", 1)

demo3_source = ClusterState(gpus=[
    _demo_make_1g_gpu(0, {
        0: ("gpt2", 1, demo3_mu_gpt2_1g_b1),
        1: ("gpt2", 1, demo3_mu_gpt2_1g_b1),
        2: ("vgg16", 8, demo3_mu_vgg16_1g_b8),
        3: ("vgg16", 8, demo3_mu_vgg16_1g_b8),
    }),
    _demo_make_1g_gpu(1, {
        0: ("gpt2", 1, demo3_mu_gpt2_1g_b1),
        1: ("gpt2", 1, demo3_mu_gpt2_1g_b1),
        2: ("vgg16", 8, demo3_mu_vgg16_1g_b8),
        3: ("vgg16", 8, demo3_mu_vgg16_1g_b8),
    }),
], metadata={})
_ensure_state_metadata(demo3_source)
_set_physical_id(demo3_source, 0, "A")
_set_physical_id(demo3_source, 1, "B")

demo3_target = ClusterState(gpus=[
    _demo_make_4g3g_gpu(0,
        payload4=("vit_base", 64, demo3_mu_vit_4g_b64),
        payload3=("llama", 1, demo3_mu_llama_3g_b1),
    ),
    _demo_make_1g_gpu(1, {
        0: ("gpt2", 1, demo3_mu_gpt2_1g_b1),
        1: ("gpt2", 1, demo3_mu_gpt2_1g_b1),
        2: ("vgg16", 8, demo3_mu_vgg16_1g_b8),
        3: ("vgg16", 8, demo3_mu_vgg16_1g_b8),
    }),
], metadata={})
_ensure_state_metadata(demo3_target)

demo3_src_arr = np.zeros_like(arrival_rate, dtype=float)
demo3_tgt_arr = np.zeros_like(arrival_rate, dtype=float)
demo3_name_to_idx = {name: i for i, name in enumerate(WORKLOAD_NAMES)}
demo3_src_arr[demo3_name_to_idx["gpt2"]] = 4.0
demo3_src_arr[demo3_name_to_idx["vgg16"]] = 4.0
demo3_tgt_arr[demo3_name_to_idx["gpt2"]] = 2.0
demo3_tgt_arr[demo3_name_to_idx["vgg16"]] = 2.0
demo3_tgt_arr[demo3_name_to_idx["vit_base"]] = 1.0
demo3_tgt_arr[demo3_name_to_idx["llama"]] = 1.0

print_stage_current_and_target(demo3_source, demo3_target, title_prefix="[V2 RECONF DEMO]")

reconf_demo_res = run_v2_stage_iterative(
    source_state=demo3_source,
    target_state=demo3_target,
    src_arrival=demo3_src_arr,
    tgt_arrival=demo3_tgt_arr,
    stage_name="reconf_demo_v2",
    max_iters=5,
)

print_v2_stage_result(reconf_demo_res, title="[V2 RECONF DEMO] iterative planner")


GPU count fixed from MILP    : 2
[V2 RECONF DEMO] CURRENT STATE
GPU 0 [A]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: vgg16 / B8 / mu=217.9872 | [3,4) 1g: vgg16 / B8 / mu=217.9872 | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
GPU 1 [B]: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: vgg16 / B8 / mu=217.9872 | [3,4) 1g: vgg16 / B8 / mu=217.9872 | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
current templates: ['1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
[V2 RECONF DEMO] TARGET STATE
GPU 0: [0,4) 4g: vit_base / B64 / mu=1718.9838 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: vgg16 / B8 / mu=217.9872 | [3,4) 1g: vgg16 / B8 / mu=217.9872 | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
target templates: ['4+3', '1+1+1+1+1+1+1']
[V2 RECONF DEMO] workload classes
shared   : ['gpt2', 'vgg16']
new      : ['llama', 'vit_base']
retiring : []
inactive : ['re

### V2 phase rules

`v2` 现在把高层动作看成可交错推进的 phase 链，而不是原子动作。

| High-level action | Main phases | Completion condition |
|---|---|---|
| `create_gpu` | `allocate -> configure -> place -> bind-ready` | target side ready and can serve |
| `workload_change` | `find/prepare takeover -> stop_accepting_new -> reroute_queued_tasks -> drain_old -> replace_slot` | old workload removed and new workload ready |
| `reconfiguration` | `prepare_target_side -> shift_routing -> drain_old_side -> clear_old_side -> finalize_target_side` | old side cleared and target side bound/ready |
| `remove_gpu` | `find/prepare takeover -> stop_route_old_gpu -> drain_old_gpu -> clear_old_gpu` | old GPU fully drained and cleared |

`v2` 的调度优先级是：

1. 先暴露 target-backed capacity / unchanged capacity
2. 再停止 old side 接新任务，并把 queued tasks reroute 掉
3. 等 old side drain 完成
4. 最后执行 destructive actions
5. 只有以上都不够时才考虑 temporary bridge


### V2 stage-order notes (0 -> t3)

- `stage0`: pure `create_gpu`; build target side directly.
- `stage1`: `create_gpu` first, then `stop_accepting_new -> reroute -> drain -> remove_old` on the old instance.
- `stage2`: stop routing on all old-side items first, then wait on drain barriers, then clear/remove old side, then fill remaining target placement.
- `stage3`: mostly target-side placement/create, with almost no visible drain chain.


### V2 mixed demo (reconfiguration + create_gpu + workload_change)

这个例子同时包含三类高层动作：

- `GPU 0`: `reconfiguration` (`1g... -> 4+3`)
- `GPU 1`: `workload_change` (`gpt2 -> resnet50`)
- `GPU 3`: `create_gpu`

`GPU 2` 保持不变，专门承担 `unchanged takeover` 的角色。


In [193]:
# ============================================================
# [Action Code Block V2-9] Mixed demo: reconfiguration + create_gpu + workload_change
# ============================================================
demo4_mu_gpt2_1g_b1 = _demo_lookup_mu("gpt2", "1g", 1)
demo4_mu_vgg16_1g_b8 = _demo_lookup_mu("vgg16", "1g", 8)
demo4_mu_resnet_1g_b8 = _demo_lookup_mu("resnet50", "1g", 8)
demo4_mu_vit_4g_b64 = _demo_lookup_mu("vit_base", "4g", 64)
demo4_mu_llama_3g_b1 = _demo_lookup_mu("llama", "3g", 1)

demo4_source = ClusterState(gpus=[
    _demo_make_1g_gpu(0, {
        0: ("gpt2", 1, demo4_mu_gpt2_1g_b1),
        1: ("gpt2", 1, demo4_mu_gpt2_1g_b1),
        2: ("vgg16", 8, demo4_mu_vgg16_1g_b8),
        3: ("vgg16", 8, demo4_mu_vgg16_1g_b8),
    }),
    _demo_make_1g_gpu(1, {
        0: ("gpt2", 1, demo4_mu_gpt2_1g_b1),
    }),
    _demo_make_1g_gpu(2, {
        0: ("gpt2", 1, demo4_mu_gpt2_1g_b1),
        1: ("vgg16", 8, demo4_mu_vgg16_1g_b8),
    }),
], metadata={})
_ensure_state_metadata(demo4_source)
_set_physical_id(demo4_source, 0, "A")
_set_physical_id(demo4_source, 1, "B")
_set_physical_id(demo4_source, 2, "C")

demo4_target = ClusterState(gpus=[
    _demo_make_4g3g_gpu(0,
        payload4=("vit_base", 64, demo4_mu_vit_4g_b64),
        payload3=("llama", 1, demo4_mu_llama_3g_b1),
    ),
    _demo_make_1g_gpu(1, {
        0: ("resnet50", 8, demo4_mu_resnet_1g_b8),
    }),
    _demo_make_1g_gpu(2, {
        0: ("gpt2", 1, demo4_mu_gpt2_1g_b1),
        1: ("vgg16", 8, demo4_mu_vgg16_1g_b8),
    }),
    _demo_make_1g_gpu(3, {
        0: ("gpt2", 1, demo4_mu_gpt2_1g_b1),
    }),
], metadata={})
_ensure_state_metadata(demo4_target)

demo4_src_arr = np.zeros_like(arrival_rate, dtype=float)
demo4_tgt_arr = np.zeros_like(arrival_rate, dtype=float)
demo4_name_to_idx = {name: i for i, name in enumerate(WORKLOAD_NAMES)}
demo4_src_arr[demo4_name_to_idx["gpt2"]] = 5.0
demo4_src_arr[demo4_name_to_idx["vgg16"]] = 3.0
demo4_tgt_arr[demo4_name_to_idx["gpt2"]] = 2.0
demo4_tgt_arr[demo4_name_to_idx["vgg16"]] = 1.0
demo4_tgt_arr[demo4_name_to_idx["resnet50"]] = 1.0
demo4_tgt_arr[demo4_name_to_idx["vit_base"]] = 1.0
demo4_tgt_arr[demo4_name_to_idx["llama"]] = 1.0

print("GPU count fixed from MILP    :", len(demo4_target.real_gpus()))
summarize_state(demo4_source, title="[V2 MIXED DEMO] CURRENT STATE")
print("current templates:", [g.template_str() for g in demo4_source.real_gpus()])
summarize_state(demo4_target, title="[V2 MIXED DEMO] TARGET STATE")
print("target templates:", [g.template_str() for g in demo4_target.real_gpus()])

mixed_demo_res = run_v2_stage_iterative(
    source_state=demo4_source,
    target_state=demo4_target,
    src_arrival=demo4_src_arr,
    tgt_arrival=demo4_tgt_arr,
    stage_name="mixed_demo_v2",
)
print_v2_stage_result(mixed_demo_res, title="[V2 MIXED DEMO] iterative planner")


GPU count fixed from MILP    : 4
[V2 MIXED DEMO] CURRENT STATE
GPU 0: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: gpt2 / B1 / mu=1.1311 | [2,3) 1g: vgg16 / B8 / mu=217.9872 | [3,4) 1g: vgg16 / B8 / mu=217.9872 | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
GPU 1: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: free | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
GPU 2: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: vgg16 / B8 / mu=217.9872 | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
current templates: ['1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
[V2 MIXED DEMO] TARGET STATE
GPU 0: [0,4) 4g: vit_base / B64 / mu=1718.9838 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1: [0,1) 1g: resnet50 / B8 / mu=333.7088 | [1,2) 1g: free | [2,3) 1g: free | [3,4) 1g: free | [4,5) 1g: free | [5,6) 1g: free | [6,7) 1g: free
GPU 2: [0,1) 1g: gpt2 / B1 / mu=1.1311 | [1,2) 1g: vgg16 / B8 / mu=217.9872 | [2,3) 1g: free | [3,4) 1g: fre

## V3

### Cost-based ordering planner (first scaffold)

`v3` 先不直接替换 `v2` 的执行器，而是先把 phase-level 的打分器写出来。

目标是：

1. 对当前 `ready` / `in_progress` 的高层动作 phase 打分
2. 用统一的 score 看 `old-first`、`target-first`、`release-old-first` 哪个更值得先推进
3. 为后面真正的 `v3` 顺序优化执行器做基础


In [194]:
from typing import Dict, Any, List, Optional, Tuple
# ============================================================
# [Action Code Block V3-1] phase-level scoring scaffold
# ============================================================
def _v3_takeover_ready_score(source_state: "ClusterState", target_state: "ClusterState", it: Dict[str, Any]) -> int:
    workload = it.get("workload")
    if not workload:
        return 0
    gid = int(it.get("gpu_id")) if it.get("gpu_id") is not None else None
    slot = tuple(it.get("slot")) if it.get("slot") is not None else None
    cands = _v2_takeover_candidates(source_state, target_state, workload, exclude_gpu_id=gid, exclude_slot=slot)
    score = 0
    for c in cands:
        score = max(score, 3 if c.get("kind") == "unchanged" else 2)
    return score

def _v3_capacity_headroom_score(source_state: "ClusterState", target_state: "ClusterState", it: Dict[str, Any]) -> int:
    workload = it.get("workload")
    if not workload:
        return 0
    gid = int(it.get("gpu_id")) if it.get("gpu_id") is not None else None
    slot = tuple(it.get("slot")) if it.get("slot") is not None else None
    cands = _v2_takeover_candidates(source_state, target_state, workload, exclude_gpu_id=gid, exclude_slot=slot)
    stable = sum(1 for c in cands if c.get("kind") == "unchanged")
    return min(stable, 2)

def _v3_peak_gpu_delta_cost(it: Dict[str, Any]) -> int:
    t = str(it.get("type", ""))
    phase = str(it.get("current_phase", ""))
    if t == "create_gpu":
        return 1
    if t == "reconfiguration" and phase == "prepare_target_side":
        return 1
    if t == "reconfiguration" and phase == "in_place_reconfigure":
        return 0
    return 0

def _v3_drain_wait_cost(it: Dict[str, Any]) -> int:
    rem = it.get("drain_remaining")
    if rem is not None:
        return int(rem)
    inflight = it.get("inflight")
    if inflight is not None and "drain" in str(it.get("current_phase", "")):
        return int(inflight)
    return 0

def _v3_unlock_count_score(items: List[Dict[str, Any]], it: Dict[str, Any]) -> int:
    t = str(it.get("type", ""))
    phase = str(it.get("current_phase", ""))
    if t == "create_gpu":
        return sum(1 for x in items if x.get("blocked_by") == "no_takeover_capacity")
    if t == "reconfiguration" and phase in {"clear_old_side", "in_place_reconfigure", "finalize_target_side"}:
        return sum(1 for x in items if x.get("status") == "blocked")
    if t in {"remove_gpu", "remove_gpu_slot", "workload_change", "remove_instance"} and "drain" in phase:
        return 1
    return 0

def _v3_release_gpu_score(it: Dict[str, Any]) -> int:
    t = str(it.get("type", ""))
    phase = str(it.get("current_phase", ""))
    if t == "remove_gpu" and phase == "clear_old_gpu":
        return 3
    if t == "reconfiguration" and phase in {"clear_old_side", "in_place_reconfigure", "finalize_target_side"}:
        return 2
    if t in {"workload_change", "remove_instance", "remove_gpu_slot", "reconfiguration_slot"} and phase == "post_drain_ready":
        return 1
    return 0

def _v3_target_backed_enable_score(it: Dict[str, Any]) -> int:
    t = str(it.get("type", ""))
    phase = str(it.get("current_phase", ""))
    if t == "create_gpu" and phase == "prepare_target_side":
        return 2
    if t == "reconfiguration" and phase == "prepare_target_side":
        return 2
    if t == "place_instance" and phase == "prepare_target_side":
        return 1
    return 0

def _v3_in_place_reconfig_score(it: Dict[str, Any]) -> int:
    return 3 if (str(it.get("type", "")) == "reconfiguration" and str(it.get("current_phase", "")) == "in_place_reconfigure") else 0

def _v3_score_item(source_state: "ClusterState", target_state: "ClusterState", items: List[Dict[str, Any]], it: Dict[str, Any]) -> Dict[str, Any]:
    takeover_ready = _v3_takeover_ready_score(source_state, target_state, it)
    capacity_headroom = _v3_capacity_headroom_score(source_state, target_state, it)
    peak_gpu_delta = _v3_peak_gpu_delta_cost(it)
    drain_wait = _v3_drain_wait_cost(it)
    unlock_count = _v3_unlock_count_score(items, it)
    release_gpu = _v3_release_gpu_score(it)
    target_backed_enable = _v3_target_backed_enable_score(it)
    in_place_bonus = _v3_in_place_reconfig_score(it)
    total = (
        3 * takeover_ready
        + 2 * capacity_headroom
        + 2 * unlock_count
        + 3 * release_gpu
        + 2 * target_backed_enable
        + 2 * in_place_bonus
        - 4 * peak_gpu_delta
        - 2 * drain_wait
    )
    return {
        "id": it.get("id"),
        "type": it.get("type"),
        "phase": it.get("current_phase"),
        "status": it.get("status"),
        "score": total,
        "takeover_ready": takeover_ready,
        "capacity_headroom": capacity_headroom,
        "peak_gpu_delta": peak_gpu_delta,
        "drain_wait": drain_wait,
        "unlock_count": unlock_count,
        "release_gpu": release_gpu,
        "target_backed_enable": target_backed_enable,
        "in_place_bonus": in_place_bonus,
    }

def rank_v3_plan_items_from_stage(stage_res: Dict[str, Any], iteration: Optional[int] = None) -> List[Dict[str, Any]]:
    if not stage_res.get("iterations"):
        return []
    iter_res = stage_res["iterations"][-1] if iteration is None else stage_res["iterations"][int(iteration) - 1]
    source_state = iter_res["state_before"]
    target_state = stage_res["target_state"]
    items = list(iter_res["plan"].get("plan_items", []))
    scored = [_v3_score_item(source_state, target_state, items, it) for it in items]
    scored.sort(key=lambda x: (x["score"], x["release_gpu"], x["takeover_ready"]), reverse=True)
    return scored

def print_v3_scores(stage_res: Dict[str, Any], iteration: Optional[int] = None, title="[V3 PHASE SCORES]"):
    scored = rank_v3_plan_items_from_stage(stage_res, iteration=iteration)
    print("=" * 90)
    print(title)
    print("=" * 90)
    if not scored:
        print("No plan items.")
        return
    for row in scored:
        print(f"- {row['id']}: score={row['score']} | type={row['type']} | phase={row['phase']} | status={row['status']} | takeover={row['takeover_ready']} | headroom={row['capacity_headroom']} | peak+={row['peak_gpu_delta']} | drain_wait={row['drain_wait']} | unlock={row['unlock_count']} | release={row['release_gpu']} | target_enable={row['target_backed_enable']} | in_place={row['in_place_bonus']}")

print_v3_scores(mixed_demo_res, iteration=1, title="[V3 PHASE SCORES] mixed demo iter1")


[V3 PHASE SCORES] mixed demo iter1
- WC_gpu1_0_1_1g: score=11 | type=workload_change | phase=drain_old | status=blocked | takeover=3 | headroom=1 | peak+=0 | drain_wait=1 | unlock=1 | release=0 | target_enable=0 | in_place=0
- RECONF_gpu0_0_1_1g: score=9 | type=reconfiguration_slot | phase=drain_old | status=blocked | takeover=3 | headroom=1 | peak+=0 | drain_wait=1 | unlock=0 | release=0 | target_enable=0 | in_place=0
- RECONF_gpu0_1_2_1g: score=9 | type=reconfiguration_slot | phase=drain_old | status=blocked | takeover=3 | headroom=1 | peak+=0 | drain_wait=1 | unlock=0 | release=0 | target_enable=0 | in_place=0
- RECONF_gpu0_2_3_1g: score=9 | type=reconfiguration_slot | phase=drain_old | status=blocked | takeover=3 | headroom=1 | peak+=0 | drain_wait=1 | unlock=0 | release=0 | target_enable=0 | in_place=0
- RECONF_gpu0_3_4_1g: score=9 | type=reconfiguration_slot | phase=drain_old | status=blocked | takeover=3 | headroom=1 | peak+=0 | drain_wait=1 | unlock=0 | release=0 | target_enabl

### V3 greedy phase runner on mixed demo

下面给 `v3` 增加一个第一版可运行执行器：

- 每轮先调用 `v2` 生成当前候选 phases / plan items
- 对 plan items 聚合成 root items（例如 `RECONF_gpu0`, `CREATE_gpu3`, `WC_gpu1_...`）
- 按 score 选择一个 root item 先推进
- 只执行这个 root item 对应的 runtime actions
- 更新状态后再进入下一轮

这不是最终版 `v3`，但已经能体现“按评分选择顺序”的核心思想。


In [195]:
from typing import Dict, Any, List, Optional, Tuple
import json
# ============================================================
# [Action Code Block V3-2] first runnable greedy phase planner
# ============================================================
def _v3_root_id_from_item(it: Dict[str, Any]) -> str:
    item_id = str(it.get("id", ""))
    if item_id.startswith("RECONF_gpu"):
        m = item_id.split("_")
        return "_".join(m[:2]) if len(m) >= 2 else item_id
    if item_id.startswith("REMOVE_gpu"):
        m = item_id.split("_")
        return "_".join(m[:2]) if len(m) >= 2 else item_id
    if item_id.startswith("CREATE_gpu"):
        return item_id
    if item_id.startswith("WC_gpu"):
        parts = item_id.split('_')
        return '_'.join(parts[:5]) if len(parts) >= 5 else item_id
    if item_id.startswith("RM_gpu"):
        parts = item_id.split('_')
        return '_'.join(parts[:5]) if len(parts) >= 5 else item_id
    if item_id.startswith("PLACE_gpu"):
        parts = item_id.split('_')
        return '_'.join(parts[:5]) if len(parts) >= 5 else item_id
    return item_id

def _v3_group_scores(source_state: "ClusterState", target_state: "ClusterState", plan_items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    scored = [_v3_score_item(source_state, target_state, plan_items, it) for it in plan_items]
    groups = {}
    for row, it in zip(scored, plan_items):
        root = _v3_root_id_from_item(it)
        g = groups.setdefault(root, {"root_id": root, "score": 0, "rows": [], "gpu_id": it.get("gpu_id")})
        g["score"] += row["score"]
        g["rows"].append(row)
    out = list(groups.values())
    out.sort(key=lambda x: x["score"], reverse=True)
    return out

def _v3_action_matches_root(a: Dict[str, Any], root_id: str) -> bool:
    t = a.get("type")
    if root_id.startswith("CREATE_gpu"):
        gid = int(root_id.split('gpu',1)[1])
        if a.get("gpu_id") == gid:
            return True
        return False
    if root_id.startswith("RECONF_gpu"):
        gid = int(root_id.split('gpu',1)[1])
        if a.get("gpu_id") == gid:
            return True
        return False
    if root_id.startswith("REMOVE_gpu"):
        gid = int(root_id.split('gpu',1)[1])
        return a.get("gpu_id") == gid
    if root_id.startswith("WC_gpu") or root_id.startswith("RM_gpu") or root_id.startswith("PLACE_gpu"):
        parts = root_id.split('_')
        gid = int(parts[1].replace('gpu',''))
        slot = (int(parts[2]), int(parts[3]), parts[4])
        return a.get("gpu_id") == gid and tuple(a.get("slot", (None,None,None))) == slot
    return False

def _v3_select_actions_for_root(plan: Dict[str, Any], root_id: str) -> List[Dict[str, Any]]:
    actions = []
    pending_prefix = []
    for a in plan.get("executed_actions", []):
        t = a.get("type")
        if t in {"allocate_gpu", "configure_full_template"}:
            pending_prefix.append(a)
            continue
        if t == "place_target_layout":
            gid = a.get("gpu_id")
            if (root_id.startswith("CREATE_gpu") or root_id.startswith("RECONF_gpu")) and int(root_id.split('gpu',1)[1]) == gid:
                actions.extend(pending_prefix)
                pending_prefix = []
                actions.append(a)
            else:
                pending_prefix = []
            continue
        if _v3_action_matches_root(a, root_id):
            actions.append(a)
        else:
            pending_prefix = []
    return actions

def _v3_groups_conflict(g1: Dict[str, Any], g2: Dict[str, Any]) -> bool:
    gid1 = g1.get("gpu_id")
    gid2 = g2.get("gpu_id")
    if gid1 is not None and gid2 is not None and gid1 == gid2:
        return True
    r1 = str(g1.get("root_id", ""))
    r2 = str(g2.get("root_id", ""))
    if r1.startswith("PLACE_gpu") and r2.startswith("WC_gpu"):
        return r1.split('_')[1:5] == r2.split('_')[1:5]
    if r2.startswith("PLACE_gpu") and r1.startswith("WC_gpu"):
        return r2.split('_')[1:5] == r1.split('_')[1:5]
    return False

def _v3_choose_nonconflicting_groups(groups: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    chosen = []
    positive_exists = any(g.get("score", 0) > 0 for g in groups)
    threshold = 1 if positive_exists else 0
    for g in groups:
        if g.get("score", 0) < threshold:
            continue
        if any(_v3_groups_conflict(g, x) for x in chosen):
            continue
        chosen.append(g)
    return chosen if chosen else ([] if not groups else [groups[0]])

def run_v3_stage_iterative(source_state: "ClusterState", target_state: "ClusterState", src_arrival, tgt_arrival, stage_name="stage_v3", max_iters: int = 20) -> Dict[str, Any]:
    current_state = _v2_prepare_source_runtime(source_state, target_state)
    _ensure_state_metadata(current_state)
    initial_runtime_state = _deepcopy_state(current_state)
    iterations = []
    all_executed_actions = []
    final_plan = None
    reached_target = _matches_target_state(current_state, target_state) and len(_v2_drain_map(current_state)) == 0
    peak_active_gpu = len(_v2_active_pid_set(current_state))
    t0 = time.perf_counter()
    for iter_idx in range(1, max_iters + 1):
        if reached_target:
            break
        _v2_advance_drain(current_state)
        full_plan = plan_naive_action_v2(current_state, target_state, src_arrival, tgt_arrival, stage_name=f"{stage_name}_iter{iter_idx}")
        final_plan = full_plan
        plan_items = list(full_plan.get("plan_items", []))
        groups = _v3_group_scores(current_state, target_state, plan_items)
        if not groups:
            break
        chosen_groups = _v3_choose_nonconflicting_groups(groups)
        chosen_actions = []
        for g in chosen_groups:
            chosen_actions.extend(_v3_select_actions_for_root(full_plan, g["root_id"]))
        dedup = []
        seen = set()
        for a in chosen_actions:
            key = json.dumps(a, sort_keys=True, default=str)
            if key in seen:
                continue
            seen.add(key)
            dedup.append(a)
        chosen_actions = dedup
        next_state = _simulate_fine_actions_v2(current_state, full_plan["planned_state"], chosen_actions, next_physical_idx=current_state.metadata.get("next_physical_idx", 0))
        made_progress = (_v2_progress_signature(next_state) != _v2_progress_signature(current_state))
        reached_target = _matches_target_state(next_state, target_state) and len(_v2_drain_map(next_state)) == 0
        iter_peak = _v2_peak_from_actions(current_state, chosen_actions)
        peak_active_gpu = max(peak_active_gpu, iter_peak, len(_v2_active_pid_set(next_state)))
        iterations.append({
            "iteration": iter_idx,
            "full_plan": full_plan,
            "chosen_roots": chosen_groups,
            "chosen_actions": chosen_actions,
            "state_before": _deepcopy_state(current_state),
            "state_after": _deepcopy_state(next_state),
            "made_progress": made_progress,
            "reached_target": reached_target,
            "iter_peak_active_gpu": iter_peak,
            "active_gpu_after": len(_v2_active_pid_set(next_state)),
        })
        all_executed_actions.extend(chosen_actions)
        current_state = next_state
        if reached_target or not made_progress:
            break
    elapsed_sec = time.perf_counter() - t0
    return {
        "stage_name": stage_name,
        "iterations": iterations,
        "iteration_count": len(iterations),
        "reached_target": reached_target,
        "elapsed_sec": elapsed_sec,
        "executed_actions": all_executed_actions,
        "executed_state": current_state,
        "target_state": _deepcopy_state(target_state),
        "initial_runtime_state": initial_runtime_state,
        "peak_active_gpu": peak_active_gpu,
        "source_active_gpu": len(_v2_active_pid_set(initial_runtime_state)),
        "final_active_gpu": len(_v2_active_pid_set(current_state)),
        "final_plan": final_plan,
    }

def print_v3_stage_result(stage_res: Dict[str, Any], title="[V3 STAGE RESULT]"):
    print("=" * 90)
    print(title)
    print("=" * 90)
    print(f"Stage: {stage_res['stage_name']}")
    print(f"Iterations: {stage_res['iteration_count']} | Reached target: {stage_res['reached_target']} | Elapsed (s): {stage_res['elapsed_sec']:.6f}")
    print(f"Active GPU: source={stage_res['source_active_gpu']} final={stage_res['final_active_gpu']} peak={stage_res['peak_active_gpu']}")
    lines = _v2_changed_slot_summary_lines(stage_res['initial_runtime_state'])
    if lines:
        print("Changed slots summary:")
        for line in lines:
            print(f"  - {line}")
    for iter_res in stage_res['iterations']:
        print('-' * 90)
        chosen = iter_res['chosen_roots']
        roots = ', '.join(f"{g['root_id']}({g['score']})" for g in chosen)
        print(f"Iter {iter_res['iteration']}: choose [{roots}] progress={iter_res['made_progress']} reached_target={iter_res['reached_target']} active_after={iter_res['active_gpu_after']} iter_peak={iter_res['iter_peak_active_gpu']}")
        print('Executed actions:')
        for i, a in enumerate(iter_res['chosen_actions'], 1):
            print(f"  X{i:02d}. {_v2_format_action_short(a)}")

v3_mixed_demo_res = run_v3_stage_iterative(
    source_state=demo4_source,
    target_state=demo4_target,
    src_arrival=demo4_src_arr,
    tgt_arrival=demo4_tgt_arr,
    stage_name="mixed_demo_v3",
)
print_v3_stage_result(v3_mixed_demo_res, title="[V3 MIXED DEMO] greedy phase planner")


NameError: name 'json' is not defined

### V3 stage0 -> stage3

下面把 `v3` 接到和 `v2` 相同的主实验 `0 -> t1 -> t2 -> t3` 上。

注意：
- `v3` 仍然复用前面的 target builder
- 差别只在 action planner 本身的 phase 选择顺序


In [ ]:
# ============================================================
# [Action Code Block V3-3] Stage 0~3 mainline
# ============================================================
source0_v3 = ClusterState(gpus=[], metadata={})
_ensure_state_metadata(source0_v3)
print_stage_current_and_target(source0_v3, target0, title_prefix="[V3 STAGE 0]")
stage0_v3_res = run_v3_stage_iterative(
    source_state=source0_v3,
    target_state=target0,
    src_arrival=np.zeros_like(arrival_rate, dtype=float),
    tgt_arrival=arrival_rate,
    stage_name="stage0_v3",
)
plan0_v3_elapsed_sec = stage0_v3_res["elapsed_sec"]
executed0_v3 = stage0_v3_res["executed_state"]
print_v3_stage_result(stage0_v3_res, title="[V3 STAGE 0] source0 -> target0 -> greedy v3 planner")
print(f"[V3 STAGE 0] greedy v3 planner elapsed: {plan0_v3_elapsed_sec:.6f} sec")
source1_v3 = _canonicalize_state_for_next_round(executed0_v3)
print_canonicalize_result(executed0_v3, source1_v3, title="[V3 STAGE 0] canonicalize for next MILP")

target1_seq_v3 = build_target_state_from_milp(milp_res=milp_res_up_manual, prev_state=source1_v3, abstract_template_topk=64, physical_layout_topk=32, per_gpu_layout_topk=4, verbose=False)
print_stage_current_and_target(source1_v3, target1_seq_v3, title_prefix="[V3 STAGE 1]")
stage1_v3_res = run_v3_stage_iterative(
    source_state=source1_v3,
    target_state=target1_seq_v3,
    src_arrival=arrival_rate,
    tgt_arrival=arrival_rate_up_manual,
    stage_name="stage1_v3",
)
plan1_v3_elapsed_sec = stage1_v3_res["elapsed_sec"]
executed1_v3 = stage1_v3_res["executed_state"]
print_v3_stage_result(stage1_v3_res, title="[V3 STAGE 1] source1_v3 -> target1_seq_v3 -> greedy v3 planner")
print(f"[V3 STAGE 1] greedy v3 planner elapsed: {plan1_v3_elapsed_sec:.6f} sec")
source2_v3 = _canonicalize_state_for_next_round(executed1_v3)
print_canonicalize_result(executed1_v3, source2_v3, title="[V3 STAGE 1] canonicalize for next MILP")

target2_seq_v3 = build_target_state_from_milp(milp_res=milp_res_t2_manual, prev_state=source2_v3, abstract_template_topk=64, physical_layout_topk=32, per_gpu_layout_topk=4, verbose=False)
print_stage_current_and_target(source2_v3, target2_seq_v3, title_prefix="[V3 STAGE 2]")
stage2_v3_res = run_v3_stage_iterative(
    source_state=source2_v3,
    target_state=target2_seq_v3,
    src_arrival=arrival_rate_up_manual,
    tgt_arrival=arrival_rate_t2_manual,
    stage_name="stage2_v3",
)
plan2_v3_elapsed_sec = stage2_v3_res["elapsed_sec"]
executed2_v3 = stage2_v3_res["executed_state"]
print_v3_stage_result(stage2_v3_res, title="[V3 STAGE 2] source2_v3 -> target2_seq_v3 -> greedy v3 planner")
print(f"[V3 STAGE 2] greedy v3 planner elapsed: {plan2_v3_elapsed_sec:.6f} sec")
source3_v3 = _canonicalize_state_for_next_round(executed2_v3)
print_canonicalize_result(executed2_v3, source3_v3, title="[V3 STAGE 2] canonicalize for next MILP")

target3_seq_v3 = build_target_state_from_milp(milp_res=milp_res_up_3, prev_state=source3_v3, abstract_template_topk=64, physical_layout_topk=32, per_gpu_layout_topk=4, verbose=False)
print_stage_current_and_target(source3_v3, target3_seq_v3, title_prefix="[V3 STAGE 3]")
stage3_v3_res = run_v3_stage_iterative(
    source_state=source3_v3,
    target_state=target3_seq_v3,
    src_arrival=arrival_rate_t2_manual,
    tgt_arrival=arrival_rate_up_3,
    stage_name="stage3_v3",
)
plan3_v3_elapsed_sec = stage3_v3_res["elapsed_sec"]
executed3_v3 = stage3_v3_res["executed_state"]
print_v3_stage_result(stage3_v3_res, title="[V3 STAGE 3] source3_v3 -> target3_seq_v3 -> greedy v3 planner")
print(f"[V3 STAGE 3] greedy v3 planner elapsed: {plan3_v3_elapsed_sec:.6f} sec")


### V2 vs V3 mainline comparison

这里把 `v2` 和 `v3` 在主实验 `stage0~3` 上的结果放在一起，重点比较：

- `iteration_count`
- `action_count`
- `peak_active_gpu`
- `final_active_gpu`

以及后续你可以补上的服务维持指标。


In [ ]:
# ============================================================
# [Action Code Block V3-4] V2 vs V3 comparison
# ============================================================
def _planner_stage_summary_row(planner: str, stage: int, stage_res: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "planner": planner,
        "stage": int(stage),
        "reached_target": bool(stage_res["reached_target"]),
        "iteration_count": int(stage_res["iteration_count"]),
        "action_count": int(len(stage_res.get("executed_actions", []))),
        "peak_active_gpu": int(stage_res.get("peak_active_gpu", 0)),
        "final_active_gpu": int(stage_res.get("final_active_gpu", 0)),
        "elapsed_sec": float(stage_res.get("elapsed_sec", 0.0)),
    }

def _stage_service_metrics(stage_res: Dict[str, Any]) -> Dict[str, int]:
    rerouted = 0
    drain_wait = 0
    max_barriers = 0
    for iter_res in stage_res.get("iterations", []):
        plan = iter_res.get("plan") or iter_res.get("full_plan") or {}
        plan_items = list(plan.get("plan_items", []))
        barrier_count = 0
        for it in plan_items:
            if _v2_is_drain_barrier_item(it):
                barrier_count += 1
                rem = it.get("drain_remaining")
                if rem is not None:
                    drain_wait += int(rem)
        max_barriers = max(max_barriers, barrier_count)
        actions = iter_res.get("executed_actions") or iter_res.get("chosen_actions") or []
        for a in actions:
            if a.get("type") == "reroute_queued_tasks":
                rerouted += int(a.get("queued", 0))
    return {
        "rerouted_queued_tasks_total": rerouted,
        "drain_wait_total": drain_wait,
        "max_simultaneous_draining_slots": max_barriers,
    }

v2_v3_compare_df = pd.DataFrame([
    {**_planner_stage_summary_row("v2", 0, stage0_v2_res), **_stage_service_metrics(stage0_v2_res)},
    {**_planner_stage_summary_row("v2", 1, stage1_v2_res), **_stage_service_metrics(stage1_v2_res)},
    {**_planner_stage_summary_row("v2", 2, stage2_v2_res), **_stage_service_metrics(stage2_v2_res)},
    {**_planner_stage_summary_row("v2", 3, stage3_v2_res), **_stage_service_metrics(stage3_v2_res)},
    {**_planner_stage_summary_row("v3", 0, stage0_v3_res), **_stage_service_metrics(stage0_v3_res)},
    {**_planner_stage_summary_row("v3", 1, stage1_v3_res), **_stage_service_metrics(stage1_v3_res)},
    {**_planner_stage_summary_row("v3", 2, stage2_v3_res), **_stage_service_metrics(stage2_v3_res)},
    {**_planner_stage_summary_row("v3", 3, stage3_v3_res), **_stage_service_metrics(stage3_v3_res)},
])
print('[v2 vs v3 mainline comparison]')
display(v2_v3_compare_df)

def _v3_root_to_coarse(root_id: str) -> str:
    if root_id.startswith('CREATE_gpu'):
        return 'create_gpu'
    if root_id.startswith('RECONF_gpu'):
        return 'reconfiguration'
    if root_id.startswith('REMOVE_gpu'):
        return 'remove_gpu'
    if root_id.startswith('WC_gpu'):
        return 'workload_change'
    if root_id.startswith('RM_gpu'):
        return 'remove_instance'
    if root_id.startswith('PLACE_gpu'):
        return 'place_instance'
    return root_id

def _ordered_unique(seq):
    out = []
    for x in seq:
        if x not in out:
            out.append(x)
    return out

def _abstract_order_v2(stage_res: Dict[str, Any]) -> List[str]:
    out = []
    for it in stage_res["iterations"]:
        item_types = []
        for x in it["plan"].get("plan_items", []):
            t = str(x.get("type"))
            if t not in item_types:
                item_types.append(t)
        out.append(f"iter{it['iteration']}: " + ', '.join(item_types))
    return out

def _abstract_order_v3(stage_res: Dict[str, Any]) -> List[str]:
    out = []
    for it in stage_res["iterations"]:
        roots = [_v3_root_to_coarse(g['root_id']) for g in it.get('chosen_roots', [])]
        roots = _ordered_unique(roots)
        out.append(f"iter{it['iteration']}: " + ', '.join(roots))
    return out

print('\n[v2 abstract order]')
for s, r in [(0, stage0_v2_res), (1, stage1_v2_res), (2, stage2_v2_res), (3, stage3_v2_res)]:
    print(f'stage{s}:')
    for line in _abstract_order_v2(r):
        print('  -', line)

print('\n[v3 abstract order]')
for s, r in [(0, stage0_v3_res), (1, stage1_v3_res), (2, stage2_v3_res), (3, stage3_v3_res)]:
    print(f'stage{s}:')
    for line in _abstract_order_v3(r):
        print('  -', line)


### V3 better-than-v2 demo (remove first vs create first)

这个例子专门展示 `v3` 的顺序优化比 `v2` 更好的地方：

- source 和 target 的最终 GPU 数量相同
- old side 完全可以先 drain 再 remove
- 所以如果 planner 先 remove old，再 create new，就能把 peak GPU 压低 1

这里比较：
- `v2`: 倾向于同轮 `remove_gpu + create_gpu`
- `v3`: 优先 `old-first`，把 `create_gpu` 延后到 old GPU 释放之后


In [ ]:
# ============================================================
# [Action Code Block V3-5] Better-than-v2 demo: remove first vs create first
# ============================================================
demo5_mu_gpt2_1g_b1 = _demo_lookup_mu("gpt2", "1g", 1)
demo5_mu_vgg16_1g_b8 = _demo_lookup_mu("vgg16", "1g", 8)
demo5_mu_resnet_1g_b8 = _demo_lookup_mu("resnet50", "1g", 8)

demo5_source = ClusterState(gpus=[
    _demo_make_1g_gpu(0, {0: ("gpt2", 1, demo5_mu_gpt2_1g_b1)}),
    _demo_make_1g_gpu(1, {0: ("gpt2", 1, demo5_mu_gpt2_1g_b1)}),
    _demo_make_1g_gpu(2, {0: ("vgg16", 8, demo5_mu_vgg16_1g_b8)}),
], metadata={})
_ensure_state_metadata(demo5_source)
_set_physical_id(demo5_source, 0, "A")
_set_physical_id(demo5_source, 1, "B")
_set_physical_id(demo5_source, 2, "C")

demo5_target = ClusterState(gpus=[
    _demo_make_1g_gpu(1, {0: ("gpt2", 1, demo5_mu_gpt2_1g_b1)}),
    _demo_make_1g_gpu(2, {0: ("vgg16", 8, demo5_mu_vgg16_1g_b8)}),
    _demo_make_1g_gpu(3, {0: ("resnet50", 8, demo5_mu_resnet_1g_b8)}),
], metadata={})
_ensure_state_metadata(demo5_target)

demo5_src_arr = np.zeros_like(arrival_rate, dtype=float)
demo5_tgt_arr = np.zeros_like(arrival_rate, dtype=float)
demo5_name_to_idx = {name: i for i, name in enumerate(WORKLOAD_NAMES)}
demo5_src_arr[demo5_name_to_idx["gpt2"]] = 2.0
demo5_src_arr[demo5_name_to_idx["vgg16"]] = 1.0
demo5_tgt_arr[demo5_name_to_idx["gpt2"]] = 1.0
demo5_tgt_arr[demo5_name_to_idx["vgg16"]] = 1.0
demo5_tgt_arr[demo5_name_to_idx["resnet50"]] = 1.0

print_stage_current_and_target(demo5_source, demo5_target, title_prefix="[V3 BETTER DEMO]")

demo5_v2_res = run_v2_stage_iterative(
    source_state=demo5_source,
    target_state=demo5_target,
    src_arrival=demo5_src_arr,
    tgt_arrival=demo5_tgt_arr,
    stage_name="better_demo_v2",
)
print_v2_stage_result(demo5_v2_res, title="[V2 BETTER DEMO] remove+create")

demo5_v3_res = run_v3_stage_iterative(
    source_state=demo5_source,
    target_state=demo5_target,
    src_arrival=demo5_src_arr,
    tgt_arrival=demo5_tgt_arr,
    stage_name="better_demo_v3",
)
print_v3_stage_result(demo5_v3_res, title="[V3 BETTER DEMO] remove+create")

print('-' * 90)
print('[better demo comparison]')
print({
    'v2': {'iters': demo5_v2_res['iteration_count'], 'actions': len(demo5_v2_res['executed_actions']), 'peak': demo5_v2_res['peak_active_gpu'], 'final': demo5_v2_res['final_active_gpu']},
    'v3': {'iters': demo5_v3_res['iteration_count'], 'actions': len(demo5_v3_res['executed_actions']), 'peak': demo5_v3_res['peak_active_gpu'], 'final': demo5_v3_res['final_active_gpu']},
})
